In [ ]:
# 두 영상 중복 영역에서 NDFVI, NDFI(절댓값x) 계산하고 otsu 적용

import rasterio
from rasterio.warp import reproject, Resampling
import numpy as np
from skimage.filters import threshold_otsu
import matplotlib.pyplot as plt

def align_images(pre_flood_path, post_flood_path):
    """
    두 이미지의 공통 유효 영역을 찾아 정렬된 데이터 쌍을 반환하는 공통 함수.
    """
    with rasterio.open(pre_flood_path) as pre_src:
        pre_data = pre_src.read(1).astype(np.float32)
        pre_valid_mask = pre_data != 0
        profile = pre_src.profile
        print(f"기준 이미지 로드 완료: {pre_data.shape}")

    with rasterio.open(post_flood_path) as post_src:
        print("홍수 후 이미지를 기준 이미지에 맞게 정렬(reproject)합니다...")
        post_data_reprojected = np.zeros(pre_data.shape, dtype=np.float32)
        reproject(
            source=post_src.read(1).astype(np.float32),
            destination=post_data_reprojected,
            src_transform=post_src.transform,
            src_crs=post_src.crs,
            dst_transform=profile['transform'],
            dst_crs=profile['crs'],
            resampling=Resampling.bilinear
        )
        print("정렬 완료.")
    
    post_valid_mask = post_data_reprojected != 0
    intersection_valid_mask = np.logical_and(pre_valid_mask, post_valid_mask)
    
    pre_data[~intersection_valid_mask] = 0
    post_data_reprojected[~intersection_valid_mask] = 0
    
    return pre_data, post_data_reprojected, intersection_valid_mask, profile

def calculate_ndfvi(pre_data, post_data, valid_mask, profile, output_path):
    """NDFVI를 계산하고 저장합니다."""
    with np.errstate(divide='ignore', invalid='ignore'):
        epsilon = 1e-9
        numerator = np.abs(post_data) - np.abs(pre_data)
        denominator = np.abs(post_data) + np.abs(pre_data) + epsilon
        index_map = numerator / denominator
    
    index_map[~valid_mask] = -9999
    profile.update(dtype='float32', nodata=-9999)
    with rasterio.open(output_path, 'w', **profile) as dst:
        dst.write(index_map.astype(np.float32), 1)
        
    print(f"NDFVI 이미지가 '{output_path}'에 저장되었습니다.")
    return index_map, profile

def calculate_ndfi(pre_data, post_data, valid_mask, profile, output_path):
    """NDFI를 계산하고 저장합니다. """
    with np.errstate(divide='ignore', invalid='ignore'):
        epsilon = 1e-9
        abs_pre = np.abs(pre_data)
        abs_post = np.abs(post_data)
        # abs_pre = pre_data
        # abs_post = post_data
        numerator = abs_pre - abs_post
        denominator = abs_pre + abs_post + epsilon
        index_map = numerator / denominator
        
    index_map[~valid_mask] = -9999
    profile.update(dtype='float32', nodata=-9999)
    with rasterio.open(output_path, 'w', **profile) as dst:
        dst.write(index_map.astype(np.float32), 1)
        
    print(f"NDFI 이미지가 '{output_path}'에 저장되었습니다.")
    return index_map, profile

def otsu_classification(index_map, profile, output_path, index_name=""):
    """표준 Otsu 방법을 사용하여 침수 지역을 자동으로 분류합니다."""
    nodata_value = profile.get('nodata')

    if nodata_value is not None:
        no_data_mask = (index_map == nodata_value) | np.isnan(index_map)
    else:
        no_data_mask = np.isnan(index_map)
        
    valid_pixels = index_map[~no_data_mask]
    
    if valid_pixels.size == 0:
        raise ValueError("임계값을 계산할 유효한 픽셀이 없습니다.")

    plt.figure(figsize=(10, 6))
    plt.hist(valid_pixels, bins=256, range=(-1, 1), color='skyblue', edgecolor='black')
    plt.title(f'Histogram for {index_name} (Standard Otsu)')
    plt.xlabel(f'{index_name} Value')
    plt.ylabel('Frequency')
    plt.grid(True, linestyle='--', alpha=0.6)
    plt.show()

    optimal_threshold = threshold_otsu(valid_pixels)
    print(f"{index_name}에 대한 최적 임계값: {optimal_threshold:.4f}")

    flood_mask = (index_map > optimal_threshold).astype(np.uint8)
    
    mask_nodata_val = 255
    flood_mask[no_data_mask] = mask_nodata_val
    profile.update(nodata=mask_nodata_val, dtype=rasterio.uint8)
    
    with rasterio.open(output_path, 'w', **profile) as dst:
        dst.write(flood_mask, 1)

    print(f"{index_name} 기반 최종 침수 지도가 '{output_path}'에 저장되었습니다.")

if __name__ == '__main__':
    # --- 사용자 설정 ---
    pre_flood_image_path = r'<DATA_ROOT>\07_disaster\flood\Korea\s1_pipeline_test6\02_preprocess\S1A_IW_GRDH_1SDV_20250712T093124_20250712T093154_060049_0775E2_5D79_VV_processed.tif'
    post_flood_image_path = r'<DATA_ROOT>\07_disaster\flood\Korea\s1_pipeline_test6\02_preprocess\S1A_IW_GRDH_1SDV_20250719T092320_20250719T092350_060151_07795E_0ABD_VV_processed.tif'
    
    output_dir = r'<DATA_ROOT>\07_disaster\flood\Korea\s1_pipeline_test6\02_preprocess'
    ndfvi_output_path = f'{output_dir}\\ndfvi_result_otsu.tif'
    ndfvi_flood_map_path = f'{output_dir}\\flood_map_from_ndfvi_otsu.tif'
    ndfi_output_path = f'{output_dir}\\ndfi_result_otsu.tif'
    ndfi_flood_map_path = f'{output_dir}\\flood_map_from_ndfi_otsu.tif'
    
    try:
        pre_aligned, post_aligned, valid_mask, profile = align_images(
            pre_flood_image_path, post_flood_image_path
        )

        print("\n--- NDFVI 분석 시작 ---")
        ndfvi_map, ndfvi_profile = calculate_ndfvi(np.copy(pre_aligned), np.copy(post_aligned), valid_mask, profile.copy(), ndfvi_output_path)
        otsu_classification(ndfvi_map, ndfvi_profile, ndfvi_flood_map_path, index_name="NDFVI")

        print("\n--- NDFI 분석 시작 ---")
        ndfi_map, ndfi_profile = calculate_ndfi(np.copy(pre_aligned), np.copy(post_aligned), valid_mask, profile.copy(), ndfi_output_path)
        otsu_classification(ndfi_map, ndfi_profile, ndfi_flood_map_path, index_name="NDFI")

        print("\n모든 비교 분석 작업이 완료되었습니다. ✅")

    except Exception as e:
        print(f"\n오류가 발생했습니다: {e}")

In [ ]:
# aoi 지정시 영역 내에서만 NDFVI, NDFI(절댓값) 계산하고 edge-otsu 적용, dem 필터

import rasterio
from rasterio.warp import reproject, Resampling
from rasterio.features import rasterize
import numpy as np
import geopandas as gpd
from skimage.feature import canny
from skimage.filters import threshold_otsu
from scipy.ndimage import binary_dilation
import matplotlib.pyplot as plt

def align_images(pre_flood_path, post_flood_path, aoi_path=None):
    # (수정 없음)
    with rasterio.open(pre_flood_path) as pre_src:
        pre_data = pre_src.read(1).astype(np.float32)
        pre_valid_mask = pre_data != 0
        profile = pre_src.profile
        print(f"기준 이미지 로드 완료: {pre_data.shape}")
    with rasterio.open(post_flood_path) as post_src:
        print("홍수 후 이미지를 기준 이미지에 맞게 정렬(reproject)합니다...")
        post_data_raw = post_src.read(1).astype(np.float32)
        post_data_reprojected = np.zeros(pre_data.shape, dtype=np.float32)
        reproject(
            source=post_data_raw,
            destination=post_data_reprojected,
            src_transform=post_src.transform, src_crs=post_src.crs,
            dst_transform=profile['transform'], dst_crs=profile['crs'],
            resampling=Resampling.bilinear
        )
        print("정렬 완료.")
    post_valid_mask = post_data_reprojected != 0
    final_valid_mask = np.logical_and(pre_valid_mask, post_valid_mask)
    if aoi_path:
        print(f"지정된 AOI('{aoi_path}')로 분석 영역을 제한합니다...")
        aoi_gdf = gpd.read_file(aoi_path).to_crs(profile['crs'])
        aoi_mask = rasterize(shapes=aoi_gdf.geometry, out_shape=pre_data.shape, transform=profile['transform'], fill=0, default_value=1, dtype=np.uint8).astype(bool)
        final_valid_mask = np.logical_and(final_valid_mask, aoi_mask)
    pre_data[~final_valid_mask] = 0
    post_data_reprojected[~final_valid_mask] = 0
    return pre_data, post_data_reprojected, final_valid_mask, profile

def calculate_ndfvi(pre_data, post_data, valid_mask, profile, output_path):
    # (수정 없음)
    with np.errstate(divide='ignore', invalid='ignore'):
        epsilon = 1e-9
        numerator = np.abs(post_data) - np.abs(pre_data)
        denominator = np.abs(post_data) + np.abs(pre_data) + epsilon
        index_map = numerator / denominator
    index_map[~valid_mask] = -9999
    profile.update(dtype='float32', nodata=-9999)
    with rasterio.open(output_path, 'w', **profile) as dst:
        dst.write(index_map.astype(np.float32), 1)
    print(f"NDFVI 이미지가 '{output_path}'에 저장되었습니다.")
    return index_map, profile

def calculate_ndfi(pre_data, post_data, valid_mask, profile, output_path):
    # (수정 없음)
    with np.errstate(divide='ignore', invalid='ignore'):
        epsilon = 1e-9
        abs_pre = np.abs(pre_data)
        abs_post = np.abs(post_data)
        numerator = abs_pre - abs_post
        denominator = abs_pre + abs_post + epsilon
        index_map = numerator / denominator
    index_map[~valid_mask] = -9999
    profile.update(dtype='float32', nodata=-9999)
    with rasterio.open(output_path, 'w', **profile) as dst:
        dst.write(index_map.astype(np.float32), 1)
    print(f"NDFI 이미지가 '{output_path}'에 저장되었습니다.")
    return index_map, profile

def edge_otsu_classification(index_map, profile, output_path, index_name="", slope_path=None, slope_threshold=15):
    """Edge Otsu 분류 후, Slope 마스크를 적용하여 최종 침수 지역을 필터링합니다."""
    nodata_value = profile.get('nodata')
    if nodata_value is not None: no_data_mask = (index_map == nodata_value) | np.isnan(index_map)
    else: no_data_mask = np.isnan(index_map)
    index_for_canny = np.copy(index_map); index_for_canny[no_data_mask] = 0
    sigma_val=1.0; iterations_val=1
    edges = canny(index_for_canny, sigma=sigma_val); edges[no_data_mask] = False
    buffered_edges = binary_dilation(edges, iterations=iterations_val)
    valid_edge_pixels_mask = buffered_edges & ~no_data_mask
    edge_pixels = index_map[valid_edge_pixels_mask]
    if nodata_value is not None: edge_pixels = edge_pixels[edge_pixels != nodata_value]
    if edge_pixels.size == 0:
        print(f"경고: {index_name}에 대한 경계 픽셀을 찾을 수 없어 분류를 건너뜁니다.")
        return

    # --- ✅ 히스토그램 시각화 코드 ---
    plt.figure(figsize=(10, 6))
    plt.hist(edge_pixels, bins=256, range=(-1, 1), color='skyblue', edgecolor='black')
    plt.title(f'Histogram for {index_name} Edge Pixels')
    plt.xlabel(f'{index_name} Value')
    plt.ylabel('Frequency')
    plt.grid(True, linestyle='--', alpha=0.6)
    plt.show() # 그래프 창을 화면에 보여줍니다.
    # -----------------------------------

    optimal_threshold = threshold_otsu(edge_pixels)
    print(f"{index_name}에 대한 최적 임계값: {optimal_threshold:.4f}")

    # 1. Otsu 임계값으로 초기 침수 후보 지역 생성
    flood_mask = (index_map > optimal_threshold).astype(np.uint8)
    
    # 2. Slope 필터링 (slope_path가 있을 때만 실행)
    if slope_path:
        print(f"경사도 필터링 적용 (임계값: {slope_threshold}도)...")
        with rasterio.open(slope_path) as slope_src:
            slope_reprojected = np.zeros(index_map.shape, dtype=np.float32)
            reproject(
                source=rasterio.band(slope_src, 1),
                destination=slope_reprojected,
                src_transform=slope_src.transform, src_crs=slope_src.crs,
                dst_transform=profile['transform'], dst_crs=profile['crs'],
                resampling=Resampling.bilinear
            )
        steep_slope_mask = slope_reprojected >= slope_threshold
        flood_mask[steep_slope_mask] = 0
    
    # 최종 결과에 NoData 값 설정
    mask_nodata_val = 255
    flood_mask[no_data_mask] = mask_nodata_val
    profile.update(nodata=mask_nodata_val, dtype=rasterio.uint8)
    with rasterio.open(output_path, 'w', **profile) as dst:
        dst.write(flood_mask, 1)
    print(f"{index_name} 기반 최종 침수 지도가 '{output_path}'에 저장되었습니다.")

if __name__ == '__main__':
    # --- 사용자 설정 ---
    pre_flood_image_path = r'<DATA_ROOT>\07_disaster\flood\Korea\s1_pipeline_test6\02_preprocess\S1A_IW_GRDH_1SDV_20250712T093124_20250712T093154_060049_0775E2_5D79_VV_processed.tif'
    post_flood_image_path = r'<DATA_ROOT>\07_disaster\flood\Korea\s1_pipeline_test6\02_preprocess\S1A_IW_GRDH_1SDV_20250719T092320_20250719T092350_060151_07795E_0ABD_VV_processed.tif'
    AOI_SHP_PATH = r"<DATA_ROOT>\07_disaster\flood\Korea\202507_Jinju\shp\AOI.shp"
    # AOI_SHP_PATH = None
    SLOPE_TIF_PATH = r"<DATA_ROOT>\07_disaster\flood\Korea\202507_Jinju\dem\slope.tif"
    
    output_dir = r'<DATA_ROOT>\07_disaster\flood\Korea\s1_pipeline_test6\02_preprocess'
    ndfvi_output_path = f'{output_dir}\\ndfvi_result.tif'
    ndfvi_flood_map_path = f'{output_dir}\\flood_map_from_ndfvi.tif'
    ndfi_output_path = f'{output_dir}\\ndfi_result.tif'
    ndfi_flood_map_path = f'{output_dir}\\flood_map_from_ndfi.tif'
    
    try:
        pre_aligned, post_aligned, valid_mask, profile = align_images(
            pre_flood_image_path, post_flood_image_path, aoi_path=AOI_SHP_PATH
        )

        print("\n--- NDFVI 분석 시작 ---")
        ndfvi_map, ndfvi_profile = calculate_ndfvi(np.copy(pre_aligned), np.copy(post_aligned), valid_mask, profile.copy(), ndfvi_output_path)
        edge_otsu_classification(ndfvi_map, ndfvi_profile, ndfvi_flood_map_path, index_name="NDFVI", slope_path=SLOPE_TIF_PATH)

        print("\n--- NDFI 분석 시작 ---")
        ndfi_map, ndfi_profile = calculate_ndfi(np.copy(pre_aligned), np.copy(post_aligned), valid_mask, profile.copy(), ndfi_output_path)
        edge_otsu_classification(ndfi_map, ndfi_profile, ndfi_flood_map_path, index_name="NDFI", slope_path=SLOPE_TIF_PATH)

        print("\n모든 비교 분석 작업이 완료되었습니다. ✅")
    except Exception as e:
        print(f"\n오류가 발생했습니다: {e}")

In [ ]:
#ndfi, ndfvi 임계값 보다 크고 작을때 각각 구분해서 수체 마스크 형성

import rasterio
from rasterio.warp import reproject, Resampling
from rasterio.features import rasterize
import numpy as np
import geopandas as gpd
from skimage.feature import canny
from skimage.filters import threshold_otsu
from scipy.ndimage import binary_dilation
import matplotlib.pyplot as plt
import os

def align_images(pre_flood_path, post_flood_path, aoi_path=None):
    """
    두 이미지의 공통 유효 영역을 찾아 정렬된 데이터 쌍을 반환하는 공통 함수.
    """
    with rasterio.open(pre_flood_path) as pre_src:
        pre_data = pre_src.read(1).astype(np.float32)
        pre_valid_mask = pre_data != 0
        profile = pre_src.profile
        print(f"기준 이미지 로드 완료: {pre_data.shape}")
    with rasterio.open(post_flood_path) as post_src:
        print("홍수 후 이미지를 기준 이미지에 맞게 정렬(reproject)합니다...")
        post_data_raw = post_src.read(1).astype(np.float32)
        post_data_reprojected = np.zeros(pre_data.shape, dtype=np.float32)
        reproject(
            source=post_data_raw,
            destination=post_data_reprojected,
            src_transform=post_src.transform, src_crs=post_src.crs,
            dst_transform=profile['transform'], dst_crs=profile['crs'],
            resampling=Resampling.bilinear
        )
        print("정렬 완료.")
    post_valid_mask = post_data_reprojected != 0
    final_valid_mask = np.logical_and(pre_valid_mask, post_valid_mask)
    if aoi_path:
        print(f"지정된 AOI('{os.path.basename(aoi_path)}')로 분석 영역을 제한합니다...")
        aoi_gdf = gpd.read_file(aoi_path).to_crs(profile['crs'])
        aoi_mask = rasterize(shapes=aoi_gdf.geometry, out_shape=pre_data.shape, transform=profile['transform'], fill=0, default_value=1, dtype=np.uint8).astype(bool)
        final_valid_mask = np.logical_and(final_valid_mask, aoi_mask)
    pre_data[~final_valid_mask] = 0
    post_data_reprojected[~final_valid_mask] = 0
    return pre_data, post_data_reprojected, final_valid_mask, profile

def calculate_ndfvi(pre_data, post_data, valid_mask, profile, output_path):
    """NDFVI를 계산하고 저장합니다. (절대값 공식 사용)"""
    with np.errstate(divide='ignore', invalid='ignore'):
        epsilon = 1e-9
        numerator = np.abs(post_data) - np.abs(pre_data)
        denominator = np.abs(post_data) + np.abs(pre_data) + epsilon
        index_map = numerator / denominator
    index_map[~valid_mask] = -9999
    profile.update(dtype='float32', nodata=-9999)
    with rasterio.open(output_path, 'w', **profile) as dst:
        dst.write(index_map.astype(np.float32), 1)
    print(f"NDFVI 이미지가 '{output_path}'에 저장되었습니다.")
    return index_map, profile

def calculate_ndfi(pre_data, post_data, valid_mask, profile, output_path):
    """NDFI를 계산하고 저장합니다. (절대값 공식 사용)"""
    with np.errstate(divide='ignore', invalid='ignore'):
        epsilon = 1e-9
        abs_pre = np.abs(pre_data)
        abs_post = np.abs(post_data)
        numerator = abs_pre - abs_post
        denominator = abs_pre + abs_post + epsilon
        index_map = numerator / denominator
    index_map[~valid_mask] = -9999
    profile.update(dtype='float32', nodata=-9999)
    with rasterio.open(output_path, 'w', **profile) as dst:
        dst.write(index_map.astype(np.float32), 1)
    print(f"NDFI 이미지가 '{output_path}'에 저장되었습니다.")
    return index_map, profile

def edge_otsu_classification(index_map, profile, output_path, index_name="", slope_path=None, slope_threshold=15, comparison_operator='>'):
    """Edge Otsu 분류 후, Slope 마스크를 적용하여 최종 침수 지역을 필터링합니다."""
    nodata_value = profile.get('nodata')
    if nodata_value is not None: no_data_mask = (index_map == nodata_value) | np.isnan(index_map)
    else: no_data_mask = np.isnan(index_map)
    
    index_for_canny = np.copy(index_map); index_for_canny[no_data_mask] = 0
    sigma_val=1.0; iterations_val=1
    edges = canny(index_for_canny, sigma=sigma_val); edges[no_data_mask] = False
    buffered_edges = binary_dilation(edges, iterations=iterations_val)
    valid_edge_pixels_mask = buffered_edges & ~no_data_mask
    edge_pixels = index_map[valid_edge_pixels_mask]
    
    if nodata_value is not None: edge_pixels = edge_pixels[edge_pixels != nodata_value]
    if edge_pixels.size == 0:
        print(f"경고: {index_name}에 대한 경계 픽셀을 찾을 수 없어 분류를 건너뜁니다.")
        return

    plt.figure(figsize=(10, 6))
    plt.hist(edge_pixels, bins=256, range=(-1, 1), color='skyblue', edgecolor='black')
    plt.title(f'Histogram for {index_name} Edge Pixels')
    plt.xlabel(f'{index_name} Value')
    plt.ylabel('Frequency')
    plt.grid(True, linestyle='--', alpha=0.6)
    plt.show()

    optimal_threshold = threshold_otsu(edge_pixels)
    print(f"{index_name}에 대한 최적 임계값: {optimal_threshold:.4f}")

    # comparison_operator 인자에 따라 분류 기준을 동적으로 변경
    if comparison_operator == '>':
        print(f"분류 기준: {index_name} > {optimal_threshold:.4f}")
        flood_mask = (index_map > optimal_threshold).astype(np.uint8)
    elif comparison_operator == '<':
        print(f"분류 기준: {index_name} < {optimal_threshold:.4f}")
        flood_mask = (index_map < optimal_threshold).astype(np.uint8)
    else:
        raise ValueError("comparison_operator는 '>' 또는 '<' 여야 합니다.")
    
    if slope_path:
        print(f"경사도 필터링 적용 (임계값: {slope_threshold}도)...")
        with rasterio.open(slope_path) as slope_src:
            slope_reprojected = np.zeros(index_map.shape, dtype=np.float32)
            reproject(
                source=rasterio.band(slope_src, 1),
                destination=slope_reprojected,
                src_transform=slope_src.transform, src_crs=slope_src.crs,
                dst_transform=profile['transform'], dst_crs=profile['crs'],
                resampling=Resampling.bilinear
            )
        steep_slope_mask = slope_reprojected >= slope_threshold
        flood_mask[steep_slope_mask] = 0
    
    mask_nodata_val = 255
    flood_mask[no_data_mask] = mask_nodata_val
    profile.update(nodata=mask_nodata_val, dtype=rasterio.uint8)
    with rasterio.open(output_path, 'w', **profile) as dst:
        dst.write(flood_mask, 1)
    print(f"{index_name} 기반 최종 침수 지도가 '{output_path}'에 저장되었습니다.")

if __name__ == '__main__':
    # --- 사용자 설정 ---
    pre_flood_image_path = r'<DATA_ROOT>\07_disaster\flood\Korea\s1_pipeline_test6\02_preprocess\S1A_IW_GRDH_1SDV_20250712T093124_20250712T093154_060049_0775E2_5D79_VV_processed.tif'
    post_flood_image_path = r'<DATA_ROOT>\07_disaster\flood\Korea\s1_pipeline_test6\02_preprocess\S1A_IW_GRDH_1SDV_20250719T092320_20250719T092350_060151_07795E_0ABD_VV_processed.tif'
    AOI_SHP_PATH = r"<DATA_ROOT>\07_disaster\flood\Korea\202507_Jinju\shp\AOI.shp"
    SLOPE_TIF_PATH = r"<DATA_ROOT>\07_disaster\flood\Korea\202507_Jinju\dem\slope.tif"
    
    output_dir = r'<DATA_ROOT>\07_disaster\flood\Korea\s1_pipeline_test6\02_preprocess'
    ndfvi_output_path = f'{output_dir}\\ndfvi_result2.tif'
    ndfvi_flood_map_path = f'{output_dir}\\flood_map_from_ndfvi2.tif'
    ndfi_output_path = f'{output_dir}\\ndfi_result2.tif'
    ndfi_flood_map_path = f'{output_dir}\\flood_map_from_ndfi2.tif'
    
    try:
        pre_aligned, post_aligned, valid_mask, profile = align_images(
            pre_flood_image_path, post_flood_image_path, aoi_path=AOI_SHP_PATH
        )

        print("\n--- NDFVI 분석 시작 ---")
        ndfvi_map, ndfvi_profile = calculate_ndfvi(np.copy(pre_aligned), np.copy(post_aligned), valid_mask, profile.copy(), ndfvi_output_path)
        # NDFVI는 임계값보다 클 때 침수로 분류
        edge_otsu_classification(ndfvi_map, ndfvi_profile, ndfvi_flood_map_path, index_name="NDFVI", slope_path=SLOPE_TIF_PATH, comparison_operator='>')

        print("\n--- NDFI 분석 시작 ---")
        ndfi_map, ndfi_profile = calculate_ndfi(np.copy(pre_aligned), np.copy(post_aligned), valid_mask, profile.copy(), ndfi_output_path)
        # NDFI는 임계값보다 작을 때 침수로 분류
        edge_otsu_classification(ndfi_map, ndfi_profile, ndfi_flood_map_path, index_name="NDFI", slope_path=SLOPE_TIF_PATH, comparison_operator='<')

        print("\n모든 비교 분석 작업이 완료되었습니다. ✅")
    except Exception as e:
        print(f"\n오류가 발생했습니다: {e}")

In [ ]:
# aoi 지정시 영역 내에서만 db 영상에 edge-otsu 적용, dem 필터

import rasterio
from rasterio.warp import reproject, Resampling
from rasterio.features import rasterize
import numpy as np
import geopandas as gpd
from skimage.feature import canny
from skimage.filters import threshold_otsu
from scipy.ndimage import binary_dilation
import matplotlib.pyplot as plt

def align_and_mask_single_image(image_path, aoi_path=None):
    # (수정 없음)
    with rasterio.open(image_path) as src:
        profile = src.profile
        data = src.read(1).astype(np.float32)
        nodata = src.nodata
        if nodata is not None: valid_mask = (data != nodata) & (~np.isnan(data))
        else: valid_mask = (data != 0) & (~np.isnan(data))
        if aoi_path:
            print(f"지정된 AOI('{aoi_path}')로 분석 영역을 제한합니다...")
            aoi_gdf = gpd.read_file(aoi_path).to_crs(profile['crs'])
            aoi_mask = rasterize(shapes=aoi_gdf.geometry, out_shape=data.shape, transform=profile['transform'], fill=0, default_value=1, dtype=np.uint8).astype(bool)
            valid_mask = np.logical_and(valid_mask, aoi_mask)
    final_data = np.copy(data); nodata_val_out = -9999
    final_data[~valid_mask] = nodata_val_out
    profile.update(nodata=nodata_val_out)
    return final_data, profile

def classify_water_with_edge_otsu(db_image, profile, output_path, slope_path=None, slope_threshold=15):
    """(Edge-Otsu 수정) dB 영상에 Edge-Otsu를 적용하고 Slope 필터링을 수행합니다."""
    nodata_value = profile.get('nodata')

    # --- ✅ 핵심 수정 부분: NaN 값도 NoData로 인식하도록 수정 ---
    if nodata_value is not None:
        no_data_mask = (db_image == nodata_value) | np.isnan(db_image)
    else:
        no_data_mask = np.isnan(db_image)
    # --------------------------------------------------------

    db_for_canny = np.copy(db_image)
    db_for_canny[no_data_mask] = 0
    
    sigma_val = 2.0; iterations_val = 30
    edges = canny(db_for_canny, sigma=sigma_val)
    edges[no_data_mask] = False
    
    buffered_edges = binary_dilation(edges, iterations=iterations_val)
    valid_edge_pixels_mask = buffered_edges & ~no_data_mask
    edge_pixels = db_image[valid_edge_pixels_mask]
    
    if nodata_value is not None:
        edge_pixels = edge_pixels[edge_pixels != nodata_value]
    edge_pixels = edge_pixels[~np.isnan(edge_pixels)] # NaN 추가 제거

    if edge_pixels.size == 0:
        print(f"경고: 유효한 경계 픽셀을 찾을 수 없어 분류를 건너뜁니다.")
        return

    plt.figure(figsize=(10, 6))
    plt.hist(edge_pixels, bins=256, color='royalblue', edgecolor='black')
    plt.title(f'Histogram for Edge-Otsu on dB Image')
    plt.xlabel('Backscatter (dB)'); plt.ylabel('Frequency')
    plt.grid(True, linestyle='--', alpha=0.6); plt.show()

    optimal_threshold = threshold_otsu(edge_pixels)
    print(f"계산된 최적 임계값 (Edge-Otsu on dB): {optimal_threshold:.4f}")

    water_mask = (db_image < optimal_threshold).astype(np.uint8)
    
    if slope_path:
        print(f"경사도 필터링 적용 (임계값: {slope_threshold}도)...")
        with rasterio.open(slope_path) as slope_src:
            slope_reprojected = np.zeros(db_image.shape, dtype=np.float32)
            reproject(
                source=rasterio.band(slope_src, 1),
                destination=slope_reprojected,
                src_transform=slope_src.transform, src_crs=slope_src.crs,
                dst_transform=profile['transform'], dst_crs=profile['crs'],
                resampling=Resampling.bilinear
            )
        steep_slope_mask = slope_reprojected >= slope_threshold
        water_mask[steep_slope_mask] = 0
    
    mask_nodata_val = 255
    water_mask[no_data_mask] = mask_nodata_val
    
    profile.update(nodata=mask_nodata_val, dtype=rasterio.uint8)
    with rasterio.open(output_path, 'w', **profile) as dst:
        dst.write(water_mask, 1)
        
    print(f"Edge-Otsu 기반 최종 수체 지도가 '{output_path}'에 저장되었습니다.")

if __name__ == '__main__':
    # --- 사용자 설정 ---
    pre_flood_image_path = r'<DATA_ROOT>\07_disaster\flood\Korea\s1_pipeline_test6\02_preprocess\S1A_IW_GRDH_1SDV_20250712T093124_20250712T093154_060049_0775E2_5D79_VV_processed.tif'
    post_flood_image_path = r'<DATA_ROOT>\07_disaster\flood\Korea\s1_pipeline_test6\02_preprocess\S1A_IW_GRDH_1SDV_20250719T092320_20250719T092350_060151_07795E_0ABD_VV_processed.tif'
    
    # AOI_SHP_PATH = r"<DATA_ROOT>\07_disaster\flood\Korea\202507_Jinju\shp\AOI.shp"
    AOI_SHP_PATH = None
    SLOPE_TIF_PATH = r"<DATA_ROOT>\07_disaster\flood\Korea\202507_Jinju\dem\slope.tif"
    # SLOPE_TIF_PATH = None

    output_dir = r'<DATA_ROOT>\07_disaster\flood\Korea\s1_pipeline_test6\02_preprocess'
    pre_flood_map_path = f'{output_dir}\\water_map_pre_flood_edge_otsu.tif'
    post_flood_map_path = f'{output_dir}\\water_map_post_flood_edge_otsu.tif'
    
    try:
        print("\n--- 홍수 전 영상 분석 시작 ---")
        pre_aligned, pre_profile = align_and_mask_single_image(pre_flood_image_path, aoi_path=AOI_SHP_PATH)
        classify_water_with_edge_otsu(pre_aligned, pre_profile, pre_flood_map_path, slope_path=SLOPE_TIF_PATH)

        print("\n--- 홍수 후 영상 분석 시작 ---")
        post_aligned, post_profile = align_and_mask_single_image(post_flood_image_path, aoi_path=AOI_SHP_PATH)
        classify_water_with_edge_otsu(post_aligned, post_profile, post_flood_map_path, slope_path=SLOPE_TIF_PATH)
        
        print("\n모든 분석 작업이 완료되었습니다. ✅")
        
    except Exception as e:
        print(f"\n오류가 발생했습니다: {e}")

### aoi 지정시 영역 내에서만 db 영상에 otsu 적용, dem 필터

In [ ]:
# aoi 지정시 영역 내에서만 db 영상에 otsu 적용, dem 필터

import rasterio
from rasterio.warp import reproject, Resampling
from rasterio.features import rasterize
import numpy as np
import geopandas as gpd
from skimage.filters import threshold_otsu
import matplotlib.pyplot as plt

def align_and_mask_single_image(image_path, aoi_path=None):
    # (수정 없음)
    with rasterio.open(image_path) as src:
        profile = src.profile
        data = src.read(1).astype(np.float32)
        nodata = src.nodata
        if nodata is not None: valid_mask = (data != nodata) & (~np.isnan(data))
        else: valid_mask = (data != 0) & (~np.isnan(data))
        if aoi_path:
            print(f"지정된 AOI('{aoi_path}')로 분석 영역을 제한합니다...")
            aoi_gdf = gpd.read_file(aoi_path).to_crs(profile['crs'])
            aoi_mask = rasterize(shapes=aoi_gdf.geometry, out_shape=data.shape, transform=profile['transform'], fill=0, default_value=1, dtype=np.uint8).astype(bool)
            valid_mask = np.logical_and(valid_mask, aoi_mask)
    final_data = np.copy(data); nodata_val_out = -9999
    final_data[~valid_mask] = nodata_val_out
    profile.update(nodata=nodata_val_out)
    return final_data, profile

def classify_water_with_otsu(db_image, profile, output_path, slope_path=None, slope_threshold=15):
    """(Otsu) dB 영상에 표준 Otsu를 적용하고 Slope 필터링을 수행합니다."""
    nodata_value = profile.get('nodata')
    
    # --- ✅ 핵심 수정: 유효한 모든 픽셀을 샘플링 ---
    # NoData가 아닌 모든 픽셀을 임계값 계산에 사용합니다.
    if nodata_value is not None:
        valid_pixels = db_image[(db_image != nodata_value) & (~np.isnan(db_image))]
    else:
        valid_pixels = db_image[~np.isnan(db_image)]
    # ----------------------------------------------
    
    if valid_pixels.size == 0:
        print(f"경고: 유효한 픽셀이 없어 분류를 건너뜁니다.")
        return

    # 히스토그램 시각화
    plt.figure(figsize=(10, 6))
    plt.hist(valid_pixels, bins=256, color='royalblue', edgecolor='black')
    plt.title(f'Histogram for Standard Otsu on dB Image')
    plt.xlabel('Backscatter (dB)'); plt.ylabel('Frequency')
    plt.grid(True, linestyle='--', alpha=0.6); plt.show()

    # Otsu 임계값 계산
    optimal_threshold = threshold_otsu(valid_pixels)
    print(f"계산된 최적 임계값 (Standard Otsu on dB): {optimal_threshold:.4f}")

    # 1. Otsu 임계값으로 초기 수체 후보 지역 생성 (값이 낮은 쪽이 물)
    water_mask = (db_image < optimal_threshold).astype(np.uint8)
    
    # 2. Slope 필터링
    if slope_path:
        print(f"경사도 필터링 적용 (임계값: {slope_threshold}도)...")
        with rasterio.open(slope_path) as slope_src:
            slope_reprojected = np.zeros(db_image.shape, dtype=np.float32)
            reproject(
                source=rasterio.band(slope_src, 1),
                destination=slope_reprojected,
                src_transform=slope_src.transform, src_crs=slope_src.crs,
                dst_transform=profile['transform'], dst_crs=profile['crs'],
                resampling=Resampling.bilinear
            )
        steep_slope_mask = slope_reprojected >= slope_threshold
        water_mask[steep_slope_mask] = 0 # 경사가 가파른 곳은 물에서 제외
    
    # 최종 결과에 NoData 값 설정
    if nodata_value is not None:
        no_data_mask = (db_image == nodata_value) | np.isnan(db_image)
    else:
        no_data_mask = np.isnan(db_image)
        
    mask_nodata_val = 255
    water_mask[no_data_mask] = mask_nodata_val
    
    profile.update(nodata=mask_nodata_val, dtype=rasterio.uint8)
    with rasterio.open(output_path, 'w', **profile) as dst:
        dst.write(water_mask, 1)
        
    print(f"Otsu 기반 최종 수체 지도가 '{output_path}'에 저장되었습니다.")

if __name__ == '__main__':
    # --- 사용자 설정 ---
    pre_flood_image_path = r'<DATA_ROOT>\07_disaster\flood\Korea\s1_pipeline_test6\02_preprocess\S1A_IW_GRDH_1SDV_20250712T093124_20250712T093154_060049_0775E2_5D79_VV_processed.tif'
    post_flood_image_path = r'<DATA_ROOT>\07_disaster\flood\Korea\s1_pipeline_test6\02_preprocess\S1A_IW_GRDH_1SDV_20250719T092320_20250719T092350_060151_07795E_0ABD_VV_processed.tif'
    
    AOI_SHP_PATH = r"<DATA_ROOT>\07_disaster\flood\Korea\202507_Jinju\shp\AOI.shp"
    # AOI_SHP_PATH = None
    SLOPE_TIF_PATH = r"<DATA_ROOT>\07_disaster\flood\Korea\202507_Jinju\dem\slope.tif"

    output_dir = r'<DATA_ROOT>\07_disaster\flood\Korea\s1_pipeline_test6\02_preprocess'
    pre_flood_map_path = f'{output_dir}\\water_map_pre_flood_otsu.tif'
    post_flood_map_path = f'{output_dir}\\water_map_post_flood_otsu.tif'
    
    try:
        print("\n--- 홍수 전 영상 분석 시작 ---")
        pre_aligned, pre_profile = align_and_mask_single_image(pre_flood_image_path, aoi_path=AOI_SHP_PATH)
        classify_water_with_otsu(pre_aligned, pre_profile, pre_flood_map_path, slope_path=SLOPE_TIF_PATH)

        print("\n--- 홍수 후 영상 분석 시작 ---")
        post_aligned, post_profile = align_and_mask_single_image(post_flood_image_path, aoi_path=AOI_SHP_PATH)
        classify_water_with_otsu(post_aligned, post_profile, post_flood_map_path, slope_path=SLOPE_TIF_PATH)
        
        print("\n모든 분석 작업이 완료되었습니다. ✅")
        
    except Exception as e:
        print(f"\n오류가 발생했습니다: {e}")

### 절대값 없는 NDFI 공식을 사용하되, 결과로 나오는 이상치(outlier)를 제거한 후 Edge-Otsu를 적용 + 이상치 제거된 맵도 출력

In [ ]:
# 절대값 없는 NDFI 공식을 사용하되, 결과로 나오는 이상치(outlier)를 제거한 후 Edge-Otsu를 적용 + 이상치 제거된 맵도 출력

import rasterio
from rasterio.warp import reproject, Resampling
from rasterio.features import rasterize
import numpy as np
import geopandas as gpd
from skimage.feature import canny
from skimage.filters import threshold_otsu
from scipy.ndimage import binary_dilation
import matplotlib.pyplot as plt
import os

def align_images(pre_flood_path, post_flood_path, aoi_path=None):
    """
    두 이미지의 공통 유효 영역을 찾아 정렬된 데이터 쌍을 반환하는 공통 함수.
    """
    with rasterio.open(pre_flood_path) as pre_src:
        pre_data = pre_src.read(1).astype(np.float32)
        pre_valid_mask = (pre_data != 0) & (~np.isnan(pre_data))
        profile = pre_src.profile
        print(f"기준 이미지 로드 완료: {pre_data.shape}")

    with rasterio.open(post_flood_path) as post_src:
        print("홍수 후 이미지를 기준 이미지에 맞게 정렬(reproject)합니다...")
        post_data_reprojected = np.zeros(pre_data.shape, dtype=np.float32)
        reproject(
            source=post_src.read(1).astype(np.float32),
            destination=post_data_reprojected,
            src_transform=post_src.transform, src_crs=post_src.crs,
            dst_transform=profile['transform'], dst_crs=profile['crs'],
            resampling=Resampling.bilinear
        )
        print("정렬 완료.")
    
    post_valid_mask = (post_data_reprojected != 0) & (~np.isnan(post_data_reprojected))
    final_valid_mask = np.logical_and(pre_valid_mask, post_valid_mask)
    
    if aoi_path:
        print(f"지정된 AOI('{os.path.basename(aoi_path)}')로 분석 영역을 제한합니다...")
        aoi_gdf = gpd.read_file(aoi_path).to_crs(profile['crs'])
        aoi_mask = rasterize(shapes=aoi_gdf.geometry, out_shape=pre_data.shape, transform=profile['transform'], fill=0, default_value=1, dtype=np.uint8).astype(bool)
        final_valid_mask = np.logical_and(final_valid_mask, aoi_mask)
        
    pre_data[~final_valid_mask] = 0
    post_data_reprojected[~final_valid_mask] = 0
    
    return pre_data, post_data_reprojected, final_valid_mask, profile

def calculate_ndfi(pre_data, post_data, valid_mask, profile, output_path):
    """NDFI를 계산하고 저장합니다. (논문 방식: 절대값 없는 원본 공식)"""
    with np.errstate(divide='ignore', invalid='ignore'):
        epsilon = 1e-9
        numerator = pre_data - post_data
        denominator = pre_data + post_data
        
        # 분모가 0에 가까운 경우를 마스킹
        zero_denom_mask = np.abs(denominator) < epsilon
        
        index_map = np.full(pre_data.shape, -9999, dtype=np.float32)
        # 유효하고 분모가 0이 아닌 영역에 대해서만 계산
        calc_mask = valid_mask & ~zero_denom_mask
        index_map[calc_mask] = numerator[calc_mask] / denominator[calc_mask]

    profile.update(dtype='float32', nodata=-9999)
    with rasterio.open(output_path, 'w', **profile) as dst:
        dst.write(index_map, 1)
        
    print(f"원본 NDFI 이미지가 '{output_path}'에 저장되었습니다.")
    return index_map, profile

# ▼▼▼ 함수 추가: 이상치 필터링 및 저장 ▼▼▼
def filter_ndfi_and_save(ndfi_map, valid_mask, profile, output_path):
    """NDFI 맵에서 이상치를 제거하고 새로운 파일로 저장합니다."""
    print("NDFI 이상치 필터링 시작 (정상 범위: -1 ~ 0)...")
    
    # ndfi_map 복사하여 작업
    filtered_map = np.copy(ndfi_map)
    
    # 유효한 픽셀 중에서 정상 범위를 벗어나는 픽셀을 찾음
    outlier_mask = (filtered_map < -1) | (filtered_map > 0)
    
    # 유효 마스크 내의 이상치에만 NoData 값을 할당
    filtered_map[valid_mask & outlier_mask] = -9999
    
    profile.update(dtype='float32', nodata=-9999)
    with rasterio.open(output_path, 'w', **profile) as dst:
        dst.write(filtered_map, 1)
        
    print(f"이상치가 제거된 NDFI 맵이 '{output_path}'에 저장되었습니다.")
    return filtered_map
# ▲▲▲ 함수 추가 끝 ▲▲▲

def edge_otsu_classification(index_map, profile, output_path, index_name="", slope_path=None, slope_threshold=5):
    """Edge-Otsu 분류를 수행합니다."""
    nodata_value = profile.get('nodata')
    if nodata_value is not None:
        no_data_mask = (index_map == nodata_value) | np.isnan(index_map)
    else:
        no_data_mask = np.isnan(index_map)
    
    index_for_canny = np.copy(index_map)
    index_for_canny[no_data_mask] = 0

    edges = canny(index_for_canny, sigma=1.0)
    edges[no_data_mask] = False
    buffered_edges = binary_dilation(edges, iterations=5)
    
    pixels_for_otsu = index_map[buffered_edges & ~no_data_mask]
    
    if pixels_for_otsu.size == 0:
        print(f"경고: {index_name}에 대한 경계 픽셀을 찾을 수 없어 분류를 건너뜁니다.")
        return

    plt.figure(figsize=(10, 6))
    plt.hist(pixels_for_otsu, bins=256, range=(-1, 1), color='skyblue', edgecolor='black')
    plt.title(f'Histogram for {index_name} Edge Pixels (After Outlier Removal)')
    plt.xlabel(f'{index_name} Value')
    plt.ylabel('Frequency')
    plt.grid(True, linestyle='--', alpha=0.6)
    plt.show()

    optimal_threshold = threshold_otsu(pixels_for_otsu)
    print(f"{index_name}에 대한 최적 임계값: {optimal_threshold:.4f}")

    flood_mask = (index_map < optimal_threshold).astype(np.uint8)
    
    if slope_path:
        print(f"경사도 필터링 적용 (임계값: {slope_threshold}도)...")
        with rasterio.open(slope_path) as slope_src:
            slope_reprojected = np.zeros(index_map.shape, dtype=np.float32)
            reproject(
                source=rasterio.band(slope_src, 1),
                destination=slope_reprojected,
                src_transform=slope_src.transform, src_crs=slope_src.crs,
                dst_transform=profile['transform'], dst_crs=profile['crs'],
                resampling=Resampling.bilinear
            )
        steep_slope_mask = slope_reprojected >= slope_threshold
        flood_mask[steep_slope_mask] = 0
    
    mask_nodata_val = 255 # NoData로 사용할 값


    # 침수가 아닌 지역(값이 0)을 NoData 값으로 변경
    flood_mask[flood_mask == 0] = mask_nodata_val


    flood_mask[no_data_mask] = mask_nodata_val # 원래 데이터가 없던 영역도 NoData 처리
    
    profile.update(nodata=mask_nodata_val, dtype=rasterio.uint8)
    with rasterio.open(output_path, 'w', **profile) as dst:
        dst.write(flood_mask, 1)
        
    print(f"{index_name} 기반 최종 침수 지도가 '{output_path}'에 저장되었습니다.")


if __name__ == '__main__':
    # --- 사용자 설정 ---
    pre_flood_image_path = r'<DATA_ROOT>\07_disaster\flood\Korea\s1_pipeline_test6\02_preprocess\S1A_IW_GRDH_1SDV_20250712T093124_20250712T093154_060049_0775E2_5D79_VV_processed.tif'
    post_flood_image_path = r'<DATA_ROOT>\07_disaster\flood\Korea\s1_pipeline_test6\02_preprocess\S1A_IW_GRDH_1SDV_20250719T092320_20250719T092350_060151_07795E_0ABD_VV_processed.tif'
    AOI_SHP_PATH = r"<DATA_ROOT>\07_disaster\flood\Korea\202507_Jinju\shp\AOI.shp"
    SLOPE_TIF_PATH = r"<DATA_ROOT>\07_disaster\flood\Korea\202507_Jinju\dem\slope.tif"

    output_dir = r'<DATA_ROOT>\07_disaster\flood\Korea\s1_pipeline_test6\03_analysis_results'
    os.makedirs(output_dir, exist_ok=True)
    
    # 파일명 재정의
    ndfi_raw_output_path = os.path.join(output_dir, 'ndfi_result_raw.tif')
    ndfi_filtered_output_path = os.path.join(output_dir, 'ndfi_result_filtered.tif')
    ndfi_flood_map_path = os.path.join(output_dir, 'flood_map_from_ndfi_edge_otsu.tif')
    
    try:
        pre_aligned, post_aligned, valid_mask, profile = align_images(
            pre_flood_image_path, post_flood_image_path, aoi_path=AOI_SHP_PATH
        )

        print("\n--- NDFI (논문 방식) 분석 시작 ---")
        # 1. 원본 NDFI 계산 및 저장
        ndfi_map_raw, ndfi_profile = calculate_ndfi(
            np.copy(pre_aligned), np.copy(post_aligned), valid_mask, profile.copy(), ndfi_raw_output_path
        )
        
        # 2. 이상치 필터링 후 새로운 파일로 저장
        ndfi_map_filtered = filter_ndfi_and_save(
            ndfi_map_raw, valid_mask, ndfi_profile.copy(), ndfi_filtered_output_path
        )

        # 3. 필터링된 맵을 사용하여 Edge-Otsu 분류 수행
        edge_otsu_classification(
            ndfi_map_filtered, ndfi_profile, ndfi_flood_map_path, index_name="NDFI", slope_path=SLOPE_TIF_PATH
        )

        print("\n모든 분석 작업이 완료되었습니다. ✅")
    except Exception as e:
        print(f"\n오류가 발생했습니다: {e}")

### 중복된 영역만 남기기


In [ ]:
# 중복된 영역만 남기기

import rasterio
import geopandas as gpd
import subprocess
import os
import shutil
from rasterio.features import shapes
from shapely.geometry import shape, MultiPolygon
from shapely.ops import unary_union

# --- 1. 사용자 설정 ---

# 중복 영역을 확인할 원본 파일 경로
file1_path = r"<DATA_ROOT>\07_disaster\flood\Korea\s1_pipeline_test6\02_preprocess\S1A_IW_GRDH_1SDV_20250712T093124_20250712T093154_060049_0775E2_5D79_VV_processed.tif"
file2_path = r"<DATA_ROOT>\07_disaster\flood\Korea\s1_pipeline_test6\02_preprocess\S1A_IW_GRDH_1SDV_20250719T092320_20250719T092350_060151_07795E_0ABD_VV_processed.tif"

# 결과 파일을 저장할 폴더 경로
output_dir = r"<DATA_ROOT>\07_disaster\flood\Korea\s1_pipeline_test6\02_preprocess\05_valid_intersection"

# --- 아래는 코드 본문입니다 ---

def find_gdalwarp():
    """시스템 PATH에서 gdalwarp 실행 파일을 찾습니다."""
    gdalwarp_path = shutil.which("gdalwarp")
    if not gdalwarp_path:
        raise FileNotFoundError(
            "gdalwarp 명령어를 찾을 수 없습니다. "
            "GDAL이 설치되어 있고 시스템 PATH에 등록되었는지 확인해주세요."
        )
    return gdalwarp_path

def get_valid_data_footprint(raster_path: str) -> MultiPolygon:
    """래스터 파일에서 NoData를 제외한 유효 데이터 영역의 Footprint(폴리곤)를 추출합니다."""
    print(f"'{os.path.basename(raster_path)}'에서 유효 데이터 영역 추출 중...")
    with rasterio.open(raster_path) as src:
        # 밴드 1의 데이터를 읽어옵니다.
        image_data = src.read(1)
        # 메타데이터에서 NoData 값 확인 (없으면 0으로 가정)
        nodata_val = src.nodata if src.nodata is not None else 0
        # NoData가 아닌 영역을 마스크로 생성 (True: 유효 데이터, False: NoData)
        mask = image_data != nodata_val
        # 마스크를 기반으로 폴리곤 형태(shape) 추출
        # transform=src.transform은 픽셀 좌표를 실제 지리 좌표로 변환해줍니다.
        results = (
            {'properties': {'raster_val': v}, 'geometry': s}
            for i, (s, v) in enumerate(shapes(mask.astype('uint8'), mask=mask, transform=src.transform))
        )
        
    # 추출된 모든 폴리곤들을 하나로 합칩니다.
    geometries = [shape(r['geometry']) for r in results]
    if not geometries:
        return MultiPolygon() # 유효한 데이터가 없으면 빈 폴리곤 반환
        
    return unary_union(geometries)


def clip_to_valid_intersection(tif1, tif2, out_dir):
    """두 GeoTIFF의 실제 데이터 Footprint 교집합으로 각각의 파일을 잘라냅니다."""
    
    os.makedirs(out_dir, exist_ok=True)
    gdalwarp_executable = find_gdalwarp()

    try:
        # 1. 각 파일에서 유효 데이터 Footprint(폴리곤) 추출
        footprint1 = get_valid_data_footprint(tif1)
        footprint2 = get_valid_data_footprint(tif2)

        # 2. 두 Footprint의 기하학적 교집합(intersection) 계산
        print("\n두 영역의 공통 중첩 영역 계산 중...")
        intersection_geom = footprint1.intersection(footprint2)
        
        # 3. 중첩 영역이 유효한지 확인
        if intersection_geom.is_empty:
            print("오류: 두 파일의 실제 데이터 영역은 서로 중첩되지 않습니다.")
            return
        print("✅ 공통 영역 계산 완료.")
        
        # 4. gdalwarp의 -cutline 옵션으로 사용하기 위해 교집합 폴리곤을 임시 파일로 저장
        temp_cutline_path = os.path.join(out_dir, "temp_cutline.geojson")
        gdf = gpd.GeoDataFrame({'id': [0]}, geometry=[intersection_geom], crs="EPSG:4326") # CRS는 원본에 맞게 설정
        gdf.to_file(temp_cutline_path, driver='GeoJSON')

        # 5. gdalwarp를 사용하여 각 파일을 교집합 영역으로 잘라내기
        for i, input_path in enumerate([tif1, tif2], 1):
            base_name = os.path.basename(input_path)
            output_path = os.path.join(out_dir, f"{os.path.splitext(base_name)[0]}_valid_intersected.tif")
            
            print(f"처리 중 ({i}/2): {base_name} -> {os.path.basename(output_path)}")
            
            # gdalwarp 명령어 생성
            # -cutline: 잘라낼 기준선(폴리곤) 파일 지정
            # -crop_to_cutline: cutline 경계에 맞춰 이미지 크기를 조절
            # -dstnodata 0: 결과물에서 NoData 픽셀 값 지정
            command = [
                gdalwarp_executable,
                "-cutline", temp_cutline_path,
                "-crop_to_cutline",
                "-dstnodata", "0",
                "-overwrite",
                input_path,
                output_path,
            ]
            subprocess.run(command, check=True, capture_output=True, text=True)

        # 6. 임시 파일 삭제
        os.remove(temp_cutline_path)
        
        print("-" * 30)
        print(f"✅ 작업 완료! 결과가 '{out_dir}' 폴더에 저장되었습니다.")

    except FileNotFoundError:
        print(f"오류: 입력 파일 경로를 확인해주세요.")
    except Exception as e:
        print(f"알 수 없는 오류가 발생했습니다: {e}")

if __name__ == "__main__":
    clip_to_valid_intersection(file1_path, file2_path, output_dir)

In [ ]:
import rasterio
import numpy as np
import os
from skimage.feature import canny
from skimage.filters import threshold_otsu

# --- 1. 사용자 설정 ---

# 이전 단계에서 중복 영역만 잘라낸 tif 파일들이 있는 폴더 경로
input_dir = r"<DATA_ROOT>\07_disaster\flood\Korea\s1_pipeline_test6\02_preprocess\05_valid_intersection"

# 결과로 나올 수체 맵(water map)을 저장할 폴더 경로
output_dir = r"<DATA_ROOT>\07_disaster\flood\Korea\s1_pipeline_test6\02_preprocess\06_water_maps"


# --- 아래는 코드 본문입니다 ---

def create_water_map_edge_otsu(tif_path: str, out_dir: str):
    """
    Edge-Otsu 알고리즘을 사용하여 단일 GeoTIFF 파일로부터 수체 맵을 생성합니다.
    
    Args:
        tif_path (str): 입력 GeoTIFF 파일 경로 (데시벨 스케일).
        out_dir (str): 결과 파일을 저장할 폴더 경로.
    """
    base_name = os.path.basename(tif_path)
    output_path = os.path.join(out_dir, f"{os.path.splitext(base_name)[0]}_water_map.tif")
    
    print(f"처리 시작: {base_name}")

    try:
        with rasterio.open(tif_path) as src:
            # 래스터 메타데이터와 밴드 1의 데이터 읽기
            meta = src.meta.copy()
            image_db = src.read(1, masked=False).astype(np.float32) # 데이터 타입을 float으로 명시

            # NoData 값 확인 (이전 단계에서 0으로 설정됨)
            nodata_val = src.nodata if src.nodata is not None else 0
            
            # --- ▼▼▼ 오류 수정 부분 ▼▼▼ ---
            # 유효한 데이터만 추출 (NoData가 아니고, NaN도 아닌 픽셀)
            valid_data_mask = (image_db != nodata_val) & (~np.isnan(image_db))
            # --- ▲▲▲ 오류 수정 부분 ▲▲▲ ---

            valid_pixels = image_db[valid_data_mask]

            if valid_pixels.size == 0:
                print(f"  -> 경고: 유효한 데이터 픽셀이 없습니다. 건너뜁니다.")
                return

            # 1. Canny Edge Detection 적용
            # Canny는 0-255 범위의 정수형 데이터에서 잘 동작하므로, 유효 데이터의 범위를 정규화합니다.
            min_val, max_val = np.min(valid_pixels), np.max(valid_pixels)
            
            # max_val과 min_val이 같은 경우(모든 픽셀값이 동일) 분모가 0이 되는 것을 방지
            if max_val == min_val:
                rescaled_pixels = np.zeros_like(valid_pixels, dtype=np.uint8)
            else:
                rescaled_pixels = ((valid_pixels - min_val) / (max_val - min_val) * 255).astype(np.uint8)

            # 원본 이미지와 동일한 크기의 캔버스에 정규화된 픽셀 배치
            rescaled_image = np.zeros_like(image_db, dtype=np.uint8)
            rescaled_image[valid_data_mask] = rescaled_pixels
            
            # sigma 값은 이미지의 노이즈 수준에 따라 조절할 수 있습니다.
            edges = canny(rescaled_image, sigma=1.0, mask=valid_data_mask)

            # 2. 가장자리가 아닌 픽셀만 사용하여 Otsu 임계값 계산
            pixels_for_otsu = image_db[~edges & valid_data_mask]
            
            if pixels_for_otsu.size == 0:
                print(f"  -> 경고: 임계값 계산에 사용할 픽셀이 없습니다. Otsu를 전체 유효 픽셀에 적용합니다.")
                pixels_for_otsu = valid_pixels

            # Otsu 임계값 계산 (데시벨 스케일)
            otsu_thresh = threshold_otsu(pixels_for_otsu)
            print(f"  -> 계산된 Otsu 임계값 (dB): {otsu_thresh:.4f}")

            # 3. 임계값을 적용하여 수체 맵 생성 (이진 분류)
            water_map = np.zeros_like(image_db, dtype=np.uint8)
            water_map[(image_db <= otsu_thresh) & (valid_data_mask)] = 1 # 1: Water

            # 4. 결과 파일을 GeoTIFF로 저장
            meta.update({
                'driver': 'GTiff',
                'height': water_map.shape[0],
                'width': water_map.shape[1],
                'count': 1,
                'dtype': 'uint8',
                'nodata': 255
            })

            water_map[~valid_data_mask] = meta['nodata']

            with rasterio.open(output_path, 'w', **meta) as dst:
                dst.write(water_map, 1)
            
            print(f"✅ 처리 완료: {os.path.basename(output_path)}")

    except Exception as e:
        print(f"오류 발생 ({base_name}): {e}")


def main():
    """입력 폴더의 모든 tif 파일에 대해 수체 맵 생성을 실행합니다."""
    
    os.makedirs(output_dir, exist_ok=True)

    try:
        tif_files = [f for f in os.listdir(input_dir) if f.lower().endswith('.tif')]
        if not tif_files:
            print(f"'{input_dir}' 폴더에서 처리할 .tif 파일을 찾을 수 없습니다.")
            return
            
        print("-" * 50)
        print(f"총 {len(tif_files)}개의 파일에 대해 수체 탐지를 시작합니다.")
        print(f"입력 폴더: {input_dir}")
        print(f"결과 폴더: {output_dir}")
        print("-" * 50)

        for tif_file in tif_files:
            file_path = os.path.join(input_dir, tif_file)
            create_water_map_edge_otsu(file_path, output_dir)
            
        print("\n모든 작업이 완료되었습니다.")

    except FileNotFoundError:
        print(f"오류: 입력 폴더를 찾을 수 없습니다 - '{input_dir}'")
    except Exception as e:
        print(f"알 수 없는 오류가 발생했습니다: {e}")


if __name__ == "__main__":
    main()

### db 영상에 초기 임계값으로 1차적으로 수체 구분, edge-otsu 적용

In [ ]:
# db 영상에 초기 임계값으로 1차적으로 수체 구분, edge-otsu 적용

import rasterio
import geopandas as gpd
import numpy as np
from rasterio.features import shapes, rasterize
from rasterio.mask import mask
from rasterio.warp import reproject, Resampling
from skimage.feature import canny
from skimage.filters import threshold_otsu
from shapely.geometry import shape
from shapely.ops import unary_union
import os
import warnings

# --- ▼▼▼ 라이브러리 추가 ▼▼▼ ---
import matplotlib.pyplot as plt
from tqdm import tqdm
# --- ▲▲▲ 라이브러리 추가 ▲▲▲ ---

# 경고 메시지 무시
warnings.filterwarnings('ignore', 'GeoSeries.isna', UserWarning)

# --- 1. 사용자 설정 ---
pre_flood_image_path = r'<DATA_ROOT>\07_disaster\flood\Korea\s1_pipeline_test6\02_preprocess\S1A_IW_GRDH_1SDV_20250712T093124_20250712T093154_060049_0775E2_5D79_VV_processed.tif'
post_flood_image_path = r'<DATA_ROOT>\07_disaster\flood\Korea\s1_pipeline_test6\02_preprocess\S1A_IW_GRDH_1SDV_20250719T092320_20250719T092350_060151_07795E_0ABD_VV_processed.tif'
AOI_SHP_PATH = r"<DATA_ROOT>\07_disaster\flood\Korea\202507_Jinju\shp\AOI.shp"
SLOPE_TIF_PATH = r"<DATA_ROOT>\07_disaster\flood\Korea\202507_Jinju\dem\slope.tif"

output_dir = r'<DATA_ROOT>\07_disaster\flood\Korea\s1_pipeline_test6\04_advanced_edge_otsu'
os.makedirs(output_dir, exist_ok=True)
final_output_path = os.path.join(output_dir, 'flood_map_advanced_edge_otsu.tif')

INITIAL_THRESHOLD_DB = -15.0
MIN_EDGE_LENGTH_M = 200.0
BUFFER_RADIUS_M = 1500.0
SLOPE_THRESHOLD_DEGREES = 5

# --- 2. 코드 본문 ---

def get_utm_crs(gdf_wgs84):
    """WGS84 GeoDataFrame의 중심점에 맞는 UTM CRS를 반환합니다."""
    centroid = gdf_wgs84.union_all().centroid
    avg_lon, avg_lat = centroid.x, centroid.y
    utm_zone = int((avg_lon + 180) / 6) + 1
    is_northern = avg_lat >= 0
    epsg = 32600 + utm_zone if is_northern else 32700 + utm_zone
    return f'EPSG:{epsg}'

def main_workflow():
    """논문 기반 Edge Otsu 알고리즘 전체 워크플로우를 실행합니다."""
    try:
        # --- ▼▼▼ tqdm 진행률 표시 바 추가 ▼▼▼ ---
        with tqdm(total=5, desc="전체 작업 진행률") as pbar:
            # --- 단계 1: 데이터 로드 및 초기 필터링 ---
            pbar.set_description("1단계: 초기 필터링")
            with rasterio.open(post_flood_image_path) as src:
                post_db = src.read(1).astype(np.float32)
                profile = src.profile
                nodata_val = src.nodata or 0
                valid_data_mask = (post_db != nodata_val) & (~np.isnan(post_db))
                initial_mask = (post_db < INITIAL_THRESHOLD_DB) & valid_data_mask
            pbar.update(1)

            # --- 단계 2: 가장자리 탐지 및 필터링 ---
            pbar.set_description("2단계: 가장자리 탐지/필터링")
            edges_raster = canny(initial_mask, sigma=1.0)
            edge_shapes_generator = shapes(edges_raster.astype(np.uint8), mask=edges_raster, transform=profile['transform'])
            edge_geometries = [shape(s[0]) for s in edge_shapes_generator]

            if not edge_geometries: raise ValueError("가장자리를 탐지할 수 없습니다.")
                
            edge_gdf = gpd.GeoDataFrame({'geometry': edge_geometries}, crs=profile['crs'])
            utm_crs = get_utm_crs(edge_gdf)
            edge_gdf_utm = edge_gdf.to_crs(utm_crs)
            edge_gdf['length_m'] = edge_gdf_utm.geometry.length
            valid_edges_gdf = edge_gdf[edge_gdf['length_m'] >= MIN_EDGE_LENGTH_M].copy()
            
            if valid_edges_gdf.empty: raise ValueError(f"{MIN_EDGE_LENGTH_M}m 이상 길이의 가장자리를 찾을 수 없습니다.")
            print(f"   -> {len(valid_edges_gdf)}개의 유효한 가장자리 발견.")
            pbar.update(1)

            # --- 단계 3: 버퍼 생성 및 픽셀 샘플링 ---
            pbar.set_description("3단계: 버퍼 생성/픽셀 샘플링")
            valid_edges_gdf_utm = valid_edges_gdf.to_crs(utm_crs)
            buffer_geom_utm = valid_edges_gdf_utm.geometry.buffer(BUFFER_RADIUS_M)
            sampling_area_utm = gpd.GeoSeries([unary_union(buffer_geom_utm)], crs=utm_crs)
            sampling_area_orig_crs = sampling_area_utm.to_crs(profile['crs'])
            
            with rasterio.open(post_flood_image_path) as src:
                try:
                    sampled_pixels_masked, _ = mask(src, sampling_area_orig_crs, crop=False)
                    sampled_pixels = sampled_pixels_masked[0]
                    sampled_pixels_valid = sampled_pixels[(sampled_pixels != nodata_val) & (~np.isnan(sampled_pixels))]
                except ValueError: raise ValueError("버퍼 영역이 이미지 범위를 벗어났을 수 있습니다.")

            if sampled_pixels_valid.size == 0: raise ValueError("버퍼 영역에서 유효한 픽셀을 샘플링할 수 없습니다.")
            print(f"   -> {sampled_pixels_valid.size}개의 픽셀 샘플링 완료.")
            pbar.update(1)

            # --- 단계 4: 최종 임계값 계산 및 분류 ---
            pbar.set_description("4단계: Otsu 임계값 계산/분류")
            
            # --- ▼▼▼ 히스토그램 시각화 코드 추가 ▼▼▼ ---
            print("   -> 샘플 픽셀 히스토그램 생성 중...")
            plt.figure(figsize=(12, 7))
            plt.hist(sampled_pixels_valid, bins=256, color='royalblue', edgecolor='black', alpha=0.7)
            plt.title('Histogram of Sampled Pixels from Buffer Zones', fontsize=16)
            plt.xlabel('Backscatter (dB)', fontsize=12)
            plt.ylabel('Frequency', fontsize=12)
            plt.grid(True, linestyle='--', alpha=0.6)
            plt.show()
            # --- ▲▲▲ 히스토그램 시각화 코드 추가 ▲▲▲ ---

            final_threshold = threshold_otsu(sampled_pixels_valid)
            print(f"   -> 계산된 최종 임계값: {final_threshold:.4f} dB")
            flood_map = ((post_db < final_threshold) & valid_data_mask).astype(np.uint8)
            pbar.update(1)

            # --- 단계 5: 후처리 (경사도, AOI) ---
            pbar.set_description("5단계: 후처리 및 저장")
            with rasterio.open(SLOPE_TIF_PATH) as slope_src:
                slope_reprojected = np.zeros_like(post_db, dtype=np.float32)
                reproject(
                    source=rasterio.band(slope_src, 1), destination=slope_reprojected,
                    src_transform=slope_src.transform, src_crs=slope_src.crs,
                    dst_transform=profile['transform'], dst_crs=profile['crs'],
                    resampling=Resampling.bilinear
                )
            flood_map[slope_reprojected >= SLOPE_THRESHOLD_DEGREES] = 0

            temp_path = os.path.join(output_dir, 'temp_flood_map.tif')
            profile.update(dtype='uint8', nodata=255)
            flood_map[~valid_data_mask] = 255
            
            with rasterio.open(temp_path, 'w', **profile) as dst: dst.write(flood_map, 1)

            aoi_gdf = gpd.read_file(AOI_SHP_PATH)
            with rasterio.open(temp_path) as src:
                aoi_reproj = aoi_gdf.to_crs(src.crs)
                out_image, out_transform = mask(src, aoi_reproj.geometry, crop=True, nodata=255)
                out_meta = src.meta.copy()

            out_meta.update({ "height": out_image.shape[1], "width": out_image.shape[2], "transform": out_transform })
            with rasterio.open(final_output_path, "w", **out_meta) as dest: dest.write(out_image)
            
            os.remove(temp_path)
            pbar.update(1)
            # --- ▲▲▲ tqdm 진행률 표시 바 종료 ▲▲▲ ---
            
        print(f"\n✅ 모든 작업 완료! 최종 침수 지도가 다음 경로에 저장되었습니다:\n{final_output_path}")

    except Exception as e:
        print(f"\n오류가 발생했습니다: {e}")

if __name__ == '__main__':
    main_workflow()

#### 논문 따라해보기 3가지 방법

In [ ]:
import rasterio
from rasterio.mask import mask
import geopandas as gpd
import numpy as np
from skimage.filters import threshold_otsu
from skimage.feature import canny
from scipy.ndimage import binary_dilation
import os
from typing import Tuple, Optional

# --- 사용자 설정 변수 ---
# 제공해주신 파일 경로를 설정합니다.
PRE_FLOOD_IMAGE_PATH = r'<DATA_ROOT>\07_disaster\flood\Korea\s1_pipeline_test6\02_preprocess\S1A_IW_GRDH_1SDV_20250712T093124_20250712T093154_060049_0775E2_5D79_VV_processed.tif'
POST_FLOOD_IMAGE_PATH = r'<DATA_ROOT>\07_disaster\flood\Korea\s1_pipeline_test6\02_preprocess\S1A_IW_GRDH_1SDV_20250719T092320_20250719T092350_060151_07795E_0ABD_VV_processed.tif'
AOI_SHP_PATH = r"<DATA_ROOT>\07_disaster\flood\Korea\202507_Jinju\shp\AOI.shp"
SLOPE_TIF_PATH = r"<DATA_ROOT>\07_disaster\flood\Korea\202507_Jinju\dem\slope.tif"
OUTPUT_DIR = r'<DATA_ROOT>\07_disaster\flood\Korea\s1_pipeline_test6\03_output2' # 결과 파일 저장 경로

# --- 유틸리티 함수 ---

def load_and_clip_raster(raster_path: str, aoi_geom: list) -> Tuple[np.ndarray, rasterio.profiles.Profile]:
    """래스터 파일을 불러와 AOI에 맞춰 자릅니다."""
    with rasterio.open(raster_path) as src:
        try:
            out_image, out_transform = mask(src, aoi_geom, crop=True)
            out_meta = src.profile
            out_meta.update({
                "driver": "GTiff",
                "height": out_image.shape[1],
                "width": out_image.shape[2],
                "transform": out_transform
            })
            return out_image.squeeze(), out_meta
        except ValueError as e:
            print(f"AOI와 래스터가 겹치지 않을 수 있습니다: {e}")
            return None, None

def save_raster(data: np.ndarray, meta: rasterio.profiles.Profile, output_path: str):
    """결과 래스터를 파일로 저장합니다."""
    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    with rasterio.open(output_path, "w", **meta) as dest:
        dest.write(data.astype(meta['dtype']), 1)
    print(f"결과 파일 저장 완료: {output_path}")

# --- 전처리 함수 ---

def preprocess_sar(image: np.ndarray, slope: Optional[np.ndarray] = None) -> np.ndarray:
    """SAR 이미지에 대한 기본 전처리를 수행합니다."""
    # 입력 이미지는 이미 스펙클 필터링이 완료된 상태입니다.
    # 1. Speckle 필터링은 생략합니다.

    # 2. 경사가 15도 이상인 지역을 필터링(제거)합니다.
    if slope is not None and slope.shape == image.shape:
        # 경사가 15도 이상인 지역을 마스킹합니다.
        # 해당 지역의 픽셀 값을 NaN으로 설정하여 분석에서 제외합니다.
        # np.copy()를 사용하여 원본 이미지에 영향을 주지 않도록 합니다.
        processed_image = np.copy(image.astype(float))
        mask_condition = slope >= 15
        processed_image[mask_condition] = np.nan
        print("경사도 15도 이상 지역을 분석에서 제외했습니다.")
        return processed_image
    else:
        # 경사도 데이터가 없거나 크기가 맞지 않으면 원본 이미지를 그대로 반환합니다.
        if slope is not None:
            print("경사도 이미지의 크기가 SAR 이미지와 맞지 않아 경사 필터링을 생략합니다.")
        return image

# --- 핵심 알고리즘 구현 ---

def edge_otsu_thresholding(image: np.ndarray) -> np.ndarray:
    """Edge Otsu 알고리즘을 사용하여 물 영역을 탐지합니다."""
    print("Edge Otsu 알고리즘 실행 중...")
    # 초기 임계값으로 이진 이미지 생성 (Otsu 사용)
    # initial_thresh = threshold_otsu(image[~np.isnan(image)])
    initial_thresh = -16 
    binary_image = image < initial_thresh

    # Canny Edge 필터로 가장자리 추출
    edges = canny(binary_image)

    # 가장자리 버퍼링 (가장자리 주변 영역을 샘플링 대상으로 함)
    buffered_edges = binary_dilation(edges, iterations=5)

    # 버퍼링된 영역의 픽셀 값만 샘플링
    sample_pixels = image[buffered_edges]
    if sample_pixels.size == 0:
        print("Edge Otsu: 샘플링된 픽셀이 없어 초기 임계값을 사용합니다.")
        final_thresh = initial_thresh
    else:
        final_thresh = threshold_otsu(sample_pixels)

    print(f"Edge Otsu 최종 임계값: {final_thresh}")
    # NaN이었던 픽셀은 False(물이 아님)로 처리
    result = image < final_thresh
    result[np.isnan(image)] = False
    return result

def bmax_otsu_thresholding(image: np.ndarray, tile_size: int = 64, variance_percentile: int = 75) -> np.ndarray:
    """Bmax Otsu 알고리즘을 사용하여 물 영역을 탐지합니다."""
    print("Bmax Otsu 알고리즘 실행 중...")
    rows, cols = image.shape
    variances = []
    tile_coords = []

    # 이미지를 체스보드처럼 타일로 분할
    for r in range(0, rows, tile_size):
        for c in range(0, cols, tile_size):
            tile = image[r:r+tile_size, c:c+tile_size]
            if tile.size > 0 and not np.all(np.isnan(tile)):
                variances.append(np.nanvar(tile))
                tile_coords.append((r, c))

    if not variances:
        print("Bmax Otsu: 유효한 타일이 없어 일반 Otsu를 적용합니다.")
        final_thresh = threshold_otsu(image[~np.isnan(image)])
    else:
        # 분산이 높은 (bimodal 특성이 강할 가능성이 있는) 타일 선택
        variance_threshold = np.percentile(variances, variance_percentile)
        selected_tiles_pixels = []
        for i, (r, c) in enumerate(tile_coords):
            if variances[i] >= variance_threshold:
                tile = image[r:r+tile_size, c:c+tile_size]
                selected_tiles_pixels.extend(tile[~np.isnan(tile)].flatten())

        if not selected_tiles_pixels:
            print("Bmax Otsu: 선택된 타일에 픽셀이 없어 일반 Otsu를 적용합니다.")
            final_thresh = threshold_otsu(image[~np.isnan(image)])
        else:
            # 선택된 픽셀들로 최종 임계값 계산
            final_thresh = threshold_otsu(np.array(selected_tiles_pixels))
    
    print(f"Bmax Otsu 최종 임계값: {final_thresh}")
    # NaN이었던 픽셀은 False(물이 아님)로 처리
    result = image < final_thresh
    result[np.isnan(image)] = False
    return result

def fuzzy_otsu_thresholding(image: np.ndarray, spread_factor: float = 0.5, cutoff: float = 0.9) -> np.ndarray:
    """Fuzzy Otsu 알고리즘을 사용하여 물 영역을 탐지합니다."""
    print("Fuzzy Otsu 알고리즘 실행 중...")
    valid_pixels = image[~np.isnan(image)]
    if valid_pixels.size == 0:
        print("Fuzzy Otsu: 유효한 픽셀이 없습니다.")
        return np.zeros_like(image, dtype=bool)

    # 1. 전체 이미지에 대해 Otsu 임계값 계산
    global_thresh = threshold_otsu(valid_pixels)

    # 2. 물과 육지 클래스로 분리
    water_pixels = valid_pixels[valid_pixels < global_thresh]
    land_pixels = valid_pixels[valid_pixels >= global_thresh]

    if water_pixels.size == 0 or land_pixels.size == 0:
        print("Fuzzy Otsu: 클래스를 분리할 수 없어 일반 Otsu 결과를 반환합니다.")
        result = image < global_thresh
        result[np.isnan(image)] = False
        return result

    # 3. 각 클래스의 중심(평균값) 계산
    min_thresh_center = np.mean(water_pixels)
    max_thresh_center = np.mean(land_pixels)

    # 4. 두 중심값의 중간 지점 계산
    midpoint = (min_thresh_center + max_thresh_center) / 2
    
    # 5. Fuzzy Gaussian 함수 적용
    spread = (max_thresh_center - min_thresh_center) * spread_factor
    if spread == 0: spread = 1

    fuzzy_values = 1 - np.exp(-((image - midpoint)**2) / (2 * spread**2))
    
    print(f"Fuzzy Otsu 중간점: {midpoint}, Spread: {spread}")
    
    # 6. 임계값(cutoff)을 기준으로 최종 물 영역 결정
    result = fuzzy_values >= cutoff
    # NaN이었던 픽셀은 False(물이 아님)로 처리
    result[np.isnan(image)] = False
    return result

# --- 메인 실행 함수 ---

def main():
    """메인 처리 흐름을 담당하는 함수"""
    print("홍수 탐지 분석을 시작합니다.")

    # 1. AOI 데이터 로드
    try:
        aoi_gdf = gpd.read_file(AOI_SHP_PATH)
        aoi_geometry = [geom for geom in aoi_gdf.geometry]
        print("AOI 파일 로드 완료.")
    except Exception as e:
        print(f"오류: AOI 파일('{AOI_SHP_PATH}')을 읽을 수 없습니다. {e}")
        return

    # 2. 데이터 로드 및 전처리
    print("데이터 로드 및 전처리 중...")
    post_flood_img, meta = load_and_clip_raster(POST_FLOOD_IMAGE_PATH, aoi_geometry)
    pre_flood_img, _ = load_and_clip_raster(PRE_FLOOD_IMAGE_PATH, aoi_geometry)
    slope_img, _ = load_and_clip_raster(SLOPE_TIF_PATH, aoi_geometry)

    if post_flood_img is None:
        print(f"오류: 홍수 후 이미지('{POST_FLOOD_IMAGE_PATH}')를 처리할 수 없습니다.")
        return

    processed_post_flood = preprocess_sar(post_flood_img, slope_img)
    if pre_flood_img is not None:
        processed_pre_flood = preprocess_sar(pre_flood_img, slope_img)
    else:
        processed_pre_flood = None
        print("경고: 홍수 전 이미지를 찾을 수 없어, 침수 변화 분석은 생략됩니다.")

    # 3. 각 알고리즘 적용 및 결과 저장
    algorithms = {
        "edge_otsu": edge_otsu_thresholding,
        "bmax_otsu": bmax_otsu_thresholding,
        "fuzzy_otsu": fuzzy_otsu_thresholding
    }

    for name, func in algorithms.items():
        # 홍수 후 이미지에 대한 물 지도 생성
        post_water_map = func(processed_post_flood)
        output_path = os.path.join(OUTPUT_DIR, f'post_flood_water_{name}.tif')
        save_raster(post_water_map, meta, output_path)

        # 홍수 전 이미지가 있을 경우, 침수 변화 탐지
        if processed_pre_flood is not None:
            pre_water_map = func(processed_pre_flood)
            
            # 홍수 변화 탐지 (Post-water - Pre-water)
            flood_extent = post_water_map & ~pre_water_map
            output_path_change = os.path.join(OUTPUT_DIR, f'flood_change_{name}.tif')
            save_raster(flood_extent, meta, output_path_change)

    print("\n모든 분석이 완료되었습니다.")
    print(f"결과는 '{OUTPUT_DIR}' 폴더에 저장되었습니다.")


if __name__ == '__main__':
    # 출력 폴더가 없으면 생성
    if not os.path.exists(OUTPUT_DIR):
        os.makedirs(OUTPUT_DIR)
    main()



In [ ]:
import rasterio
from rasterio.mask import mask
from rasterio.warp import reproject, Resampling
import geopandas as gpd
import numpy as np
from skimage.filters import threshold_otsu
from skimage.feature import canny
from scipy.ndimage import binary_dilation
import os
from typing import Tuple, Optional

# --- 사용자 설정 변수 ---
# 제공해주신 파일 경로를 설정합니다.
PRE_FLOOD_IMAGE_PATH = r'<DATA_ROOT>\07_disaster\flood\Korea\s1_pipeline_test6\02_preprocess\S1A_IW_GRDH_1SDV_20250712T093124_20250712T093154_060049_0775E2_5D79_VV_processed.tif'
POST_FLOOD_IMAGE_PATH = r'<DATA_ROOT>\07_disaster\flood\Korea\s1_pipeline_test6\02_preprocess\S1A_IW_GRDH_1SDV_20250719T092320_20250719T092350_060151_07795E_0ABD_VV_processed.tif'
AOI_SHP_PATH = r"<DATA_ROOT>\07_disaster\flood\Korea\202507_Jinju\shp\AOI.shp"
SLOPE_TIF_PATH = r"<DATA_ROOT>\07_disaster\flood\Korea\202507_Jinju\dem\slope.tif"
OUTPUT_DIR = r'<DATA_ROOT>\07_disaster\flood\Korea\s1_pipeline_test6\03_output3' # 결과 파일 저장 경로

# --- 유틸리티 함수 ---

def load_and_clip_raster(raster_path: str, aoi_geom: list) -> Tuple[Optional[np.ndarray], Optional[rasterio.profiles.Profile]]:
    """래스터 파일을 불러와 AOI에 맞춰 자릅니다."""
    with rasterio.open(raster_path) as src:
        try:
            out_image, out_transform = mask(src, aoi_geom, crop=True)
            out_meta = src.profile
            out_meta.update({
                "driver": "GTiff",
                "height": out_image.shape[1],
                "width": out_image.shape[2],
                "transform": out_transform
            })
            return out_image.squeeze(), out_meta
        except ValueError as e:
            print(f"경고: AOI와 래스터 '{os.path.basename(raster_path)}'가 겹치지 않을 수 있습니다: {e}")
            return None, None

def save_raster(data: np.ndarray, meta: rasterio.profiles.Profile, output_path: str):
    """결과 래스터를 파일로 저장합니다."""
    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    # 데이터 타입을 bool에서 uint8로 변경하여 저장
    save_data = data.astype(np.uint8)
    meta['dtype'] = 'uint8'
    with rasterio.open(output_path, "w", **meta) as dest:
        dest.write(save_data, 1)
    print(f"결과 파일 저장 완료: {output_path}")

# --- 전처리 함수 ---

def preprocess_sar(image: np.ndarray, slope: Optional[np.ndarray] = None) -> np.ndarray:
    """SAR 이미지에 대해 경사도 기반 필터링을 수행합니다."""
    # 입력 이미지는 이미 스펙클 필터링이 완료된 상태이므로 해당 단계는 생략합니다.
    if slope is not None and slope.shape == image.shape:
        processed_image = np.copy(image.astype(float))
        mask_condition = slope >= 15
        processed_image[mask_condition] = np.nan
        print("경사도 15도 이상 지역을 분석에서 제외했습니다.")
        return processed_image
    else:
        if slope is not None:
            print("경고: 경사도 이미지의 크기가 SAR 이미지와 맞지 않아 경사 필터링을 생략합니다.")
        return image.astype(float) # 경사 필터링이 없어도 타입은 float으로 통일

# --- 핵심 알고리즘 구현 ---

def edge_otsu_thresholding(image: np.ndarray) -> np.ndarray:
    """Edge Otsu 알고리즘을 사용하여 물 영역을 탐지합니다."""
    print("Edge Otsu 알고리즘 실행 중...")
    valid_pixels = image[~np.isnan(image)]
    if valid_pixels.size == 0:
        print("Edge Otsu: 유효한 픽셀이 없습니다.")
        return np.zeros_like(image, dtype=bool)

    initial_thresh = threshold_otsu(valid_pixels)
    binary_image = image < initial_thresh

    edges = canny(binary_image)
    buffered_edges = binary_dilation(edges, iterations=5)
    sample_pixels = image[buffered_edges & ~np.isnan(image)]

    if sample_pixels.size == 0:
        print("Edge Otsu: 샘플링된 픽셀이 없어 초기 임계값을 사용합니다.")
        final_thresh = initial_thresh
    else:
        final_thresh = threshold_otsu(sample_pixels)

    print(f"Edge Otsu 최종 임계값: {final_thresh}")
    result = image < final_thresh
    result[np.isnan(image)] = False
    return result

def bmax_otsu_thresholding(image: np.ndarray, tile_size: int = 64, variance_percentile: int = 75) -> np.ndarray:
    """Bmax Otsu 알고리즘을 사용하여 물 영역을 탐지합니다."""
    print("Bmax Otsu 알고리즘 실행 중...")
    rows, cols = image.shape
    variances = []
    tile_coords = []

    for r in range(0, rows, tile_size):
        for c in range(0, cols, tile_size):
            tile = image[r:r+tile_size, c:c+tile_size]
            if tile.size > 0 and not np.all(np.isnan(tile)):
                variances.append(np.nanvar(tile))
                tile_coords.append((r, c))

    if not variances:
        print("Bmax Otsu: 유효한 타일이 없어 일반 Otsu를 적용합니다.")
        final_thresh = threshold_otsu(image[~np.isnan(image)])
    else:
        variance_threshold = np.percentile(variances, variance_percentile)
        selected_tiles_pixels = []
        for i, (r, c) in enumerate(tile_coords):
            if variances[i] >= variance_threshold:
                tile = image[r:r+tile_size, c:c+tile_size]
                selected_tiles_pixels.extend(tile[~np.isnan(tile)].flatten())

        if not selected_tiles_pixels:
            print("Bmax Otsu: 선택된 타일에 픽셀이 없어 일반 Otsu를 적용합니다.")
            final_thresh = threshold_otsu(image[~np.isnan(image)])
        else:
            final_thresh = threshold_otsu(np.array(selected_tiles_pixels))
    
    print(f"Bmax Otsu 최종 임계값: {final_thresh}")
    result = image < final_thresh
    result[np.isnan(image)] = False
    return result

def fuzzy_otsu_thresholding(image: np.ndarray, spread_factor: float = 0.5, cutoff: float = 0.9) -> np.ndarray:
    """Fuzzy Otsu 알고리즘을 사용하여 물 영역을 탐지합니다."""
    print("Fuzzy Otsu 알고리즘 실행 중...")
    valid_pixels = image[~np.isnan(image)]
    if valid_pixels.size == 0:
        print("Fuzzy Otsu: 유효한 픽셀이 없습니다.")
        return np.zeros_like(image, dtype=bool)

    global_thresh = threshold_otsu(valid_pixels)
    water_pixels = valid_pixels[valid_pixels < global_thresh]
    land_pixels = valid_pixels[valid_pixels >= global_thresh]

    if water_pixels.size == 0 or land_pixels.size == 0:
        print("Fuzzy Otsu: 클래스를 분리할 수 없어 일반 Otsu 결과를 반환합니다.")
        result = image < global_thresh
        result[np.isnan(image)] = False
        return result

    min_thresh_center = np.mean(water_pixels)
    max_thresh_center = np.mean(land_pixels)
    midpoint = (min_thresh_center + max_thresh_center) / 2
    spread = (max_thresh_center - min_thresh_center) * spread_factor
    if spread == 0: spread = 1

    fuzzy_values = 1 - np.exp(-((image - midpoint)**2) / (2 * spread**2))
    
    print(f"Fuzzy Otsu 중간점: {midpoint}, Spread: {spread}")
    result = fuzzy_values >= cutoff
    result[np.isnan(image)] = False
    return result

# --- 메인 실행 함수 ---

def main():
    """메인 처리 흐름을 담당하는 함수"""
    print("홍수 탐지 분석을 시작합니다.")

    try:
        aoi_gdf = gpd.read_file(AOI_SHP_PATH)
        aoi_geometry = [geom for geom in aoi_gdf.geometry]
        print("AOI 파일 로드 완료.")
    except Exception as e:
        print(f"오류: AOI 파일('{AOI_SHP_PATH}')을 읽을 수 없습니다. {e}")
        return

    print("\n데이터 로드 및 전처리 중...")
    post_flood_img, meta = load_and_clip_raster(POST_FLOOD_IMAGE_PATH, aoi_geometry)
    if post_flood_img is None:
        print(f"오류: 홍수 후 이미지('{POST_FLOOD_IMAGE_PATH}')를 처리할 수 없습니다. 분석을 중단합니다.")
        return
        
    pre_flood_img, _ = load_and_clip_raster(PRE_FLOOD_IMAGE_PATH, aoi_geometry)
    slope_img_raw, slope_meta_raw = load_and_clip_raster(SLOPE_TIF_PATH, aoi_geometry)

    resampled_slope = None
    if slope_img_raw is not None:
        print("경사도 이미지를 SAR 이미지 격자에 맞춰 재조정합니다...")
        destination = np.empty_like(post_flood_img, dtype=slope_img_raw.dtype)
        reproject(
            source=slope_img_raw,
            destination=destination,
            src_transform=slope_meta_raw['transform'],
            src_crs=slope_meta_raw['crs'],
            dst_transform=meta['transform'],
            dst_crs=meta['crs'],
            resampling=Resampling.bilinear
        )
        resampled_slope = destination
        print("재조정 완료.")
    else:
        print("경고: 경사도 이미지를 찾을 수 없어 경사 필터링을 생략합니다.")
        
    processed_post_flood = preprocess_sar(post_flood_img, resampled_slope)
    
    if pre_flood_img is not None:
        processed_pre_flood = preprocess_sar(pre_flood_img, resampled_slope)
    else:
        processed_pre_flood = None
        print("경고: 홍수 전 이미지를 찾을 수 없어, 침수 변화 분석은 생략됩니다.")

    algorithms = {
        "edge_otsu": edge_otsu_thresholding,
        "bmax_otsu": bmax_otsu_thresholding,
        "fuzzy_otsu": fuzzy_otsu_thresholding
    }

    print("\n알고리즘 적용 및 결과 저장 중...")
    for name, func in algorithms.items():
        post_water_map = func(processed_post_flood)
        output_path = os.path.join(OUTPUT_DIR, f'post_flood_water_{name}.tif')
        save_raster(post_water_map, meta, output_path)

        if processed_pre_flood is not None:
            pre_water_map = func(processed_pre_flood)
            flood_extent = post_water_map & ~pre_water_map
            output_path_change = os.path.join(OUTPUT_DIR, f'flood_change_{name}.tif')
            save_raster(flood_extent, meta, output_path_change)

    print("\n모든 분석이 완료되었습니다.")
    print(f"결과는 '{OUTPUT_DIR}' 폴더에 저장되었습니다.")

if __name__ == '__main__':
    if not os.path.exists(OUTPUT_DIR):
        os.makedirs(OUTPUT_DIR)
    main()



In [ ]:
import rasterio
from rasterio.mask import mask
from rasterio.warp import reproject, Resampling
import geopandas as gpd
import numpy as np
from skimage.filters import threshold_otsu
from skimage.feature import canny
from scipy.ndimage import binary_dilation
import os
from typing import Tuple, Optional

# --- 사용자 설정 변수 (파일 경로) ---
PRE_FLOOD_IMAGE_PATH = r'<DATA_ROOT>\07_disaster\flood\Korea\s1_pipeline_test6\02_preprocess\S1A_IW_GRDH_1SDV_20250712T093124_20250712T093154_060049_0775E2_5D79_VV_processed.tif'
POST_FLOOD_IMAGE_PATH = r'<DATA_ROOT>\07_disaster\flood\Korea\s1_pipeline_test6\02_preprocess\S1A_IW_GRDH_1SDV_20250719T092320_20250719T092350_060151_07795E_0ABD_VV_processed.tif'
AOI_SHP_PATH = r"<DATA_ROOT>\07_disaster\flood\Korea\202507_Jinju\shp\AOI.shp"
SLOPE_TIF_PATH = r"<DATA_ROOT>\07_disaster\flood\Korea\202507_Jinju\dem\slope.tif"
OUTPUT_DIR = r'<DATA_ROOT>\07_disaster\flood\Korea\s1_pipeline_test6\03_output4'

# --- 알고리즘 및 필터링 매개변수 (조절 가능) ---
SLOPE_THRESHOLD = 15  # 이 경사도(도) 이상인 지역은 최종 결과에서 제외됩니다.

# Edge Otsu 매개변수
EDGE_OTSU_BUFFER_ITERATIONS = 5  # 경계선 버퍼링 반복 횟수 (값이 클수록 넓은 영역 샘플링)

# Bmax Otsu 매개변수
BMAX_OTSU_TILE_SIZE = 64  # 분산을 계산할 타일 크기 (픽셀 단위)
BMAX_OTSU_VARIANCE_PERCENTILE = 75  # 사용할 분산의 백분위수 (e.g., 75는 상위 25% 타일 선택)

# Fuzzy Otsu 매개변수
FUZZY_OTSU_SPREAD_FACTOR = 0.5  # 퍼지 함수의 확산 계수 (값이 작을수록 민감)
FUZZY_OTSU_CUTOFF = 0.9  # 물로 판단할 멤버십 임계값 (0~1 사이, 1에 가까울수록 엄격)

# --- 유틸리티 함수 ---

def load_and_clip_raster(raster_path: str, aoi_geom: list) -> Tuple[Optional[np.ndarray], Optional[rasterio.profiles.Profile]]:
    """래스터 파일을 불러와 AOI에 맞춰 자릅니다."""
    with rasterio.open(raster_path) as src:
        try:
            out_image, out_transform = mask(src, aoi_geom, crop=True)
            out_meta = src.profile
            out_meta.update({
                "driver": "GTiff",
                "height": out_image.shape[1],
                "width": out_image.shape[2],
                "transform": out_transform
            })
            return out_image.squeeze(), out_meta
        except ValueError as e:
            print(f"경고: AOI와 래스터 '{os.path.basename(raster_path)}'가 겹치지 않을 수 있습니다: {e}")
            return None, None

def save_raster(data: np.ndarray, meta: rasterio.profiles.Profile, output_path: str):
    """결과 래스터를 파일로 저장합니다."""
    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    save_data = data.astype(np.uint8)
    meta['dtype'] = 'uint8'
    with rasterio.open(output_path, "w", **meta) as dest:
        dest.write(save_data, 1)
    print(f"결과 파일 저장 완료: {output_path}")

# --- 핵심 알고리즘 구현 ---

def edge_otsu_thresholding(image: np.ndarray, iterations: int) -> np.ndarray:
    """Edge Otsu 알고리즘을 사용하여 물 영역을 탐지합니다."""
    print("Edge Otsu 알고리즘 실행 중...")
    valid_pixels = image[~np.isnan(image)]
    if valid_pixels.size == 0:
        return np.zeros_like(image, dtype=bool)

    initial_thresh = threshold_otsu(valid_pixels)
    binary_image = image < initial_thresh
    edges = canny(binary_image)
    buffered_edges = binary_dilation(edges, iterations=iterations)
    sample_pixels = image[buffered_edges & ~np.isnan(image)]

    final_thresh = threshold_otsu(sample_pixels) if sample_pixels.size > 0 else initial_thresh
    print(f"Edge Otsu 최종 임계값: {final_thresh}")
    result = image < final_thresh
    result[np.isnan(image)] = False
    return result

def bmax_otsu_thresholding(image: np.ndarray, tile_size: int, variance_percentile: int) -> np.ndarray:
    """Bmax Otsu 알고리즘을 사용하여 물 영역을 탐지합니다."""
    print("Bmax Otsu 알고리즘 실행 중...")
    rows, cols = image.shape
    variances, tile_coords = [], []

    for r in range(0, rows, tile_size):
        for c in range(0, cols, tile_size):
            tile = image[r:r+tile_size, c:c+tile_size]
            if tile.size > 0 and not np.all(np.isnan(tile)):
                variances.append(np.nanvar(tile))
                tile_coords.append((r, c))

    if not variances:
        final_thresh = threshold_otsu(image[~np.isnan(image)])
    else:
        variance_threshold = np.percentile(variances, variance_percentile)
        selected_pixels = []
        for i, (r, c) in enumerate(tile_coords):
            if variances[i] >= variance_threshold:
                tile = image[r:r+tile_size, c:c+tile_size]
                selected_pixels.extend(tile[~np.isnan(tile)].flatten())
        
        final_thresh = threshold_otsu(np.array(selected_pixels)) if selected_pixels else threshold_otsu(image[~np.isnan(image)])

    print(f"Bmax Otsu 최종 임계값: {final_thresh}")
    result = image < final_thresh
    result[np.isnan(image)] = False
    return result

def fuzzy_otsu_thresholding(image: np.ndarray, spread_factor: float, cutoff: float) -> np.ndarray:
    """Fuzzy Otsu 알고리즘을 사용하여 물 영역을 탐지합니다."""
    print("Fuzzy Otsu 알고리즘 실행 중...")
    valid_pixels = image[~np.isnan(image)]
    if valid_pixels.size == 0:
        return np.zeros_like(image, dtype=bool)

    global_thresh = threshold_otsu(valid_pixels)
    water_pixels = valid_pixels[valid_pixels < global_thresh]
    land_pixels = valid_pixels[valid_pixels >= global_thresh]

    if water_pixels.size == 0 or land_pixels.size == 0:
        result = image < global_thresh
    else:
        min_center = np.mean(water_pixels)
        max_center = np.mean(land_pixels)
        midpoint = (min_center + max_center) / 2
        spread = (max_center - min_center) * spread_factor
        if spread == 0: spread = 1

        fuzzy_values = 1 - np.exp(-((image - midpoint)**2) / (2 * spread**2))
        print(f"Fuzzy Otsu 중간점: {midpoint}, Spread: {spread}")
        result = fuzzy_values >= cutoff
    
    result[np.isnan(image)] = False
    return result

# --- 메인 실행 함수 ---

def main():
    """메인 처리 흐름을 담당하는 함수"""
    print("홍수 탐지 분석을 시작합니다.")

    try:
        aoi_gdf = gpd.read_file(AOI_SHP_PATH)
        aoi_geometry = [geom for geom in aoi_gdf.geometry]
        print("AOI 파일 로드 완료.")
    except Exception as e:
        print(f"오류: AOI 파일('{AOI_SHP_PATH}')을 읽을 수 없습니다. {e}")
        return

    print("\n데이터 로드 중...")
    post_flood_img, meta = load_and_clip_raster(POST_FLOOD_IMAGE_PATH, aoi_geometry)
    if post_flood_img is None:
        print(f"오류: 홍수 후 이미지('{POST_FLOOD_IMAGE_PATH}')를 처리할 수 없습니다.")
        return
        
    pre_flood_img, _ = load_and_clip_raster(PRE_FLOOD_IMAGE_PATH, aoi_geometry)
    slope_img_raw, slope_meta_raw = load_and_clip_raster(SLOPE_TIF_PATH, aoi_geometry)

    resampled_slope = None
    if slope_img_raw is not None:
        print("경사도 이미지를 SAR 이미지 격자에 맞춰 재조정합니다...")
        destination = np.empty_like(post_flood_img, dtype=slope_img_raw.dtype)
        reproject(
            source=slope_img_raw, destination=destination,
            src_transform=slope_meta_raw['transform'], src_crs=slope_meta_raw['crs'],
            dst_transform=meta['transform'], dst_crs=meta['crs'],
            resampling=Resampling.bilinear
        )
        resampled_slope = destination
        print("재조정 완료.")
    else:
        print("경고: 경사도 이미지를 찾을 수 없어 경사 필터링을 생략합니다.")

    post_flood_float = post_flood_img.astype(float)
    pre_flood_float = pre_flood_img.astype(float) if pre_flood_img is not None else None

    algorithms = {
        "edge_otsu": (edge_otsu_thresholding, {"iterations": EDGE_OTSU_BUFFER_ITERATIONS}),
        "bmax_otsu": (bmax_otsu_thresholding, {"tile_size": BMAX_OTSU_TILE_SIZE, "variance_percentile": BMAX_OTSU_VARIANCE_PERCENTILE}),
        "fuzzy_otsu": (fuzzy_otsu_thresholding, {"spread_factor": FUZZY_OTSU_SPREAD_FACTOR, "cutoff": FUZZY_OTSU_CUTOFF})
    }

    print("\n알고리즘 적용 및 결과 저장 중...")
    for name, (func, params) in algorithms.items():
        # 1. 알고리즘으로 물 영역 탐지
        post_water_map = func(post_flood_float, **params)
        
        # 2. (마지막 단계) 경사도 필터링 적용
        if resampled_slope is not None:
            slope_mask = resampled_slope >= SLOPE_THRESHOLD
            post_water_map[slope_mask] = False
            print(f"[{name}] 경사도 {SLOPE_THRESHOLD}도 이상 지역을 최종 결과에서 제외했습니다.")

        output_path = os.path.join(OUTPUT_DIR, f'post_flood_water_{name}.tif')
        save_raster(post_water_map, meta, output_path)

        if pre_flood_float is not None:
            pre_water_map = func(pre_flood_float, **params)
            if resampled_slope is not None:
                pre_water_map[slope_mask] = False
            
            flood_extent = post_water_map & ~pre_water_map
            output_path_change = os.path.join(OUTPUT_DIR, f'flood_change_{name}.tif')
            save_raster(flood_extent, meta, output_path_change)

    print(f"\n모든 분석이 완료되었습니다. 결과는 '{OUTPUT_DIR}' 폴더에 저장되었습니다.")

if __name__ == '__main__':
    if not os.path.exists(OUTPUT_DIR):
        os.makedirs(OUTPUT_DIR)
    main()



### AOI 영역 내에서 Edge Otsu, Bmax Otsu, Fuzzy Otsu 기법 적용하고 히스토그램 저장

In [ ]:
# AOI 영역 내에서 Edge Otsu, Bmax Otsu, Fuzzy Otsu 기법 적용하고 히스토그램 저장

import rasterio
from rasterio.mask import mask
from rasterio.warp import reproject, Resampling
import geopandas as gpd
import numpy as np
from skimage.filters import threshold_otsu
from skimage.feature import canny
from scipy.ndimage import binary_dilation
import os
from typing import Tuple, Optional, Dict, Any
import matplotlib.pyplot as plt

# --- 사용자 설정 변수 (파일 경로) ---
PRE_FLOOD_IMAGE_PATH = r'<DATA_ROOT>\07_disaster\flood\Korea\s1_pipeline_test6\02_preprocess\S1A_IW_GRDH_1SDV_20250712T093124_20250712T093154_060049_0775E2_5D79_VV_processed.tif'
POST_FLOOD_IMAGE_PATH = r'<DATA_ROOT>\07_disaster\flood\Korea\s1_pipeline_test6\02_preprocess\S1A_IW_GRDH_1SDV_20250719T092320_20250719T092350_060151_07795E_0ABD_VV_processed.tif'
AOI_SHP_PATH = r"<DATA_ROOT>\07_disaster\flood\Korea\202507_Jinju\shp\AOI_small4.shp"
SLOPE_TIF_PATH = r"<DATA_ROOT>\07_disaster\flood\Korea\202507_Jinju\dem\slope.tif"
OUTPUT_DIR = r'<DATA_ROOT>\07_disaster\flood\Korea\s1_pipeline_test6\03_output_small4'

# --- 알고리즘 및 필터링 매개변수 (조절 가능) ---
SLOPE_THRESHOLD = 15  # 이 경사도(도) 이상인 지역은 최종 결과에서 제외됩니다.

# Edge Otsu 매개변수
EDGE_OTSU_INITIAL_THRESHOLD = -15  # 초기 임계값. `None`이면 자동 계산, -15과 같은 숫자를 입력하면 고정값 사용.
EDGE_OTSU_BUFFER_ITERATIONS = 5  # 경계선 버퍼링 반복 횟수 (값이 클수록 넓은 영역 샘플링)

# Bmax Otsu 매개변수
BMAX_OTSU_INITIAL_THRESHOLD = -15  # Bmax 계산을 시작하기 위한 초기 임계값. `None`이면 분산만으로 계산.
BMAX_OTSU_TILE_SIZE = 64  # 분산을 계산할 타일 크기 (픽셀 단위)
BMAX_OTSU_VARIANCE_PERCENTILE = 75  # 사용할 분산의 백분위수 (e.g., 75는 상위 25% 타일 선택)

# Fuzzy Otsu 매개변수
FUZZY_OTSU_SPREAD_FACTOR = 0.5  # 퍼지 함수의 확산 계수 (값이 작을수록 민감)
FUZZY_OTSU_CUTOFF = 0.9  # 물로 판단할 멤버십 임계값 (0~1 사이, 1에 가까울수록 엄격)

# --- 유틸리티 함수 ---

def load_and_clip_raster(raster_path: str, aoi_geom: list) -> Tuple[Optional[np.ndarray], Optional[rasterio.profiles.Profile]]:
    """래스터 파일을 불러와 AOI에 맞춰 자르고 NoData 영역을 NaN으로 처리합니다."""
    with rasterio.open(raster_path) as src:
        try:
            nodata_value = src.nodata
            out_image, out_transform = mask(src, aoi_geom, crop=True, nodata=nodata_value)
            out_meta = src.profile
            out_meta.update({
                "driver": "GTiff", "height": out_image.shape[1], "width": out_image.shape[2],
                "transform": out_transform, "nodata": nodata_value
            })
            squeezed_image = out_image.squeeze()
            float_image = squeezed_image.astype(float)
            
            if nodata_value is not None:
                float_image[float_image == nodata_value] = np.nan
            else:
                float_image[squeezed_image == 0] = np.nan

            if float_image.ndim == 0:
                return np.array([[float_image]]), out_meta
            return float_image, out_meta
        except ValueError as e:
            print(f"경고: AOI와 래스터 '{os.path.basename(raster_path)}'가 겹치지 않을 수 있습니다: {e}")
            return None, None

def save_raster(data: np.ndarray, meta: rasterio.profiles.Profile, output_path: str):
    """결과 래스터를 파일로 저장합니다."""
    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    save_data = data.astype(np.uint8)
    # 메타데이터의 밴드 수를 1로 고정
    meta.update({'dtype': 'uint8', 'nodata': 0, 'count': 1})
    with rasterio.open(output_path, "w", **meta) as dest:
        # 2차원 배열을 1번 밴드에 직접 씀
        dest.write(save_data, 1)
    print(f"결과 파일 저장 완료: {output_path}")

def plot_and_save_histograms(algorithm_name: str, output_dir: str, post_hist: Dict[str, Any], pre_hist: Optional[Dict[str, Any]] = None):
    """홍수 전/후 히스토그램을 하나의 이미지에 그리고 저장합니다."""
    post_data = post_hist.get("data")
    if post_data is None or post_data.size == 0:
        print(f"[{algorithm_name}] 히스토그램을 그릴 데이터가 없습니다.")
        return

    plt.figure(figsize=(12, 7))

    if pre_hist and pre_hist.get("data") is not None and pre_hist.get("data").size > 0:
        pre_data = pre_hist.get("data")
        # 두 데이터의 범위를 모두 포함하는 bin 설정
        min_val = min(np.min(pre_data) if pre_data.size > 0 else np.inf, np.min(post_data) if post_data.size > 0 else np.inf)
        max_val = max(np.max(pre_data) if pre_data.size > 0 else -np.inf, np.max(post_data) if post_data.size > 0 else -np.inf)
        bins = np.linspace(min_val, max_val, 100)
        
        plt.hist(pre_data, bins=bins, color='skyblue', alpha=0.7, label='Pre-flood')
        plt.hist(post_data, bins=bins, color='salmon', alpha=0.7, label='Post-flood')
        
        pre_thresh, post_thresh = pre_hist.get("threshold"), post_hist.get("threshold")
        pre_label, post_label = pre_hist.get("label", "Thresh"), post_hist.get("label", "Thresh")
        
        if pre_thresh is not None: plt.axvline(pre_thresh, color='blue', linestyle='--', linewidth=2, label=f'Pre-flood {pre_label}: {pre_thresh:.4f}')
        if post_thresh is not None: plt.axvline(post_thresh, color='red', linestyle='--', linewidth=2, label=f'Post-flood {post_label}: {post_thresh:.4f}')
        
        title = f'Pre vs. Post Flood Histogram for {algorithm_name}'
        file_name = f'histogram_combined_{algorithm_name}.png'
    else:
        plt.hist(post_data, bins=100, color='gray', alpha=0.7, label='Post-flood')
        post_thresh = post_hist.get("threshold")
        post_label = post_hist.get("label", "Thresh")
        if post_thresh is not None: plt.axvline(post_thresh, color='red', linestyle='--', linewidth=2, label=f'Post-flood {post_label}: {post_thresh:.4f}')
        
        title = f'Histogram for {algorithm_name} (Post-flood)'
        file_name = f'histogram_post_flood_{algorithm_name}.png'

    plt.title(title)
    plt.xlabel('SAR Backscatter (dB)')
    plt.ylabel('Frequency')
    plt.legend()
    plt.grid(True, linestyle='--', alpha=0.6)
    
    output_path = os.path.join(output_dir, file_name)
    plt.savefig(output_path)
    plt.close()
    print(f"히스토그램 저장 완료: {output_path}")

# --- 핵심 알고리즘 구현 ---

def edge_otsu_thresholding(image: np.ndarray, iterations: int, initial_threshold: Optional[float]) -> Tuple[np.ndarray, Dict[str, Any]]:
    print("Edge Otsu 알고리즘 실행 중...")
    valid_pixels = image[~np.isnan(image)]
    if valid_pixels.size == 0: return np.zeros_like(image, dtype=bool), {}

    if initial_threshold is not None:
        initial_thresh = initial_threshold
        print(f"Edge Otsu: 고정 초기 임계값({initial_thresh}) 사용.")
    else:
        initial_thresh = threshold_otsu(valid_pixels)
        print(f"Edge Otsu: 자동 계산된 초기 임계값({initial_thresh:.4f}) 사용.")

    binary_image = image < initial_thresh
    edges = canny(binary_image)
    buffered_edges = binary_dilation(edges, iterations=iterations)
    sample_pixels = image[buffered_edges & ~np.isnan(image)]

    final_thresh = threshold_otsu(sample_pixels) if sample_pixels.size > 0 else initial_thresh
    print(f"Edge Otsu 최종 임계값: {final_thresh:.4f}")
    result = image < final_thresh
    result[np.isnan(image)] = False
    return result, {"data": sample_pixels, "threshold": final_thresh, "label": "Final Threshold"}

def bmax_otsu_thresholding(image: np.ndarray, tile_size: int, variance_percentile: int, initial_threshold: Optional[float]) -> Tuple[np.ndarray, Dict[str, Any]]:
    print("Bmax Otsu 알고리즘 실행 중...")
    rows, cols = image.shape
    candidate_tiles = []
    
    for r in range(0, rows, tile_size):
        for c in range(0, cols, tile_size):
            tile = image[r:r+tile_size, c:c+tile_size]
            valid_tile_pixels = tile[~np.isnan(tile)]
            if valid_tile_pixels.size == 0: continue
            
            if initial_threshold is not None:
                if np.any(valid_tile_pixels < initial_threshold) and np.any(valid_tile_pixels > initial_threshold):
                    candidate_tiles.append(tile)
            else: candidate_tiles.append(tile)

    if not candidate_tiles:
        print("Bmax Otsu: 후보 타일이 없어 일반 Otsu 적용.")
        valid_pixels = image[~np.isnan(image)]
        final_thresh = threshold_otsu(valid_pixels)
        hist_info = {"data": valid_pixels, "threshold": final_thresh, "label": "Otsu Threshold"}
    else:
        variances = [np.nanvar(t) for t in candidate_tiles]
        variance_threshold = np.percentile(variances, variance_percentile)
        selected_pixels = [p for i, t in enumerate(candidate_tiles) if variances[i] >= variance_threshold for p in t[~np.isnan(t)].flatten()]
        hist_data = np.array(selected_pixels)
        final_thresh = threshold_otsu(hist_data) if hist_data.size > 0 else threshold_otsu(image[~np.isnan(image)])
        hist_info = {"data": hist_data, "threshold": final_thresh, "label": "Final Threshold"}

    print(f"Bmax Otsu 최종 임계값: {final_thresh:.4f}")
    result = image < final_thresh
    result[np.isnan(image)] = False
    return result, hist_info

def fuzzy_otsu_thresholding(image: np.ndarray, spread_factor: float, cutoff: float) -> Tuple[np.ndarray, Dict[str, Any]]:
    print("Fuzzy Otsu 알고리즘 실행 중...")
    valid_pixels = image[~np.isnan(image)]
    if valid_pixels.size == 0: return np.zeros_like(image, dtype=bool), {}

    global_thresh = threshold_otsu(valid_pixels)
    water_pixels, land_pixels = valid_pixels[valid_pixels < global_thresh], valid_pixels[valid_pixels >= global_thresh]

    if water_pixels.size == 0 or land_pixels.size == 0:
        result = image < global_thresh
        hist_info = {"data": valid_pixels, "threshold": global_thresh, "label": "Otsu Threshold"}
    else:
        min_center, max_center = np.mean(water_pixels), np.mean(land_pixels)
        midpoint = (min_center + max_center) / 2
        spread = (max_center - min_center) * spread_factor if (max_center - min_center) != 0 else 1
        
        fuzzy_values = 1 - np.exp(-((image - midpoint)**2) / (2 * spread**2))
        print(f"Fuzzy Otsu 중간점: {midpoint:.4f}, Spread: {spread:.4f}")
        result = fuzzy_values >= cutoff
        hist_info = {"data": valid_pixels, "threshold": midpoint, "label": "Fuzzy Midpoint"}
    
    result[np.isnan(image)] = False
    return result, hist_info

# --- 메인 실행 함수 ---

def main():
    print("홍수 탐지 분석을 시작합니다.")

    try:
        aoi_gdf = gpd.read_file(AOI_SHP_PATH)
        aoi_geometry = [geom for geom in aoi_gdf.geometry]
        print("AOI 파일 로드 완료.")
    except Exception as e:
        print(f"오류: AOI 파일('{AOI_SHP_PATH}')을 읽을 수 없습니다. {e}")
        return

    print("\n데이터 로드 및 자르기 중...")
    post_flood_img, meta = load_and_clip_raster(POST_FLOOD_IMAGE_PATH, aoi_geometry)
    if post_flood_img is None:
        print(f"오류: 홍수 후 이미지('{POST_FLOOD_IMAGE_PATH}')를 처리할 수 없습니다.")
        return
        
    pre_flood_img_raw, pre_meta_raw = load_and_clip_raster(PRE_FLOOD_IMAGE_PATH, aoi_geometry)
    slope_img_raw, slope_meta_raw = load_and_clip_raster(SLOPE_TIF_PATH, aoi_geometry)
    
    resampled_pre_flood, resampled_slope = None, None
    if pre_flood_img_raw is not None and pre_meta_raw:
        if pre_flood_img_raw.shape != post_flood_img.shape:
            print("홍수 전 이미지를 홍수 후 이미지 격자에 맞춰 재조정합니다...")
            dest = np.empty_like(post_flood_img, dtype=float)
            reproject(source=pre_flood_img_raw, destination=dest, src_transform=pre_meta_raw['transform'], src_crs=pre_meta_raw['crs'], dst_transform=meta['transform'], dst_crs=meta['crs'], resampling=Resampling.bilinear)
            resampled_pre_flood = dest
        else: resampled_pre_flood = pre_flood_img_raw

    if slope_img_raw is not None and slope_meta_raw:
        if slope_img_raw.shape != post_flood_img.shape:
            print("경사도 이미지를 홍수 후 이미지 격자에 맞춰 재조정합니다...")
            dest = np.empty_like(post_flood_img, dtype=float)
            reproject(source=slope_img_raw, destination=dest, src_transform=slope_meta_raw['transform'], src_crs=slope_meta_raw['crs'], dst_transform=meta['transform'], dst_crs=meta['crs'], resampling=Resampling.bilinear)
            resampled_slope = dest
        else: resampled_slope = slope_img_raw

    algorithms = {
        "edge_otsu": (edge_otsu_thresholding, {"iterations": EDGE_OTSU_BUFFER_ITERATIONS, "initial_threshold": EDGE_OTSU_INITIAL_THRESHOLD}),
        "bmax_otsu": (bmax_otsu_thresholding, {"tile_size": BMAX_OTSU_TILE_SIZE, "variance_percentile": BMAX_OTSU_VARIANCE_PERCENTILE, "initial_threshold": BMAX_OTSU_INITIAL_THRESHOLD}),
        "fuzzy_otsu": (fuzzy_otsu_thresholding, {"spread_factor": FUZZY_OTSU_SPREAD_FACTOR, "cutoff": FUZZY_OTSU_CUTOFF})
    }

    print("\n알고리즘 적용 및 결과 저장 중...")
    for name, (func, params) in algorithms.items():
        post_water_map, post_hist_info = func(post_flood_img, **params)
        
        slope_mask = None
        if resampled_slope is not None:
            slope_mask = resampled_slope >= SLOPE_THRESHOLD
            post_water_map[slope_mask] = False
            print(f"[{name}] 경사도 {SLOPE_THRESHOLD}도 이상 지역을 최종 결과에서 제외했습니다.")

        output_path = os.path.join(OUTPUT_DIR, f'post_flood_water_{name}.tif')
        save_raster(post_water_map, meta, output_path)

        pre_hist_info = None
        if resampled_pre_flood is not None:
            pre_water_map, pre_hist_info = func(resampled_pre_flood, **params)
            if slope_mask is not None: pre_water_map[slope_mask] = False
            
            flood_extent = post_water_map & ~pre_water_map
            output_path_change = os.path.join(OUTPUT_DIR, f'flood_change_{name}.tif')
            save_raster(flood_extent, meta, output_path_change)
            
        plot_and_save_histograms(name, OUTPUT_DIR, post_hist_info, pre_hist_info)

    print(f"\n모든 분석이 완료되었습니다. 결과는 '{OUTPUT_DIR}' 폴더에 저장되었습니다.")

if __name__ == '__main__':
    if not os.path.exists(OUTPUT_DIR):
        os.makedirs(OUTPUT_DIR)
    main()



In [ ]:
# 두 마스크 얼마나 차이 나는지

import rasterio
import numpy as np

file1 = r"<DATA_ROOT>\07_disaster\flood\Korea\s1_pipeline_test6\03_output_6\post_flood_water_edge_otsu.tif"
file2 = r"<DATA_ROOT>\07_disaster\flood\Korea\202507_Jinju\process_python\S1A_20250719_slopeFiltered_binaryMask.tif"

with rasterio.open(file1) as src1, rasterio.open(file2) as src2:
    arr1 = src1.read(1)
    arr2 = src2.read(1)

    diff = arr1 - arr2
    y_idx, x_idx = np.where(diff != 0)

    # 추가: 총 픽셀 수 / 차이 픽셀 수 / 퍼센트
    total_pixels = arr1.size
    diff_count = len(y_idx)
    percent = (diff_count / total_pixels * 100) if total_pixels else 0.0

    print(f"총 픽셀 수: {total_pixels}")
    print(f"차이 픽셀 수: {diff_count}")
    print(f"차이 비율: {percent:.6f}%")

    # 기존: 차이가 나는 픽셀 10개만 예시 출력
    print(f"총 {diff_count} 개의 픽셀에서 값이 다릅니다.")
    for i in range(min(10, diff_count)):
        y, x = y_idx[i], x_idx[i]
        print(f"좌표 (행={y}, 열={x}) → file1={arr1[y, x]}, file2={arr2[y, x]}")


In [ ]:
# AOI 영역 내에서 Edge Otsu, Bmax Otsu, Fuzzy Otsu 기법 적용하고 히스토그램 저장
# + Edge/Bmax 초기 임계값(0 ~ -15 dB) 스윕 기능 포함

import rasterio
from rasterio.mask import mask
from rasterio.warp import reproject, Resampling
import geopandas as gpd
import numpy as np
from skimage.filters import threshold_otsu
from skimage.feature import canny
from scipy.ndimage import binary_dilation
import os
from typing import Tuple, Optional, Dict, Any
import matplotlib.pyplot as plt

# --- 사용자 설정 변수 (파일 경로) ---
PRE_FLOOD_IMAGE_PATH = r'<DATA_ROOT>\07_disaster\flood\Korea\s1_pipeline_test6\02_preprocess\S1A_IW_GRDH_1SDV_20250712T093124_20250712T093154_060049_0775E2_5D79_VV_processed.tif'
POST_FLOOD_IMAGE_PATH = r'<DATA_ROOT>\07_disaster\flood\Korea\s1_pipeline_test6\02_preprocess\S1A_IW_GRDH_1SDV_20250719T092320_20250719T092350_060151_07795E_0ABD_VV_processed.tif'
AOI_SHP_PATH = r"<DATA_ROOT>\07_disaster\flood\Korea\202507_Jinju\shp\AOI.shp"
SLOPE_TIF_PATH = r"<DATA_ROOT>\07_disaster\flood\Korea\202507_Jinju\dem\slope.tif"
OUTPUT_DIR = r'<DATA_ROOT>\07_disaster\flood\Korea\s1_pipeline_test6\05_output'

# --- 알고리즘 및 필터링 매개변수 (조절 가능) ---
SLOPE_THRESHOLD = 15  # 이 경사도(도) 이상인 지역은 최종 결과에서 제외됩니다.

# Edge Otsu 매개변수
EDGE_OTSU_INITIAL_THRESHOLD = -15  # 기본값(단일 실행 시), 스윕에서는 무시되고 루프 값이 주입됨
EDGE_OTSU_BUFFER_ITERATIONS = 5    # 경계선 버퍼링 반복 횟수 (값이 클수록 넓은 영역 샘플링)

# Bmax Otsu 매개변수
BMAX_OTSU_INITIAL_THRESHOLD = -15  # 기본값(단일 실행 시), 스윕에서는 무시되고 루프 값이 주입됨
BMAX_OTSU_TILE_SIZE = 64           # 분산을 계산할 타일 크기 (픽셀 단위)
BMAX_OTSU_VARIANCE_PERCENTILE = 75 # 사용할 분산의 백분위수 (예: 75는 상위 25% 타일 선택)

# Fuzzy Otsu 매개변수
FUZZY_OTSU_SPREAD_FACTOR = 0.5  # 퍼지 함수의 확산 계수 (값이 작을수록 민감)
FUZZY_OTSU_CUTOFF = 0.9         # 물로 판단할 멤버십 임계값 (0~1 사이, 1에 가까울수록 엄격)

# --- 초기 임계값 스윕 범위 설정(0, -1, ..., -15) ---
SWEEP_INITIALS = list(range(0, -16, -1))

# --- 유틸리티 함수 ---

def load_and_clip_raster(raster_path: str, aoi_geom: list) -> Tuple[Optional[np.ndarray], Optional[rasterio.profiles.Profile]]:
    """래스터 파일을 불러와 AOI에 맞춰 자르고 NoData 영역을 NaN으로 처리합니다."""
    with rasterio.open(raster_path) as src:
        try:
            nodata_value = src.nodata
            out_image, out_transform = mask(src, aoi_geom, crop=True, nodata=nodata_value)
            out_meta = src.profile
            out_meta.update({
                "driver": "GTiff", "height": out_image.shape[1], "width": out_image.shape[2],
                "transform": out_transform, "nodata": nodata_value
            })
            squeezed_image = out_image.squeeze()
            float_image = squeezed_image.astype(float)

            if nodata_value is not None:
                float_image[float_image == nodata_value] = np.nan
            else:
                float_image[squeezed_image == 0] = np.nan

            if float_image.ndim == 0:
                return np.array([[float_image]]), out_meta
            return float_image, out_meta
        except ValueError as e:
            print(f"경고: AOI와 래스터 '{os.path.basename(raster_path)}'가 겹치지 않을 수 있습니다: {e}")
            return None, None

def save_raster(data: np.ndarray, meta: rasterio.profiles.Profile, output_path: str):
    """결과 래스터를 파일로 저장합니다."""
    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    save_data = data.astype(np.uint8)  # bool -> uint8 (True=1, False=0)
    # 메타데이터의 밴드 수를 1로 고정
    meta = meta.copy()
    meta.update({'dtype': 'uint8', 'nodata': 0, 'count': 1})
    with rasterio.open(output_path, "w", **meta) as dest:
        dest.write(save_data, 1)
    print(f"결과 파일 저장 완료: {output_path}")

def plot_and_save_histograms(algorithm_name: str, output_dir: str, post_hist: Dict[str, Any], pre_hist: Optional[Dict[str, Any]] = None):
    """홍수 전/후 히스토그램을 하나의 이미지에 그리고 저장합니다."""
    post_data = post_hist.get("data") if post_hist else None
    if post_data is None or post_data.size == 0:
        print(f"[{algorithm_name}] 히스토그램을 그릴 데이터가 없습니다.")
        return

    os.makedirs(output_dir, exist_ok=True)
    plt.figure(figsize=(12, 7))

    if pre_hist and pre_hist.get("data") is not None and pre_hist.get("data").size > 0:
        pre_data = pre_hist.get("data")
        # 두 데이터의 범위를 모두 포함하는 bin 설정
        min_val = min(np.min(pre_data) if pre_data.size > 0 else np.inf,
                      np.min(post_data) if post_data.size > 0 else np.inf)
        max_val = max(np.max(pre_data) if pre_data.size > 0 else -np.inf,
                      np.max(post_data) if post_data.size > 0 else -np.inf)
        bins = np.linspace(min_val, max_val, 100)

        plt.hist(pre_data, bins=bins, color='skyblue', alpha=0.7, label='Pre-flood')
        plt.hist(post_data, bins=bins, color='salmon',  alpha=0.7, label='Post-flood')

        pre_thresh, post_thresh = pre_hist.get("threshold"), post_hist.get("threshold")
        pre_label,  post_label  = pre_hist.get("label", "Thresh"), post_hist.get("label", "Thresh")

        if pre_thresh is not None:
            plt.axvline(pre_thresh,  color='blue', linestyle='--', linewidth=2, label=f'Pre-flood {pre_label}: {pre_thresh:.4f}')
        if post_thresh is not None:
            plt.axvline(post_thresh, color='red',  linestyle='--', linewidth=2, label=f'Post-flood {post_label}: {post_thresh:.4f}')

        title = f'Pre vs. Post Flood Histogram for {algorithm_name}'
        file_name = f'histogram_combined_{algorithm_name}.png'
    else:
        plt.hist(post_data, bins=100, color='gray', alpha=0.7, label='Post-flood')
        post_thresh = post_hist.get("threshold")
        post_label  = post_hist.get("label", "Thresh")
        if post_thresh is not None:
            plt.axvline(post_thresh, color='red', linestyle='--', linewidth=2,
                        label=f'Post-flood {post_label}: {post_thresh:.4f}')
        title = f'Histogram for {algorithm_name} (Post-flood)'
        file_name = f'histogram_post_flood_{algorithm_name}.png'

    plt.title(title)
    plt.xlabel('SAR Backscatter (dB)')
    plt.ylabel('Frequency')
    plt.legend()
    plt.grid(True, linestyle='--', alpha=0.6)

    output_path = os.path.join(output_dir, file_name)
    plt.savefig(output_path)
    plt.close()
    print(f"히스토그램 저장 완료: {output_path}")

# --- 핵심 알고리즘 구현 ---

def edge_otsu_thresholding(image: np.ndarray, iterations: int, initial_threshold: Optional[float]) -> Tuple[np.ndarray, Dict[str, Any]]:
    print("Edge Otsu 알고리즘 실행 중...")
    valid_pixels = image[~np.isnan(image)]
    if valid_pixels.size == 0:
        return np.zeros_like(image, dtype=bool), {}

    if initial_threshold is not None:
        initial_thresh = float(initial_threshold)
        print(f"Edge Otsu: 고정 초기 임계값({initial_thresh}) 사용.")
    else:
        initial_thresh = threshold_otsu(valid_pixels)
        print(f"Edge Otsu: 자동 계산된 초기 임계값({initial_thresh:.4f}) 사용.")

    binary_image = image < initial_thresh
    edges = canny(binary_image.astype(float))
    buffered_edges = binary_dilation(edges, iterations=iterations)
    sample_pixels = image[buffered_edges & ~np.isnan(image)]

    final_thresh = threshold_otsu(sample_pixels) if sample_pixels.size > 0 else initial_thresh
    print(f"Edge Otsu 최종 임계값: {final_thresh:.4f}")
    result = image < final_thresh
    result[np.isnan(image)] = False
    return result, {"data": sample_pixels if sample_pixels.size > 0 else valid_pixels,
                    "threshold": final_thresh, "label": "Final Threshold"}

def bmax_otsu_thresholding(image: np.ndarray, tile_size: int, variance_percentile: int, initial_threshold: Optional[float]) -> Tuple[np.ndarray, Dict[str, Any]]:
    print("Bmax Otsu 알고리즘 실행 중...")
    rows, cols = image.shape
    candidate_tiles = []

    for r in range(0, rows, tile_size):
        for c in range(0, cols, tile_size):
            tile = image[r:r+tile_size, c:c+tile_size]
            valid_tile_pixels = tile[~np.isnan(tile)]
            if valid_tile_pixels.size == 0:
                continue

            if initial_threshold is not None:
                t0 = float(initial_threshold)
                # 초기 임계값을 가르는(양쪽 모두 존재) 타일만 후보로
                if np.any(valid_tile_pixels < t0) and np.any(valid_tile_pixels > t0):
                    candidate_tiles.append(tile)
            else:
                candidate_tiles.append(tile)

    if not candidate_tiles:
        print("Bmax Otsu: 후보 타일이 없어 일반 Otsu 적용.")
        valid_pixels = image[~np.isnan(image)]
        final_thresh = threshold_otsu(valid_pixels) if valid_pixels.size > 0 else np.nan
        hist_info = {"data": valid_pixels, "threshold": final_thresh, "label": "Otsu Threshold"}
    else:
        variances = [np.nanvar(t) for t in candidate_tiles]
        variance_threshold = np.percentile(variances, variance_percentile)
        selected_pixels = []
        for i, t in enumerate(candidate_tiles):
            if variances[i] >= variance_threshold:
                selected_pixels.extend(t[~np.isnan(t)].ravel().tolist())
        hist_data = np.array(selected_pixels)
        if hist_data.size == 0:
            valid_pixels = image[~np.isnan(image)]
            final_thresh = threshold_otsu(valid_pixels) if valid_pixels.size > 0 else np.nan
            hist_info = {"data": valid_pixels, "threshold": final_thresh, "label": "Otsu Threshold"}
        else:
            final_thresh = threshold_otsu(hist_data)
            hist_info = {"data": hist_data, "threshold": final_thresh, "label": "Final Threshold"}

    print(f"Bmax Otsu 최종 임계값: {final_thresh:.4f}")
    result = image < final_thresh
    result[np.isnan(image)] = False
    return result, hist_info

def fuzzy_otsu_thresholding(image: np.ndarray, spread_factor: float, cutoff: float) -> Tuple[np.ndarray, Dict[str, Any]]:
    print("Fuzzy Otsu 알고리즘 실행 중...")
    valid_pixels = image[~np.isnan(image)]
    if valid_pixels.size == 0:
        return np.zeros_like(image, dtype=bool), {}

    global_thresh = threshold_otsu(valid_pixels)
    water_pixels = valid_pixels[valid_pixels < global_thresh]
    land_pixels  = valid_pixels[valid_pixels >= global_thresh]

    if water_pixels.size == 0 or land_pixels.size == 0:
        result = image < global_thresh
        hist_info = {"data": valid_pixels, "threshold": global_thresh, "label": "Otsu Threshold"}
    else:
        min_center = np.mean(water_pixels)
        max_center = np.mean(land_pixels)
        midpoint = (min_center + max_center) / 2
        spread = (max_center - min_center) * spread_factor if (max_center - min_center) != 0 else 1.0

        fuzzy_values = 1 - np.exp(-((image - midpoint) ** 2) / (2 * spread ** 2))
        print(f"Fuzzy Otsu 중간점: {midpoint:.4f}, Spread: {spread:.4f}")
        result = fuzzy_values >= cutoff
        hist_info = {"data": valid_pixels, "threshold": midpoint, "label": "Fuzzy Midpoint"}

    result[np.isnan(image)] = False
    return result, hist_info

# --- 메인 실행 함수 ---

def main():
    print("홍수 탐지 분석을 시작합니다.")

    # 1) AOI 로드
    try:
        aoi_gdf = gpd.read_file(AOI_SHP_PATH)
        aoi_geometry = [geom for geom in aoi_gdf.geometry]  # Shapely geometry list
        print("AOI 파일 로드 완료.")
    except Exception as e:
        print(f"오류: AOI 파일('{AOI_SHP_PATH}')을 읽을 수 없습니다. {e}")
        return

    # 2) 데이터 로드/클립 + 격자 맞춤
    print("\n데이터 로드 및 자르기 중...")
    post_flood_img, meta = load_and_clip_raster(POST_FLOOD_IMAGE_PATH, aoi_geometry)
    if post_flood_img is None:
        print(f"오류: 홍수 후 이미지('{POST_FLOOD_IMAGE_PATH}')를 처리할 수 없습니다.")
        return

    pre_flood_img_raw, pre_meta_raw = load_and_clip_raster(PRE_FLOOD_IMAGE_PATH, aoi_geometry)
    slope_img_raw, slope_meta_raw = load_and_clip_raster(SLOPE_TIF_PATH, aoi_geometry)

    resampled_pre_flood, resampled_slope = None, None

    if pre_flood_img_raw is not None and pre_meta_raw:
        if pre_flood_img_raw.shape != post_flood_img.shape:
            print("홍수 전 이미지를 홍수 후 이미지 격자에 맞춰 재조정합니다...")
            dest = np.empty_like(post_flood_img, dtype=float)
            reproject(
                source=pre_flood_img_raw, destination=dest,
                src_transform=pre_meta_raw['transform'], src_crs=pre_meta_raw['crs'],
                dst_transform=meta['transform'], dst_crs=meta['crs'],
                resampling=Resampling.bilinear
            )
            resampled_pre_flood = dest
        else:
            resampled_pre_flood = pre_flood_img_raw

    if slope_img_raw is not None and slope_meta_raw:
        if slope_img_raw.shape != post_flood_img.shape:
            print("경사도 이미지를 홍수 후 이미지 격자에 맞춰 재조정합니다...")
            dest = np.empty_like(post_flood_img, dtype=float)
            reproject(
                source=slope_img_raw, destination=dest,
                src_transform=slope_meta_raw['transform'], src_crs=slope_meta_raw['crs'],
                dst_transform=meta['transform'], dst_crs=meta['crs'],
                resampling=Resampling.bilinear
            )
            resampled_slope = dest
        else:
            resampled_slope = slope_img_raw

    # slope_mask 한 번만 만들어 재사용
    slope_mask = None
    if resampled_slope is not None:
        slope_mask = resampled_slope >= SLOPE_THRESHOLD

    # 3) Fuzzy Otsu는 초기 임계값과 무관하므로 1회만 실행 (기본 OUTPUT_DIR에 저장)
    print("\n[Fuzzy Otsu] 1회 실행 및 저장...")
    fuzzy_map, fuzzy_post_hist = fuzzy_otsu_thresholding(
        post_flood_img,
        spread_factor=FUZZY_OTSU_SPREAD_FACTOR,
        cutoff=FUZZY_OTSU_CUTOFF
    )
    if slope_mask is not None:
        fuzzy_map[slope_mask] = False
        print(f"[fuzzy_otsu] 경사도 {SLOPE_THRESHOLD}도 이상 지역을 최종 결과에서 제외했습니다.")

    save_raster(fuzzy_map, meta, os.path.join(OUTPUT_DIR, 'post_flood_water_fuzzy_otsu.tif'))

    fuzzy_pre_hist = None
    if resampled_pre_flood is not None:
        fuzzy_pre_map, fuzzy_pre_hist = fuzzy_otsu_thresholding(
            resampled_pre_flood,
            spread_factor=FUZZY_OTSU_SPREAD_FACTOR,
            cutoff=FUZZY_OTSU_CUTOFF
        )
        if slope_mask is not None:
            fuzzy_pre_map[slope_mask] = False
        fuzzy_change = fuzzy_map & ~fuzzy_pre_map
        save_raster(fuzzy_change, meta, os.path.join(OUTPUT_DIR, 'flood_change_fuzzy_otsu.tif'))

    plot_and_save_histograms("fuzzy_otsu", OUTPUT_DIR, fuzzy_post_hist, fuzzy_pre_hist)

    # 4) Edge Otsu, Bmax Otsu는 초기 임계값 스윕 (각 임계값별 서브폴더에 저장)
    print("\n[Edge/Bmax Otsu] 임계값 스윕 시작...")
    for init in SWEEP_INITIALS:  # 0, -1, ..., -15
        print(f"\n=== 초기 임계값 스윕: {init} dB ===")
        sweep_dir = os.path.join(OUTPUT_DIR, f'init_{init}')
        os.makedirs(sweep_dir, exist_ok=True)

        # 스윕용 알고리즘 설정 (해당 init 값 주입)
        algorithms = {
            "edge_otsu": (
                edge_otsu_thresholding,
                {"iterations": EDGE_OTSU_BUFFER_ITERATIONS, "initial_threshold": float(init)}
            ),
            "bmax_otsu": (
                bmax_otsu_thresholding,
                {
                    "tile_size": BMAX_OTSU_TILE_SIZE,
                    "variance_percentile": BMAX_OTSU_VARIANCE_PERCENTILE,
                    "initial_threshold": float(init)
                }
            )
        }

        # 알고리즘 적용
        for name, (func, params) in algorithms.items():
            # Post
            post_water_map, post_hist_info = func(post_flood_img, **params)
            if slope_mask is not None:
                post_water_map[slope_mask] = False
                print(f"[{name} / init {init}] 경사도 {SLOPE_THRESHOLD}도 이상 제외")

            save_raster(
                post_water_map, meta,
                os.path.join(sweep_dir, f'post_flood_water_{name}_init{init}.tif')
            )

            # Pre & 변화
            pre_hist_info = None
            if resampled_pre_flood is not None:
                pre_water_map, pre_hist_info = func(resampled_pre_flood, **params)
                if slope_mask is not None:
                    pre_water_map[slope_mask] = False

                flood_extent = post_water_map & ~pre_water_map
                save_raster(
                    flood_extent, meta,
                    os.path.join(sweep_dir, f'flood_change_{name}_init{init}.tif')
                )

            # 히스토그램(파일명에 init 값 포함되게 이름에 태그 포함)
            algo_tag = f"{name}_init{init}"
            plot_and_save_histograms(algo_tag, sweep_dir, post_hist_info, pre_hist_info)

    print(f"\n모든 분석이 완료되었습니다. 결과는 '{OUTPUT_DIR}' 및 하위 'init_#' 폴더에 저장되었습니다.")

if __name__ == '__main__':
    if not os.path.exists(OUTPUT_DIR):
        os.makedirs(OUTPUT_DIR)
    main()


In [ ]:
# ==============================================================================
# HYDRAFloods 민감도 분석 스크립트 (Jupyter Notebook용) - 최종 수정본
#
# 기능:
# 1. 초기 임계값을 0부터 -15까지 자동으로 변경하며 홍수 탐지 알고리즘 실행
# 2. 각 결과물을 정답(Ground Truth) 파일과 비교하여 정확한 차이 비율 계산
# 3. 각 알고리즘별 홍수 전/후 비교 히스토그램을 이미지 파일로 저장
# 4. 최종적으로 가장 성능이 좋았던 최적의 임계값을 찾아 요약 리포트 출력
# ==============================================================================

import rasterio
from rasterio.mask import mask
from rasterio.warp import reproject, Resampling
import geopandas as gpd
import numpy as np
from skimage.filters import threshold_otsu
from skimage.feature import canny
from scipy.ndimage import binary_dilation
import os
import pandas as pd
from typing import Tuple, Optional, Dict, Any
import matplotlib.pyplot as plt

# --- 사용자 설정 변수 (이 부분만 수정하세요) ---

# 1. 파일 경로
PRE_FLOOD_IMAGE_PATH = r'<DATA_ROOT>\07_disaster\flood\Korea\s1_pipeline_test6\02_preprocess\S1A_IW_GRDH_1SDV_20250712T093124_20250712T093154_060049_0775E2_5D79_VV_processed.tif'
POST_FLOOD_IMAGE_PATH = r'<DATA_ROOT>\07_disaster\flood\Korea\s1_pipeline_test6\02_preprocess\S1A_IW_GRDH_1SDV_20250719T092320_20250719T092350_060151_07795E_0ABD_VV_processed.tif'
AOI_SHP_PATH = r"<DATA_ROOT>\07_disaster\flood\Korea\202507_Jinju\shp\AOI.shp"
SLOPE_TIF_PATH = r"<DATA_ROOT>\07_disaster\flood\Korea\202507_Jinju\dem\slope.tif"
GROUND_TRUTH_PATH = r"<DATA_ROOT>\07_disaster\flood\Korea\202507_Jinju\process_python\S1A_20250719_slopeFiltered_binaryMask.tif"
BASE_OUTPUT_DIR = r'<DATA_ROOT>\07_disaster\flood\Korea\s1_pipeline_test6\sensitivity_analysis_results'

# 2. 테스트할 초기 임계값 범위
THRESHOLDS_TO_TEST = range(0, -40, -1)  # 0, -1, -2, ..., -15

# 3. 고정 알고리즘 매개변수
SLOPE_THRESHOLD = 15
EDGE_OTSU_BUFFER_ITERATIONS = 5
BMAX_OTSU_TILE_SIZE = 64
BMAX_OTSU_VARIANCE_PERCENTILE = 75
FUZZY_OTSU_SPREAD_FACTOR = 0.5
FUZZY_OTSU_CUTOFF = 0.9

# --- 유틸리티 및 알고리즘 함수 (수정 불필요) ---

def load_and_clip_raster(raster_path: str, aoi_geom: list) -> Tuple[Optional[np.ndarray], Optional[rasterio.profiles.Profile]]:
    with rasterio.open(raster_path) as src:
        try:
            nodata_value = src.nodata if src.nodata is not None else 0
            out_image, out_transform = mask(src, aoi_geom, crop=True, nodata=nodata_value, filled=True)
            out_meta = src.profile
            out_meta.update({"driver": "GTiff", "height": out_image.shape[1], "width": out_image.shape[2], "transform": out_transform, "nodata": nodata_value})
            squeezed_image = out_image.squeeze()
            float_image = squeezed_image.astype(float)
            float_image[squeezed_image == nodata_value] = np.nan
            if float_image.ndim == 0: return np.array([[float_image]]), out_meta
            return float_image, out_meta
        except (ValueError, rasterio.errors.RasterioIOError): return None, None

def save_raster(data: np.ndarray, meta: rasterio.profiles.Profile, output_path: str, nan_mask: np.ndarray):
    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    save_data = data.astype(np.uint8)
    save_data[nan_mask] = 255 # NoData 영역을 255로 설정
    meta.update({'dtype': 'uint8', 'nodata': 255, 'count': 1})
    with rasterio.open(output_path, "w", **meta) as dest:
        dest.write(save_data, 1)

def plot_and_save_histograms(algorithm_name: str, output_dir: str, post_hist: Dict[str, Any], pre_hist: Optional[Dict[str, Any]] = None):
    post_data = post_hist.get("data")
    if post_data is None or post_data.size == 0: return
    plt.figure(figsize=(12, 7))
    if pre_hist and pre_hist.get("data") is not None and pre_hist.get("data").size > 0:
        pre_data = pre_hist.get("data")
        min_val = min(np.min(pre_data) if pre_data.size > 0 else np.inf, np.min(post_data) if post_data.size > 0 else np.inf)
        max_val = max(np.max(pre_data) if pre_data.size > 0 else -np.inf, np.max(post_data) if post_data.size > 0 else -np.inf)
        bins = np.linspace(min_val, max_val, 100)
        plt.hist(pre_data, bins=bins, color='skyblue', alpha=0.7, label='Pre-flood')
        plt.hist(post_data, bins=bins, color='salmon', alpha=0.7, label='Post-flood')
        pre_thresh, post_thresh = pre_hist.get("threshold"), post_hist.get("threshold")
        pre_label, post_label = pre_hist.get("label", "Thresh"), post_hist.get("label", "Thresh")
        if pre_thresh is not None: plt.axvline(pre_thresh, color='blue', linestyle='--', linewidth=2, label=f'Pre-flood {pre_label}: {pre_thresh:.4f}')
        if post_thresh is not None: plt.axvline(post_thresh, color='red', linestyle='--', linewidth=2, label=f'Post-flood {post_label}: {post_thresh:.4f}')
        title, file_name = f'Pre vs. Post Flood Histogram for {algorithm_name}', f'histogram_combined_{algorithm_name}.png'
    else:
        plt.hist(post_data, bins=100, color='gray', alpha=0.7, label='Post-flood')
        post_thresh, post_label = post_hist.get("threshold"), post_hist.get("label", "Thresh")
        if post_thresh is not None: plt.axvline(post_thresh, color='red', linestyle='--', linewidth=2, label=f'Post-flood {post_label}: {post_thresh:.4f}')
        title, file_name = f'Histogram for {algorithm_name} (Post-flood)', f'histogram_post_flood_{algorithm_name}.png'
    plt.title(title)
    plt.xlabel('SAR Backscatter (dB)')
    plt.ylabel('Frequency')
    plt.legend()
    plt.grid(True, linestyle='--', alpha=0.6)
    plt.savefig(os.path.join(output_dir, file_name))
    plt.close()

def edge_otsu_thresholding(image: np.ndarray, iterations: int, initial_threshold: Optional[float]) -> Tuple[np.ndarray, Dict[str, Any]]:
    valid_pixels = image[~np.isnan(image)]
    if valid_pixels.size == 0: return np.zeros_like(image, dtype=bool), {}
    initial_thresh = initial_threshold if initial_threshold is not None else threshold_otsu(valid_pixels)
    binary_image = image < initial_thresh
    edges = canny(binary_image)
    buffered_edges = binary_dilation(edges, iterations=iterations)
    sample_pixels = image[buffered_edges & ~np.isnan(image)]
    final_thresh = threshold_otsu(sample_pixels) if sample_pixels.size > 0 else initial_thresh
    result = image < final_thresh
    return result, {"data": sample_pixels, "threshold": final_thresh, "label": "Final Threshold"}

def bmax_otsu_thresholding(image: np.ndarray, tile_size: int, variance_percentile: int, initial_threshold: Optional[float]) -> Tuple[np.ndarray, Dict[str, Any]]:
    rows, cols = image.shape
    candidate_tiles = [image[r:r+tile_size, c:c+tile_size] for r in range(0, rows, tile_size) for c in range(0, cols, tile_size)]
    candidate_tiles = [t for t in candidate_tiles if t[~np.isnan(t)].size > 0]
    if initial_threshold is not None:
        candidate_tiles = [t for t in candidate_tiles if np.any(t[~np.isnan(t)] < initial_threshold) and np.any(t[~np.isnan(t)] > initial_threshold)]
    if not candidate_tiles:
        valid_pixels = image[~np.isnan(image)]
        final_thresh = threshold_otsu(valid_pixels)
        hist_info = {"data": valid_pixels, "threshold": final_thresh, "label": "Otsu Threshold"}
    else:
        variances = [np.nanvar(t) for t in candidate_tiles]
        variance_threshold = np.percentile(variances, variance_percentile)
        selected_pixels = [p for i, t in enumerate(candidate_tiles) if variances[i] >= variance_threshold for p in t[~np.isnan(t)].flatten()]
        hist_data = np.array(selected_pixels)
        final_thresh = threshold_otsu(hist_data) if hist_data.size > 0 else threshold_otsu(image[~np.isnan(image)])
        hist_info = {"data": hist_data, "threshold": final_thresh, "label": "Final Threshold"}
    result = image < final_thresh
    return result, hist_info

def fuzzy_otsu_thresholding(image: np.ndarray, spread_factor: float, cutoff: float) -> Tuple[np.ndarray, Dict[str, Any]]:
    valid_pixels = image[~np.isnan(image)]
    if valid_pixels.size == 0: return np.zeros_like(image, dtype=bool), {}
    global_thresh = threshold_otsu(valid_pixels)
    water_pixels, land_pixels = valid_pixels[valid_pixels < global_thresh], valid_pixels[valid_pixels >= global_thresh]
    if water_pixels.size == 0 or land_pixels.size == 0:
        result, hist_info = image < global_thresh, {"data": valid_pixels, "threshold": global_thresh, "label": "Otsu Threshold"}
    else:
        min_center, max_center = np.mean(water_pixels), np.mean(land_pixels)
        midpoint = (min_center + max_center) / 2
        spread = (max_center - min_center) * spread_factor if (max_center - min_center) != 0 else 1
        fuzzy_values = 1 - np.exp(-((image - midpoint)**2) / (2 * spread**2))
        result, hist_info = fuzzy_values >= cutoff, {"data": valid_pixels, "threshold": midpoint, "label": "Fuzzy Midpoint"}
    return result, hist_info

def compare_rasters(result_file_path: str, truth_file_path: str) -> dict:
    if not os.path.exists(result_file_path) or not os.path.exists(truth_file_path): return {"error": "File not found"}
    with rasterio.open(result_file_path) as src_result, rasterio.open(truth_file_path) as src_truth:
        # 결과 파일 읽기 (1=물, 0=땅, 255=NoData)
        arr_result = src_result.read(1)
        
        # 정답 파일을 결과 파일의 격자에 맞춰 재조정
        arr_truth_reprojected = np.empty_like(arr_result, dtype=src_truth.dtypes[0])
        reproject(
            source=rasterio.band(src_truth, 1),
            destination=arr_truth_reprojected,
            src_transform=src_truth.transform, src_crs=src_truth.crs, src_nodata=src_truth.nodata,
            dst_transform=src_result.transform, dst_crs=src_result.crs, dst_nodata=src_truth.nodata,
            resampling=Resampling.nearest
        )
        
        # 정답 파일의 NoData를 땅(0)으로 변환
        # 정답 파일은 1=물, NoData=땅 이므로, 1이 아닌 모든 값을 0으로 변환
        arr_truth_final = np.where(arr_truth_reprojected == 1, 1, 0)
        
        # 비교 영역 설정: 결과 파일에서 NoData(255)가 아닌 영역만
        valid_mask = arr_result != 255
        
        arr1_valid = arr_result[valid_mask]
        arr2_valid = arr_truth_final[valid_mask]
        
        total, diff = arr1_valid.size, np.sum(arr1_valid != arr2_valid)
        return {"total_pixels": total, "diff_pixels": diff, "diff_percent": (diff / total * 100) if total else 0.0}

def run_single_analysis(params: Dict[str, Any]):
    aoi_gdf = gpd.read_file(params["aoi_shp"])
    aoi_geometry = [geom for geom in aoi_gdf.geometry]
    post_flood_img, meta = load_and_clip_raster(params["post_flood_tif"], aoi_geometry)
    if post_flood_img is None: return
    
    nan_mask = np.isnan(post_flood_img)
    
    pre_flood_img_raw, pre_meta_raw = load_and_clip_raster(params["pre_flood_tif"], aoi_geometry)
    slope_img_raw, slope_meta_raw = load_and_clip_raster(params["slope_tif"], aoi_geometry)
    resampled_pre_flood, resampled_slope = None, None
    if pre_flood_img_raw is not None and pre_meta_raw:
        dest = np.empty_like(post_flood_img, dtype=float)
        reproject(source=pre_flood_img_raw, destination=dest, src_transform=pre_meta_raw['transform'], src_crs=pre_meta_raw['crs'], dst_transform=meta['transform'], dst_crs=meta['crs'], resampling=Resampling.bilinear)
        resampled_pre_flood = dest
    if slope_img_raw is not None and slope_meta_raw:
        dest = np.empty_like(post_flood_img, dtype=float)
        reproject(source=slope_img_raw, destination=dest, src_transform=slope_meta_raw['transform'], src_crs=slope_meta_raw['crs'], dst_transform=meta['transform'], dst_crs=meta['crs'], resampling=Resampling.bilinear)
        resampled_slope = dest
    
    algorithms = {
        "edge_otsu": (edge_otsu_thresholding, {"iterations": params["edge_buffer"], "initial_threshold": params["edge_thresh"]}),
        "bmax_otsu": (bmax_otsu_thresholding, {"tile_size": params["bmax_tile_size"], "variance_percentile": params["bmax_percentile"], "initial_threshold": params["bmax_thresh"]}),
        "fuzzy_otsu": (fuzzy_otsu_thresholding, {"spread_factor": params["fuzzy_spread"], "cutoff": params["fuzzy_cutoff"]})
    }
    
    for name, (func, func_params) in algorithms.items():
        water_map, hist_info = func(post_flood_img, **func_params)
        water_map[nan_mask] = False
        
        slope_mask = None
        if resampled_slope is not None:
            slope_mask = resampled_slope >= params["slope_thresh"]
            water_map[slope_mask] = False
        save_raster(water_map, meta, os.path.join(params["output_dir"], f'post_flood_water_{name}.tif'), nan_mask)
        
        pre_hist_info = None
        if resampled_pre_flood is not None:
            pre_water_map, pre_hist_info = func(resampled_pre_flood, **func_params)
            pre_water_map[nan_mask] = False
            if slope_mask is not None: pre_water_map[slope_mask] = False
            flood_extent = water_map & ~pre_water_map
            save_raster(flood_extent, meta, os.path.join(params["output_dir"], f'flood_change_{name}.tif'), nan_mask)
        plot_and_save_histograms(name, params["output_dir"], hist_info, pre_hist_info)

# ==============================================================================
# --- 자동화된 민감도 분석 실행 ---
# ==============================================================================
results_log = []
for threshold in THRESHOLDS_TO_TEST:
    print(f"\n{'='*20} 테스트 시작: 초기 임계값 = {threshold} {'='*20}")
    current_output_dir = os.path.join(BASE_OUTPUT_DIR, f"thresh_{threshold}")
    
    params = {
        "pre_flood_tif": PRE_FLOOD_IMAGE_PATH, "post_flood_tif": POST_FLOOD_IMAGE_PATH,
        "aoi_shp": AOI_SHP_PATH, "slope_tif": SLOPE_TIF_PATH, "output_dir": current_output_dir,
        "slope_thresh": SLOPE_THRESHOLD, "edge_buffer": EDGE_OTSU_BUFFER_ITERATIONS,
        "bmax_tile_size": BMAX_OTSU_TILE_SIZE, "bmax_percentile": BMAX_OTSU_VARIANCE_PERCENTILE,
        "fuzzy_spread": FUZZY_OTSU_SPREAD_FACTOR, "fuzzy_cutoff": FUZZY_OTSU_CUTOFF,
        "edge_thresh": threshold, "bmax_thresh": threshold
    }
    
    run_single_analysis(params)
    
    print("\n--- 결과 비교 ---")
    for algorithm in ["edge_otsu", "bmax_otsu", "fuzzy_otsu"]:
        result_file = os.path.join(current_output_dir, f"post_flood_water_{algorithm}.tif")
        comparison = compare_rasters(result_file, GROUND_TRUTH_PATH)
        if "error" in comparison: continue
        print(f"[{algorithm.upper():<10}] 차이 비율: {comparison['diff_percent']:.4f}%")
        results_log.append({"threshold": threshold, "algorithm": algorithm, "diff_percent": comparison['diff_percent']})

# --- 최종 결과 요약 ---
if results_log:
    print(f"\n{'='*25} 최종 분석 결과 요약 {'='*25}")
    df = pd.DataFrame(results_log)
    if not df.empty:
        print("\n--- 전체 결과 ---")
        print(df.to_string())

        print("\n--- 최적 임계값 ---")
        for algo_name in ["edge_otsu", "bmax_otsu", "fuzzy_otsu"]:
            algo_df = df[df['algorithm'] == algo_name]
            if not algo_df.empty:
                best_row = algo_df.loc[algo_df['diff_percent'].idxmin()]
                print(f"[{algo_name.upper():<10}] 최적 임계값: {best_row['threshold']:<3} (차이: {best_row['diff_percent']:.4f}%)")
else:
    print("\n분석이 실행되었지만 비교할 결과가 없습니다.")

print("\n모든 분석이 완료되었습니다.")



In [ ]:
# ==============================================================================
# HYDRAFloods 민감도 분석 스크립트 (Jupyter Notebook용) - 최종 수정본
#
# 기능:
# 1. 초기 임계값을 0부터 -39까지 자동으로 변경하며 홍수 탐지 알고리즘 실행
# 2. 각 결과물을 정답(Ground Truth) 파일과 비교하여 정확한 차이 비율 계산
# 3. 각 알고리즘별 홍수 전/후 비교 히스토그램을 이미지 파일로 저장
# 4. 최종적으로 가장 성능이 좋았던 최적의 임계값을 찾아 요약 리포트 출력
# 5. 임계값 변화에 따른 차이 비율을 그래프로 시각화하여 출력
# ==============================================================================

import rasterio
from rasterio.mask import mask
from rasterio.warp import reproject, Resampling
import geopandas as gpd
import numpy as np
from skimage.filters import threshold_otsu
from skimage.feature import canny
from scipy.ndimage import binary_dilation
import os
import pandas as pd
from typing import Tuple, Optional, Dict, Any
import matplotlib.pyplot as plt

# --- 사용자 설정 변수 (이 부분만 수정하세요) ---

# 1. 파일 경로
PRE_FLOOD_IMAGE_PATH = r'<DATA_ROOT>\07_disaster\flood\Korea\s1_pipeline_test6\02_preprocess\S1A_IW_GRDH_1SDV_20250712T093124_20250712T093154_060049_0775E2_5D79_VV_processed.tif'
POST_FLOOD_IMAGE_PATH = r'<DATA_ROOT>\07_disaster\flood\Korea\s1_pipeline_test6\02_preprocess\S1A_IW_GRDH_1SDV_20250719T092320_20250719T092350_060151_07795E_0ABD_VV_processed.tif'
AOI_SHP_PATH = r"<DATA_ROOT>\07_disaster\flood\Korea\202507_Jinju\shp\AOI.shp"
SLOPE_TIF_PATH = r"<DATA_ROOT>\07_disaster\flood\Korea\202507_Jinju\dem\slope.tif"
GROUND_TRUTH_PATH = r"<DATA_ROOT>\07_disaster\flood\Korea\202507_Jinju\process_python\S1A_20250719_slopeFiltered_binaryMask.tif"
BASE_OUTPUT_DIR = r'<DATA_ROOT>\07_disaster\flood\Korea\s1_pipeline_test6\sensitivity_analysis_results39'

# 2. 테스트할 초기 임계값 범위
THRESHOLDS_TO_TEST = range(0, -40, -1)  # 0, -1, -2, ..., -39

# 3. 고정 알고리즘 매개변수
SLOPE_THRESHOLD = 15
EDGE_OTSU_BUFFER_ITERATIONS = 5
BMAX_OTSU_TILE_SIZE = 64
BMAX_OTSU_VARIANCE_PERCENTILE = 75
FUZZY_OTSU_SPREAD_FACTOR = 0.5
FUZZY_OTSU_CUTOFF = 0.9

# --- 유틸리티 및 알고리즘 함수 ---

def load_and_clip_raster(raster_path: str, aoi_geom: list) -> Tuple[Optional[np.ndarray], Optional[rasterio.profiles.Profile]]:
    with rasterio.open(raster_path) as src:
        try:
            nodata_value = src.nodata if src.nodata is not None else 0
            out_image, out_transform = mask(src, aoi_geom, crop=True, nodata=nodata_value, filled=True)
            out_meta = src.profile
            out_meta.update({"driver": "GTiff", "height": out_image.shape[1], "width": out_image.shape[2], "transform": out_transform, "nodata": nodata_value})
            squeezed_image = out_image.squeeze()
            float_image = squeezed_image.astype(float)
            float_image[squeezed_image == nodata_value] = np.nan
            if float_image.ndim == 0: return np.array([[float_image]]), out_meta
            return float_image, out_meta
        except (ValueError, rasterio.errors.RasterioIOError): return None, None

def save_raster(data: np.ndarray, meta: rasterio.profiles.Profile, output_path: str, nan_mask: np.ndarray):
    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    save_data = data.astype(np.uint8)
    save_data[nan_mask] = 255 # NoData 영역을 255로 설정
    meta.update({'dtype': 'uint8', 'nodata': 255, 'count': 1})
    with rasterio.open(output_path, "w", **meta) as dest:
        dest.write(save_data, 1)

def plot_and_save_histograms(algorithm_name: str, output_dir: str, post_hist: Dict[str, Any], pre_hist: Optional[Dict[str, Any]] = None):
    post_data = post_hist.get("data")
    if post_data is None or post_data.size == 0: return
    plt.figure(figsize=(12, 7))
    if pre_hist and pre_hist.get("data") is not None and pre_hist.get("data").size > 0:
        pre_data = pre_hist.get("data")
        min_val = min(np.min(pre_data) if pre_data.size > 0 else np.inf, np.min(post_data) if post_data.size > 0 else np.inf)
        max_val = max(np.max(pre_data) if pre_data.size > 0 else -np.inf, np.max(post_data) if post_data.size > 0 else -np.inf)
        bins = np.linspace(min_val, max_val, 100)
        plt.hist(pre_data, bins=bins, color='skyblue', alpha=0.7, label='Pre-flood')
        plt.hist(post_data, bins=bins, color='salmon', alpha=0.7, label='Post-flood')
        pre_thresh, post_thresh = pre_hist.get("threshold"), post_hist.get("threshold")
        pre_label, post_label = pre_hist.get("label", "Thresh"), post_hist.get("label", "Thresh")
        if pre_thresh is not None: plt.axvline(pre_thresh, color='blue', linestyle='--', linewidth=2, label=f'Pre-flood {pre_label}: {pre_thresh:.4f}')
        if post_thresh is not None: plt.axvline(post_thresh, color='red', linestyle='--', linewidth=2, label=f'Post-flood {post_label}: {post_thresh:.4f}')
        title, file_name = f'Pre vs. Post Flood Histogram for {algorithm_name}', f'histogram_combined_{algorithm_name}.png'
    else:
        plt.hist(post_data, bins=100, color='gray', alpha=0.7, label='Post-flood')
        post_thresh, post_label = post_hist.get("threshold"), post_hist.get("label", "Thresh")
        if post_thresh is not None: plt.axvline(post_thresh, color='red', linestyle='--', linewidth=2, label=f'Post-flood {post_label}: {post_thresh:.4f}')
        title, file_name = f'Histogram for {algorithm_name} (Post-flood)', f'histogram_post_flood_{algorithm_name}.png'
    plt.title(title)
    plt.xlabel('SAR Backscatter (dB)')
    plt.ylabel('Frequency')
    plt.legend()
    plt.grid(True, linestyle='--', alpha=0.6)
    plt.savefig(os.path.join(output_dir, file_name))
    plt.close()

def edge_otsu_thresholding(image: np.ndarray, iterations: int, initial_threshold: Optional[float]) -> Tuple[np.ndarray, Dict[str, Any]]:
    valid_pixels = image[~np.isnan(image)]
    if valid_pixels.size == 0: return np.zeros_like(image, dtype=bool), {}
    initial_thresh = initial_threshold if initial_threshold is not None else threshold_otsu(valid_pixels)
    binary_image = image < initial_thresh
    edges = canny(binary_image)
    buffered_edges = binary_dilation(edges, iterations=iterations)
    sample_pixels = image[buffered_edges & ~np.isnan(image)]
    final_thresh = threshold_otsu(sample_pixels) if sample_pixels.size > 0 else initial_thresh
    result = image < final_thresh
    return result, {"data": sample_pixels, "threshold": final_thresh, "label": "Final Threshold"}

def bmax_otsu_thresholding(image: np.ndarray, tile_size: int, variance_percentile: int, initial_threshold: Optional[float]) -> Tuple[np.ndarray, Dict[str, Any]]:
    rows, cols = image.shape
    candidate_tiles = [image[r:r+tile_size, c:c+tile_size] for r in range(0, rows, tile_size) for c in range(0, cols, tile_size)]
    candidate_tiles = [t for t in candidate_tiles if t[~np.isnan(t)].size > 0]
    if initial_threshold is not None:
        candidate_tiles = [t for t in candidate_tiles if np.any(t[~np.isnan(t)] < initial_threshold) and np.any(t[~np.isnan(t)] > initial_threshold)]
    if not candidate_tiles:
        valid_pixels = image[~np.isnan(image)]
        final_thresh = threshold_otsu(valid_pixels)
        hist_info = {"data": valid_pixels, "threshold": final_thresh, "label": "Otsu Threshold"}
    else:
        variances = [np.nanvar(t) for t in candidate_tiles]
        variance_threshold = np.percentile(variances, variance_percentile)
        selected_pixels = [p for i, t in enumerate(candidate_tiles) if variances[i] >= variance_threshold for p in t[~np.isnan(t)].flatten()]
        hist_data = np.array(selected_pixels)
        final_thresh = threshold_otsu(hist_data) if hist_data.size > 0 else threshold_otsu(image[~np.isnan(image)])
        hist_info = {"data": hist_data, "threshold": final_thresh, "label": "Final Threshold"}
    result = image < final_thresh
    return result, hist_info

def fuzzy_otsu_thresholding(image: np.ndarray, spread_factor: float, cutoff: float) -> Tuple[np.ndarray, Dict[str, Any]]:
    valid_pixels = image[~np.isnan(image)]
    if valid_pixels.size == 0: return np.zeros_like(image, dtype=bool), {}
    global_thresh = threshold_otsu(valid_pixels)
    water_pixels, land_pixels = valid_pixels[valid_pixels < global_thresh], valid_pixels[valid_pixels >= global_thresh]
    if water_pixels.size == 0 or land_pixels.size == 0:
        result, hist_info = image < global_thresh, {"data": valid_pixels, "threshold": global_thresh, "label": "Otsu Threshold"}
    else:
        min_center, max_center = np.mean(water_pixels), np.mean(land_pixels)
        midpoint = (min_center + max_center) / 2
        spread = (max_center - min_center) * spread_factor if (max_center - min_center) != 0 else 1
        fuzzy_values = 1 - np.exp(-((image - midpoint)**2) / (2 * spread**2))
        result, hist_info = fuzzy_values >= cutoff, {"data": valid_pixels, "threshold": midpoint, "label": "Fuzzy Midpoint"}
    return result, hist_info

def compare_rasters(result_file_path: str, truth_file_path: str) -> dict:
    if not os.path.exists(result_file_path) or not os.path.exists(truth_file_path): return {"error": "File not found"}
    with rasterio.open(result_file_path) as src_result, rasterio.open(truth_file_path) as src_truth:
        arr_result = src_result.read(1)
        arr_truth_reprojected = np.empty_like(arr_result, dtype=src_truth.dtypes[0])
        reproject(
            source=rasterio.band(src_truth, 1), destination=arr_truth_reprojected,
            src_transform=src_truth.transform, src_crs=src_truth.crs, src_nodata=src_truth.nodata,
            dst_transform=src_result.transform, dst_crs=src_result.crs, dst_nodata=255, # 결과 파일의 nodata와 동일하게 설정
            resampling=Resampling.nearest
        )
        arr_truth_final = np.where(arr_truth_reprojected == 1, 1, 0)
        valid_mask = arr_result != 255
        arr1_valid, arr2_valid = arr_result[valid_mask], arr_truth_final[valid_mask]
        total, diff = arr1_valid.size, np.sum(arr1_valid != arr2_valid)
        return {"total_pixels": total, "diff_pixels": diff, "diff_percent": (diff / total * 100) if total else 0.0}

def run_single_analysis(params: Dict[str, Any]):
    aoi_gdf = gpd.read_file(params["aoi_shp"])
    aoi_geometry = [geom for geom in aoi_gdf.geometry]
    post_flood_img, meta = load_and_clip_raster(params["post_flood_tif"], aoi_geometry)
    if post_flood_img is None: return
    
    nan_mask = np.isnan(post_flood_img)
    
    pre_flood_img_raw, pre_meta_raw = load_and_clip_raster(params["pre_flood_tif"], aoi_geometry)
    slope_img_raw, slope_meta_raw = load_and_clip_raster(params["slope_tif"], aoi_geometry)
    resampled_pre_flood, resampled_slope = None, None
    if pre_flood_img_raw is not None and pre_meta_raw:
        dest = np.empty_like(post_flood_img, dtype=float)
        reproject(source=pre_flood_img_raw, destination=dest, src_transform=pre_meta_raw['transform'], src_crs=pre_meta_raw['crs'], dst_transform=meta['transform'], dst_crs=meta['crs'], resampling=Resampling.bilinear)
        resampled_pre_flood = dest
    if slope_img_raw is not None and slope_meta_raw:
        dest = np.empty_like(post_flood_img, dtype=float)
        reproject(source=slope_img_raw, destination=dest, src_transform=slope_meta_raw['transform'], src_crs=slope_meta_raw['crs'], dst_transform=meta['transform'], dst_crs=meta['crs'], resampling=Resampling.bilinear)
        resampled_slope = dest
    
    algorithms = {
        "edge_otsu": (edge_otsu_thresholding, {"iterations": params["edge_buffer"], "initial_threshold": params["edge_thresh"]}),
        "bmax_otsu": (bmax_otsu_thresholding, {"tile_size": params["bmax_tile_size"], "variance_percentile": params["bmax_percentile"], "initial_threshold": params["bmax_thresh"]}),
        "fuzzy_otsu": (fuzzy_otsu_thresholding, {"spread_factor": params["fuzzy_spread"], "cutoff": params["fuzzy_cutoff"]})
    }
    
    for name, (func, func_params) in algorithms.items():
        water_map, hist_info = func(post_flood_img, **func_params)
        water_map[nan_mask] = False
        
        slope_mask = None
        if resampled_slope is not None:
            slope_mask = resampled_slope >= params["slope_thresh"]
            water_map[slope_mask] = False
        save_raster(water_map, meta, os.path.join(params["output_dir"], f'post_flood_water_{name}.tif'), nan_mask)
        
        pre_hist_info = None
        if resampled_pre_flood is not None:
            pre_water_map, pre_hist_info = func(resampled_pre_flood, **func_params)
            pre_water_map[nan_mask] = False
            if slope_mask is not None: pre_water_map[slope_mask] = False
            flood_extent = water_map & ~pre_water_map
            save_raster(flood_extent, meta, os.path.join(params["output_dir"], f'flood_change_{name}.tif'), nan_mask)
        plot_and_save_histograms(name, params["output_dir"], hist_info, pre_hist_info)

# ==============================================================================
# --- 자동화된 민감도 분석 실행 ---
# ==============================================================================
results_log = []
for threshold in THRESHOLDS_TO_TEST:
    print(f"\n{'='*20} 테스트 시작: 초기 임계값 = {threshold} {'='*20}")
    current_output_dir = os.path.join(BASE_OUTPUT_DIR, f"thresh_{threshold}")
    
    params = {
        "pre_flood_tif": PRE_FLOOD_IMAGE_PATH, "post_flood_tif": POST_FLOOD_IMAGE_PATH,
        "aoi_shp": AOI_SHP_PATH, "slope_tif": SLOPE_TIF_PATH, "output_dir": current_output_dir,
        "slope_thresh": SLOPE_THRESHOLD, "edge_buffer": EDGE_OTSU_BUFFER_ITERATIONS,
        "bmax_tile_size": BMAX_OTSU_TILE_SIZE, "bmax_percentile": BMAX_OTSU_VARIANCE_PERCENTILE,
        "fuzzy_spread": FUZZY_OTSU_SPREAD_FACTOR, "fuzzy_cutoff": FUZZY_OTSU_CUTOFF,
        "edge_thresh": threshold, "bmax_thresh": threshold
    }
    
    run_single_analysis(params)
    
    print("\n--- 결과 비교 ---")
    for algorithm in ["edge_otsu", "bmax_otsu", "fuzzy_otsu"]:
        result_file = os.path.join(current_output_dir, f"post_flood_water_{algorithm}.tif")
        comparison = compare_rasters(result_file, GROUND_TRUTH_PATH)
        if "error" in comparison: continue
        print(f"[{algorithm.upper():<10}] 차이 비율: {comparison['diff_percent']:.4f}%")
        results_log.append({"threshold": threshold, "algorithm": algorithm, "diff_percent": comparison['diff_percent']})

# --- 최종 결과 요약 ---
if results_log:
    print(f"\n{'='*25} 최종 분석 결과 요약 {'='*25}")
    df = pd.DataFrame(results_log)
    if not df.empty:
        print("\n--- 전체 결과 ---")
        print(df.to_string())

        print("\n--- 최적 임계값 ---")
        for algo_name in ["edge_otsu", "bmax_otsu", "fuzzy_otsu"]:
            algo_df = df[df['algorithm'] == algo_name]
            if not algo_df.empty:
                best_row = algo_df.loc[algo_df['diff_percent'].idxmin()]
                print(f"[{algo_name.upper():<10}] 최적 임계값: {best_row['threshold']:<3} (차이: {best_row['diff_percent']:.4f}%)")
        
        # --- 결과 그래프 시각화 ---
        print("\n--- 민감도 분석 그래프 ---")
        plt.figure(figsize=(15, 8))
        for algo_name in df['algorithm'].unique():
            algo_df = df[df['algorithm'] == algo_name].sort_values(by='threshold')
            plt.plot(algo_df['threshold'], algo_df['diff_percent'], marker='o', linestyle='-', label=algo_name)
        
        plt.title('Initial Threshold vs. Difference Percentage', fontsize=16)
        plt.xlabel('Initial Threshold (dB)', fontsize=12)
        plt.ylabel('Difference Percentage (%)', fontsize=12)
        plt.gca().invert_xaxis() # X축을 거꾸로 (0, -1, -2...)
        plt.legend(fontsize=10)
        plt.grid(True, which='both', linestyle='--', linewidth=0.5)
        plt.show()

else:
    print("\n분석이 실행되었지만 비교할 결과가 없습니다.")

print("\n모든 분석이 완료되었습니다.")



In [ ]:
# ==============================================================================
# HYDRAFloods 민감도 분석 스크립트 (Jupyter Notebook용) - 최종 최적화 버전
#
# 기능:
# 1. 데이터 준비(재투영 등)를 분석 시작 전 1회만 수행하여 속도 향상
# 2. 초기 임계값을 0부터 -39까지 자동으로 변경하며 홍수 탐지 알고리즘 실행
# 3. 각 결과물을 정답(Ground Truth) 파일과 비교하여 정확한 차이 비율 계산
# 4. 각 알고리즘별 홍수 전/후 비교 히스토그램을 이미지 파일로 저장
# 5. 최종적으로 가장 성능이 좋았던 최적의 임계값을 찾아 요약 리포트 출력
# 6. 임계값 변화에 따른 차이 비율을 그래프로 시각화하여 출력
# ==============================================================================

import rasterio
from rasterio.mask import mask
from rasterio.warp import reproject, Resampling, calculate_default_transform
import geopandas as gpd
import numpy as np
from skimage.filters import threshold_otsu
from skimage.feature import canny
from scipy.ndimage import binary_dilation
import os
import pandas as pd
from typing import Tuple, Optional, Dict, Any
import matplotlib.pyplot as plt

# --- 사용자 설정 변수 (이 부분만 수정하세요) ---

# 1. 파일 경로
# --- [EPSG:4326] 좌표계여야 하는 파일들 ---
PRE_FLOOD_IMAGE_PATH = r'<DATA_ROOT>\07_disaster\flood\Korea\202507_Yesan2\02_preprocess\S1A_IW_GRDH_1SDV_20250630T093154_20250630T093219_059874_076FDD_4EE3_VV_processed.tif'
POST_FLOOD_IMAGE_PATH = r'<DATA_ROOT>\07_disaster\flood\Korea\202507_Yesan2\02_preprocess\S1A_IW_GRDH_1SDV_20250724T093153_20250724T093218_060224_077BED_A1F0_VV_processed.tif'
AOI_SHP_PATH = r"<DATA_ROOT>\07_disaster\flood\Korea\202507_Yesan\shp\AOI.shp"
GROUND_TRUTH_PATH = r"<DATA_ROOT>\07_disaster\flood\Korea\202507_Yesan2\03_analysis\water_mask\S1A_IW_GRDH_1SDV_20250724T093153_20250724T093218_060224_077BED_A1F0_VV_processed_watermask.tif"

# --- [EPSG:32652] 좌표계여야 하는 파일 ---
SLOPE_TIF_PATH_32652 = r"<DATA_ROOT>\07_disaster\flood\Korea\202507_Yesan2\00_dem_slope\AOI_SLOPE_GLO30_EPSG32652.tif"

# --- 결과 저장 경로 ---
BASE_OUTPUT_DIR = r'<DATA_ROOT>\07_disaster\flood\Korea\202507_Yesan2\sensitivity_analysis_results'

# 2. 테스트할 초기 임계값 범위
THRESHOLDS_TO_TEST = range(0, -40, -1)  # 0, -1, -2, ..., -39

# 3. 고정 알고리즘 매개변수
SLOPE_THRESHOLD = 15
EDGE_OTSU_BUFFER_ITERATIONS = 5
BMAX_OTSU_TILE_SIZE = 64
BMAX_OTSU_VARIANCE_PERCENTILE = 75
FUZZY_OTSU_SPREAD_FACTOR = 0.5
FUZZY_OTSU_CUTOFF = 0.9

# --- 유틸리티 및 알고리즘 함수 (수정 없음) ---
def load_and_clip_raster(raster_path: str, aoi_geom: list) -> Tuple[Optional[np.ndarray], Optional[rasterio.profiles.Profile]]:
    with rasterio.open(raster_path) as src:
        try:
            nodata_value = src.nodata if src.nodata is not None else 0
            out_image, out_transform = mask(src, aoi_geom, crop=True, nodata=nodata_value, filled=True)
            out_meta = src.profile
            out_meta.update({"driver": "GTiff", "height": out_image.shape[1], "width": out_image.shape[2], "transform": out_transform, "nodata": nodata_value})
            squeezed_image = out_image.squeeze()
            float_image = squeezed_image.astype(float)
            float_image[squeezed_image == nodata_value] = np.nan
            if float_image.ndim == 0: return np.array([[float_image]]), out_meta
            return float_image, out_meta
        except (ValueError, rasterio.errors.RasterioIOError): return None, None

def save_raster(data: np.ndarray, meta: rasterio.profiles.Profile, output_path: str, nan_mask: np.ndarray):
    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    save_data = data.astype(np.uint8)
    save_data[nan_mask] = 255
    meta.update({'dtype': 'uint8', 'nodata': 255, 'count': 1})
    with rasterio.open(output_path, "w", **meta) as dest:
        dest.write(save_data, 1)

def plot_and_save_histograms(algorithm_name: str, output_dir: str, post_hist: Dict[str, Any], pre_hist: Optional[Dict[str, Any]] = None):
    post_data = post_hist.get("data")
    if post_data is None or post_data.size == 0: return
    plt.figure(figsize=(12, 7))
    if pre_hist and pre_hist.get("data") is not None and pre_hist.get("data").size > 0:
        pre_data = pre_hist.get("data")
        min_val = min(np.min(pre_data) if pre_data.size > 0 else np.inf, np.min(post_data) if post_data.size > 0 else np.inf)
        max_val = max(np.max(pre_data) if pre_data.size > 0 else -np.inf, np.max(post_data) if post_data.size > 0 else -np.inf)
        bins = np.linspace(min_val, max_val, 100)
        plt.hist(pre_data, bins=bins, color='skyblue', alpha=0.7, label='Pre-flood')
        plt.hist(post_data, bins=bins, color='salmon', alpha=0.7, label='Post-flood')
        pre_thresh, post_thresh = pre_hist.get("threshold"), post_hist.get("threshold")
        pre_label, post_label = pre_hist.get("label", "Thresh"), post_hist.get("label", "Thresh")
        if pre_thresh is not None: plt.axvline(pre_thresh, color='blue', linestyle='--', linewidth=2, label=f'Pre-flood {pre_label}: {pre_thresh:.4f}')
        if post_thresh is not None: plt.axvline(post_thresh, color='red', linestyle='--', linewidth=2, label=f'Post-flood {post_label}: {post_thresh:.4f}')
        title, file_name = f'Pre vs. Post Flood Histogram for {algorithm_name}', f'histogram_combined_{algorithm_name}.png'
    else:
        plt.hist(post_data, bins=100, color='gray', alpha=0.7, label='Post-flood')
        post_thresh, post_label = post_hist.get("threshold"), post_hist.get("label", "Thresh")
        if post_thresh is not None: plt.axvline(post_thresh, color='red', linestyle='--', linewidth=2, label=f'Post-flood {post_label}: {post_thresh:.4f}')
        title, file_name = f'Histogram for {algorithm_name} (Post-flood)', f'histogram_post_flood_{algorithm_name}.png'
    plt.title(title)
    plt.xlabel('SAR Backscatter (dB)')
    plt.ylabel('Frequency')
    plt.legend()
    plt.grid(True, linestyle='--', alpha=0.6)
    plt.savefig(os.path.join(output_dir, file_name))
    plt.close()

def edge_otsu_thresholding(image: np.ndarray, iterations: int, initial_threshold: Optional[float]) -> Tuple[np.ndarray, Dict[str, Any]]:
    valid_pixels = image[~np.isnan(image)]
    if valid_pixels.size == 0: return np.zeros_like(image, dtype=bool), {}
    initial_thresh = initial_threshold if initial_threshold is not None else threshold_otsu(valid_pixels)
    binary_image = image < initial_thresh
    edges = canny(binary_image)
    buffered_edges = binary_dilation(edges, iterations=iterations)
    sample_pixels = image[buffered_edges & ~np.isnan(image)]
    final_thresh = threshold_otsu(sample_pixels) if sample_pixels.size > 0 else initial_thresh
    result = image < final_thresh
    return result, {"data": sample_pixels, "threshold": final_thresh, "label": "Final Threshold"}

def bmax_otsu_thresholding(image: np.ndarray, tile_size: int, variance_percentile: int, initial_threshold: Optional[float]) -> Tuple[np.ndarray, Dict[str, Any]]:
    rows, cols = image.shape
    candidate_tiles = [image[r:r+tile_size, c:c+tile_size] for r in range(0, rows, tile_size) for c in range(0, cols, tile_size)]
    candidate_tiles = [t for t in candidate_tiles if t[~np.isnan(t)].size > 0]
    if initial_threshold is not None:
        candidate_tiles = [t for t in candidate_tiles if np.any(t[~np.isnan(t)] < initial_threshold) and np.any(t[~np.isnan(t)] > initial_threshold)]
    if not candidate_tiles:
        valid_pixels = image[~np.isnan(image)]
        final_thresh = threshold_otsu(valid_pixels)
        hist_info = {"data": valid_pixels, "threshold": final_thresh, "label": "Otsu Threshold"}
    else:
        variances = [np.nanvar(t) for t in candidate_tiles]
        variance_threshold = np.percentile(variances, variance_percentile)
        selected_pixels = [p for i, t in enumerate(candidate_tiles) if variances[i] >= variance_threshold for p in t[~np.isnan(t)].flatten()]
        hist_data = np.array(selected_pixels)
        final_thresh = threshold_otsu(hist_data) if hist_data.size > 0 else threshold_otsu(image[~np.isnan(image)])
        hist_info = {"data": hist_data, "threshold": final_thresh, "label": "Final Threshold"}
    result = image < final_thresh
    return result, hist_info

def fuzzy_otsu_thresholding(image: np.ndarray, spread_factor: float, cutoff: float) -> Tuple[np.ndarray, Dict[str, Any]]:
    valid_pixels = image[~np.isnan(image)]
    if valid_pixels.size == 0: return np.zeros_like(image, dtype=bool), {}
    global_thresh = threshold_otsu(valid_pixels)
    water_pixels, land_pixels = valid_pixels[valid_pixels < global_thresh], valid_pixels[valid_pixels >= global_thresh]
    if water_pixels.size == 0 or land_pixels.size == 0:
        result, hist_info = image < global_thresh, {"data": valid_pixels, "threshold": global_thresh, "label": "Otsu Threshold"}
    else:
        min_center, max_center = np.mean(water_pixels), np.mean(land_pixels)
        midpoint = (min_center + max_center) / 2
        spread = (max_center - min_center) * spread_factor if (max_center - min_center) != 0 else 1
        fuzzy_values = 1 - np.exp(-((image - midpoint)**2) / (2 * spread**2))
        result, hist_info = fuzzy_values >= cutoff, {"data": valid_pixels, "threshold": midpoint, "label": "Fuzzy Midpoint"}
    return result, hist_info

def compare_rasters(result_file_path: str, truth_file_path: str) -> dict:
    if not os.path.exists(result_file_path) or not os.path.exists(truth_file_path): return {"error": "File not found"}
    with rasterio.open(result_file_path) as src_result, rasterio.open(truth_file_path) as src_truth:
        arr_result = src_result.read(1)
        arr_truth_reprojected = np.empty_like(arr_result, dtype=src_truth.dtypes[0])
        reproject(
            source=rasterio.band(src_truth, 1), destination=arr_truth_reprojected,
            src_transform=src_truth.transform, src_crs=src_truth.crs, src_nodata=src_truth.nodata,
            dst_transform=src_result.transform, dst_crs=src_result.crs, dst_nodata=255, # 결과 파일의 nodata와 동일하게 설정
            resampling=Resampling.nearest
        )
        arr_truth_final = np.where(arr_truth_reprojected == 1, 1, 0)
        valid_mask = arr_result != 255
        arr1_valid, arr2_valid = arr_result[valid_mask], arr_truth_final[valid_mask]
        total, diff = arr1_valid.size, np.sum(arr1_valid != arr2_valid)
        return {"total_pixels": total, "diff_pixels": diff, "diff_percent": (diff / total * 100) if total else 0.0}


# --- [수정] 분석 함수: 이제 데이터 준비가 아닌 실제 분석만 수행 ---
def run_single_threshold_analysis(
    params: Dict[str, Any], 
    post_flood_img: np.ndarray, 
    pre_flood_img: Optional[np.ndarray], 
    slope_mask: Optional[np.ndarray],
    nan_mask: np.ndarray,
    meta: rasterio.profiles.Profile
):
    """사전에 준비된 데이터 배열을 받아 특정 임계값에 대한 분석만 수행"""
    algorithms = {
        "edge_otsu": (edge_otsu_thresholding, {"iterations": params["edge_buffer"], "initial_threshold": params["edge_thresh"]}),
        "bmax_otsu": (bmax_otsu_thresholding, {"tile_size": params["bmax_tile_size"], "variance_percentile": params["bmax_percentile"], "initial_threshold": params["bmax_thresh"]}),
        "fuzzy_otsu": (fuzzy_otsu_thresholding, {"spread_factor": params["fuzzy_spread"], "cutoff": params["fuzzy_cutoff"]})
    }

    for name, (func, func_params) in algorithms.items():
        water_map, hist_info = func(post_flood_img, **func_params)
        water_map[nan_mask] = False
        
        if slope_mask is not None:
            water_map[slope_mask] = False
            
        save_raster(water_map, meta, os.path.join(params["output_dir"], f'post_flood_water_{name}.tif'), nan_mask)

        pre_hist_info = None
        if pre_flood_img is not None:
            pre_water_map, pre_hist_info = func(pre_flood_img, **func_params)
            pre_water_map[nan_mask] = False
            if slope_mask is not None:
                pre_water_map[slope_mask] = False
            flood_extent = water_map & ~pre_water_map
            save_raster(flood_extent, meta, os.path.join(params["output_dir"], f'flood_change_{name}.tif'), nan_mask)
        
        plot_and_save_histograms(name, params["output_dir"], hist_info, pre_hist_info)

# ==============================================================================
# --- [수정] 메인 실행 로직 ---
# ==============================================================================

# 1. 데이터 준비 (분석 시작 전 1회만 실행)
print("="*50)
print("데이터 준비를 시작합니다... (이 과정은 한 번만 실행됩니다)")

aoi_gdf = gpd.read_file(AOI_SHP_PATH)
aoi_geometry = [geom for geom in aoi_gdf.geometry]

# 1a. 기준 데이터 로드 (Post-flood)
post_flood_img, meta = load_and_clip_raster(POST_FLOOD_IMAGE_PATH, aoi_geometry)
if post_flood_img is None:
    raise FileNotFoundError("홍수 후 이미지를 불러오는 데 실패했습니다. 분석을 중단합니다.")
nan_mask = np.isnan(post_flood_img)

# 1b. 홍수 전 이미지 로드 및 정렬
pre_flood_img_raw, pre_meta_raw = load_and_clip_raster(PRE_FLOOD_IMAGE_PATH, aoi_geometry)
resampled_pre_flood = None
if pre_flood_img_raw is not None:
    print("홍수 전 이미지를 홍수 후 이미지 격자에 맞게 정렬(재투영)합니다...")
    resampled_pre_flood = np.empty_like(post_flood_img)
    reproject(
        source=pre_flood_img_raw, destination=resampled_pre_flood,
        src_transform=pre_meta_raw['transform'], src_crs=pre_meta_raw['crs'],
        dst_transform=meta['transform'], dst_crs=meta['crs'],
        resampling=Resampling.bilinear
    )
    print("-> 정렬 완료.")

# 1c. 경사도 파일 변환 및 마스크 생성
slope_mask = None
try:
    with rasterio.open(SLOPE_TIF_PATH_32652) as slope_src:
        print(f"경사도 파일({slope_src.crs})을 기준 좌표계({meta['crs']})로 변환 중...")
        reprojected_slope_full = np.empty(post_flood_img.shape, dtype=slope_src.dtypes[0])
        reproject(
            source=rasterio.band(slope_src, 1), destination=reprojected_slope_full,
            src_transform=slope_src.transform, src_crs=slope_src.crs,
            dst_transform=meta['transform'], dst_crs=meta['crs'],
            resampling=Resampling.bilinear
        )
        
        with rasterio.io.MemoryFile() as memfile:
            with memfile.open(**meta) as dataset:
                dataset.write(reprojected_slope_full, 1)
            with memfile.open() as dataset:
                clipped_data, _ = mask(dataset, aoi_geometry, crop=True, nodata=np.nan)
                clipped_slope_4326 = clipped_data.squeeze()
        print("-> 경사도 변환 및 클리핑 완료.")
        
        h, w = post_flood_img.shape
        clipped_slope_4326 = clipped_slope_4326[:h, :w]
        slope_mask = (~np.isnan(clipped_slope_4326)) & (clipped_slope_4326 >= SLOPE_THRESHOLD)
except Exception as e:
    print(f"!!! 경고: 경사도 파일 처리 실패: {e}. 경사도 필터링 없이 분석을 진행합니다.")

print("데이터 준비 완료.")
print("="*50)


# 2. 민감도 분석 실행 (임계값 변경하며 반복)
results_log = []
for threshold in THRESHOLDS_TO_TEST:
    print(f"\n{'='*20} 테스트 시작: 초기 임계값 = {threshold} {'='*20}")
    current_output_dir = os.path.join(BASE_OUTPUT_DIR, f"thresh_{threshold}")
    os.makedirs(current_output_dir, exist_ok=True)
    
    params = {
        "output_dir": current_output_dir,
        "slope_thresh": SLOPE_THRESHOLD,
        "edge_buffer": EDGE_OTSU_BUFFER_ITERATIONS,
        "bmax_tile_size": BMAX_OTSU_TILE_SIZE,
        "bmax_percentile": BMAX_OTSU_VARIANCE_PERCENTILE,
        "fuzzy_spread": FUZZY_OTSU_SPREAD_FACTOR,
        "fuzzy_cutoff": FUZZY_OTSU_CUTOFF,
        "edge_thresh": threshold,
        "bmax_thresh": threshold
    }
    
    # 미리 준비된 데이터를 인자로 전달
    run_single_threshold_analysis(
        params,
        post_flood_img=post_flood_img,
        pre_flood_img=resampled_pre_flood,
        slope_mask=slope_mask,
        nan_mask=nan_mask,
        meta=meta
    )
    
    print("\n--- 결과 비교 ---")
    for algorithm in ["edge_otsu", "bmax_otsu", "fuzzy_otsu"]:
        result_file = os.path.join(current_output_dir, f"post_flood_water_{algorithm}.tif")
        comparison = compare_rasters(result_file, GROUND_TRUTH_PATH)
        if "error" in comparison: continue
        print(f"[{algorithm.upper():<10}] 차이 비율: {comparison['diff_percent']:.4f}%")
        results_log.append({"threshold": threshold, "algorithm": algorithm, "diff_percent": comparison['diff_percent']})

# 3. 최종 결과 요약
if results_log:
    print(f"\n{'='*25} 최종 분석 결과 요약 {'='*25}")
    df = pd.DataFrame(results_log)
    if not df.empty:
        print("\n--- 전체 결과 ---")
        print(df.to_string())

        print("\n--- 최적 임계값 ---")
        for algo_name in ["edge_otsu", "bmax_otsu", "fuzzy_otsu"]:
            algo_df = df[df['algorithm'] == algo_name]
            if not algo_df.empty:
                best_row = algo_df.loc[algo_df['diff_percent'].idxmin()]
                print(f"[{algo_name.upper():<10}] 최적 임계값: {best_row['threshold']:<3} (차이: {best_row['diff_percent']:.4f}%)")
        
        # --- 결과 그래프 시각화 ---
        print("\n--- 민감도 분석 그래프 ---")
        plt.figure(figsize=(15, 8))
        for algo_name in df['algorithm'].unique():
            algo_df = df[df['algorithm'] == algo_name].sort_values(by='threshold')
            plt.plot(algo_df['threshold'], algo_df['diff_percent'], marker='o', linestyle='-', label=algo_name)
        
        plt.title('Initial Threshold vs. Difference Percentage', fontsize=16)
        plt.xlabel('Initial Threshold (dB)', fontsize=12)
        plt.ylabel('Difference Percentage (%)', fontsize=12)
        plt.gca().invert_xaxis() # X축을 거꾸로 (0, -1, -2...)
        plt.legend(fontsize=10)
        plt.grid(True, which='both', linestyle='--', linewidth=0.5)
        plt.show()

else:
    print("\n분석이 실행되었지만 비교할 결과가 없습니다.")

print("\n모든 분석이 완료되었습니다.")

### 최종

In [ ]:

# ==============================================================================
# HYDRAFloods 민감도 분석 스크립트 (Jupyter Notebook용) - 최종 버전
#
# 기능:
# 1. 모든 입력 파일과 파라미터를 txt 파일로 저장하여 분석 이력 관리
# 2. 사전에 준비된 모든 EPSG:4326 데이터를 입력받아 처리
# 3. 초기 임계값을 0부터 -39까지 자동으로 변경하며 홍수 탐지 알고리즘 실행
# 4. 각 결과물을 정답(Ground Truth) 파일과 비교하여 정확한 차이 비율 계산
# 5. 각 알고리즘별 홍수 전/후 비교 히스토그램을 이미지 파일로 저장
# 6. 최종적으로 가장 성능이 좋았던 최적의 임계값을 찾아 요약 리포트 출력
# 7. 임계값 변화에 따른 차이 비율을 그래프로 시각화하여 출력
# ==============================================================================

import rasterio
from rasterio.mask import mask
from rasterio.warp import reproject, Resampling
import geopandas as gpd
import numpy as np
from skimage.filters import threshold_otsu
from skimage.feature import canny
from scipy.ndimage import binary_dilation
import os
import pandas as pd
from typing import Tuple, Optional, Dict, Any
import matplotlib.pyplot as plt
from datetime import datetime # 시간 기록을 위해 임포트

# --- 사용자 설정 변수 (이 부분만 수정하세요) ---

# 1. 파일 경로 (모든 입력 파일은 EPSG:4326 좌표계여야 합니다)
PRE_FLOOD_IMAGE_PATH = r'<DATA_ROOT>\07_disaster\flood\Philippines\Luzon\S1_flood_detection\02_preprocess\S1C_IW_GRDH_1SDV_20250905T100631_20250905T100656_003988_007EEE_E31E_VV_processed.tif'
POST_FLOOD_IMAGE_PATH = r'<DATA_ROOT>\07_disaster\flood\Philippines\Luzon\S1_flood_detection\02_preprocess\S1C_IW_GRDH_1SDV_20250917T100623_20250917T100648_004163_008444_C4AA_VV_processed.tif'
AOI_SHP_PATH = r"<DATA_ROOT>\07_disaster\flood\Philippines\Luzon\shp\AOI.shp"
GROUND_TRUTH_PATH = r"<DATA_ROOT>\07_disaster\flood\Philippines\Luzon\S1_flood_detection\03_analysis\water_mask\S1C_IW_GRDH_1SDV_20250917T100623_20250917T100648_004163_008444_C4AA_VV_processed_watermask.tif"
SLOPE_TIF_PATH = r"<DATA_ROOT>\07_disaster\flood\Philippines\Luzon\S1_flood_detection\00_dem_slope\Full_Scene_SLOPE_GLO30_WGS84.tif"

# 결과 저장 경로
BASE_OUTPUT_DIR = r'<DATA_ROOT>\07_disaster\flood\Philippines\Luzon\S1_flood_detection\sensitivity_analysis_results'

# 2. 테스트할 초기 임계값 범위
THRESHOLDS_TO_TEST = range(0, -40, -1)  # 0, -1, -2, ..., -39

# 3. 고정 알고리즘 매개변수
SLOPE_THRESHOLD = 15
EDGE_OTSU_BUFFER_ITERATIONS = 5
BMAX_OTSU_TILE_SIZE = 64
BMAX_OTSU_VARIANCE_PERCENTILE = 75
FUZZY_OTSU_SPREAD_FACTOR = 0.5
FUZZY_OTSU_CUTOFF = 0.9

# --- [추가된 기능] 설정 파일 저장 함수 ---
def save_parameters_to_txt(output_dir, file_paths, params):
    """실행 시점의 모든 설정값을 txt 파일로 저장합니다."""
    os.makedirs(output_dir, exist_ok=True)
    filepath = os.path.join(output_dir, 'analysis_parameters.txt')
    
    with open(filepath, 'w', encoding='utf-8') as f:
        f.write("="*60 + "\n")
        f.write("HYDRAFloods Sensitivity Analysis - Parameters\n")
        f.write("="*60 + "\n")
        f.write(f"Analysis Time: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n\n")

        f.write("--- Input Files ---\n")
        for name, path in file_paths.items():
            f.write(f"{name:<25}: {path}\n")
        f.write("\n")

        f.write("--- Analysis Parameters ---\n")
        for name, value in params.items():
            f.write(f"{name:<25}: {value}\n")
        f.write("\n")
        
        f.write("--- Output Location ---\n")
        f.write(f"{'BASE_OUTPUT_DIR':<25}: {output_dir}\n")
    
    print(f"✅ 분석 설정 정보가 '{filepath}'에 저장되었습니다.")

# --- 유틸리티 및 알고리즘 함수 (수정 없음) ---
def load_and_clip_raster(raster_path: str, aoi_geom: list) -> Tuple[Optional[np.ndarray], Optional[rasterio.profiles.Profile]]:
    with rasterio.open(raster_path) as src:
        try:
            nodata_value = src.nodata if src.nodata is not None else 0
            out_image, out_transform = mask(src, aoi_geom, crop=True, nodata=nodata_value, filled=True)
            out_meta = src.profile
            out_meta.update({"driver": "GTiff", "height": out_image.shape[1], "width": out_image.shape[2], "transform": out_transform, "nodata": nodata_value})
            squeezed_image = out_image.squeeze()
            float_image = squeezed_image.astype(float)
            float_image[squeezed_image == nodata_value] = np.nan
            if float_image.ndim == 0: return np.array([[float_image]]), out_meta
            return float_image, out_meta
        except (ValueError, rasterio.errors.RasterioIOError): return None, None

def save_raster(data: np.ndarray, meta: rasterio.profiles.Profile, output_path: str, nan_mask: np.ndarray):
    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    save_data = data.astype(np.uint8)
    save_data[nan_mask] = 255
    meta.update({'dtype': 'uint8', 'nodata': 255, 'count': 1})
    with rasterio.open(output_path, "w", **meta) as dest:
        dest.write(save_data, 1)

def plot_and_save_histograms(algorithm_name: str, output_dir: str, post_hist: Dict[str, Any], pre_hist: Optional[Dict[str, Any]] = None):
    post_data = post_hist.get("data")
    if post_data is None or post_data.size == 0: return
    plt.figure(figsize=(12, 7))
    if pre_hist and pre_hist.get("data") is not None and pre_hist.get("data").size > 0:
        pre_data = pre_hist.get("data")
        min_val = min(np.min(pre_data) if pre_data.size > 0 else np.inf, np.min(post_data) if post_data.size > 0 else np.inf)
        max_val = max(np.max(pre_data) if pre_data.size > 0 else -np.inf, np.max(post_data) if post_data.size > 0 else -np.inf)
        bins = np.linspace(min_val, max_val, 100)
        plt.hist(pre_data, bins=bins, color='skyblue', alpha=0.7, label='Pre-flood')
        plt.hist(post_data, bins=bins, color='salmon', alpha=0.7, label='Post-flood')
        pre_thresh, post_thresh = pre_hist.get("threshold"), post_hist.get("threshold")
        pre_label, post_label = pre_hist.get("label", "Thresh"), post_hist.get("label", "Thresh")
        if pre_thresh is not None: plt.axvline(pre_thresh, color='blue', linestyle='--', linewidth=2, label=f'Pre-flood {pre_label}: {pre_thresh:.4f}')
        if post_thresh is not None: plt.axvline(post_thresh, color='red', linestyle='--', linewidth=2, label=f'Post-flood {post_label}: {post_thresh:.4f}')
        title, file_name = f'Pre vs. Post Flood Histogram for {algorithm_name}', f'histogram_combined_{algorithm_name}.png'
    else:
        plt.hist(post_data, bins=100, color='gray', alpha=0.7, label='Post-flood')
        post_thresh, post_label = post_hist.get("threshold"), post_hist.get("label", "Thresh")
        if post_thresh is not None: plt.axvline(post_thresh, color='red', linestyle='--', linewidth=2, label=f'Post-flood {post_label}: {post_thresh:.4f}')
        title, file_name = f'Histogram for {algorithm_name} (Post-flood)', f'histogram_post_flood_{algorithm_name}.png'
    plt.title(title)
    plt.xlabel('SAR Backscatter (dB)')
    plt.ylabel('Frequency')
    plt.legend()
    plt.grid(True, linestyle='--', alpha=0.6)
    plt.savefig(os.path.join(output_dir, file_name))
    plt.close()

def edge_otsu_thresholding(image: np.ndarray, iterations: int, initial_threshold: Optional[float]) -> Tuple[np.ndarray, Dict[str, Any]]:
    valid_pixels = image[~np.isnan(image)]
    if valid_pixels.size == 0: return np.zeros_like(image, dtype=bool), {}
    initial_thresh = initial_threshold if initial_threshold is not None else threshold_otsu(valid_pixels)
    binary_image = image < initial_thresh
    edges = canny(binary_image)
    buffered_edges = binary_dilation(edges, iterations=iterations)
    sample_pixels = image[buffered_edges & ~np.isnan(image)]
    final_thresh = threshold_otsu(sample_pixels) if sample_pixels.size > 0 else initial_thresh
    result = image < final_thresh
    return result, {"data": sample_pixels, "threshold": final_thresh, "label": "Final Threshold"}

def bmax_otsu_thresholding(image: np.ndarray, tile_size: int, variance_percentile: int, initial_threshold: Optional[float]) -> Tuple[np.ndarray, Dict[str, Any]]:
    rows, cols = image.shape
    candidate_tiles = [image[r:r+tile_size, c:c+tile_size] for r in range(0, rows, tile_size) for c in range(0, cols, tile_size)]
    candidate_tiles = [t for t in candidate_tiles if t[~np.isnan(t)].size > 0]
    if initial_threshold is not None:
        candidate_tiles = [t for t in candidate_tiles if np.any(t[~np.isnan(t)] < initial_threshold) and np.any(t[~np.isnan(t)] > initial_threshold)]
    if not candidate_tiles:
        valid_pixels = image[~np.isnan(image)]
        final_thresh = threshold_otsu(valid_pixels)
        hist_info = {"data": valid_pixels, "threshold": final_thresh, "label": "Otsu Threshold"}
    else:
        variances = [np.nanvar(t) for t in candidate_tiles]
        variance_threshold = np.percentile(variances, variance_percentile)
        selected_pixels = [p for i, t in enumerate(candidate_tiles) if variances[i] >= variance_threshold for p in t[~np.isnan(t)].flatten()]
        hist_data = np.array(selected_pixels)
        final_thresh = threshold_otsu(hist_data) if hist_data.size > 0 else threshold_otsu(image[~np.isnan(image)])
        hist_info = {"data": hist_data, "threshold": final_thresh, "label": "Final Threshold"}
    result = image < final_thresh
    return result, hist_info

def fuzzy_otsu_thresholding(image: np.ndarray, spread_factor: float, cutoff: float) -> Tuple[np.ndarray, Dict[str, Any]]:
    valid_pixels = image[~np.isnan(image)]
    if valid_pixels.size == 0: return np.zeros_like(image, dtype=bool), {}
    global_thresh = threshold_otsu(valid_pixels)
    water_pixels, land_pixels = valid_pixels[valid_pixels < global_thresh], valid_pixels[valid_pixels >= global_thresh]
    if water_pixels.size == 0 or land_pixels.size == 0:
        result, hist_info = image < global_thresh, {"data": valid_pixels, "threshold": global_thresh, "label": "Otsu Threshold"}
    else:
        min_center, max_center = np.mean(water_pixels), np.mean(land_pixels)
        midpoint = (min_center + max_center) / 2
        spread = (max_center - min_center) * spread_factor if (max_center - min_center) != 0 else 1
        fuzzy_values = 1 - np.exp(-((image - midpoint)**2) / (2 * spread**2))
        result, hist_info = fuzzy_values >= cutoff, {"data": valid_pixels, "threshold": midpoint, "label": "Fuzzy Midpoint"}
    return result, hist_info

def compare_rasters(result_file_path: str, truth_file_path: str) -> dict:
    if not os.path.exists(result_file_path) or not os.path.exists(truth_file_path): return {"error": "File not found"}
    with rasterio.open(result_file_path) as src_result, rasterio.open(truth_file_path) as src_truth:
        arr_result = src_result.read(1)
        arr_truth_reprojected = np.empty_like(arr_result, dtype=src_truth.dtypes[0])
        reproject(
            source=rasterio.band(src_truth, 1), destination=arr_truth_reprojected,
            src_transform=src_truth.transform, src_crs=src_truth.crs, src_nodata=src_truth.nodata,
            dst_transform=src_result.transform, dst_crs=src_result.crs, dst_nodata=255, # 결과 파일의 nodata와 동일하게 설정
            resampling=Resampling.nearest
        )
        arr_truth_final = np.where(arr_truth_reprojected == 1, 1, 0)
        valid_mask = arr_result != 255
        arr1_valid, arr2_valid = arr_result[valid_mask], arr_truth_final[valid_mask]
        total, diff = arr1_valid.size, np.sum(arr1_valid != arr2_valid)
        return {"total_pixels": total, "diff_pixels": diff, "diff_percent": (diff / total * 100) if total else 0.0}

def run_single_threshold_analysis(
    params: Dict[str, Any], 
    post_flood_img: np.ndarray, 
    pre_flood_img: Optional[np.ndarray], 
    slope_mask: Optional[np.ndarray],
    nan_mask: np.ndarray,
    meta: rasterio.profiles.Profile
):
    """사전에 준비된 데이터 배열을 받아 특정 임계값에 대한 분석만 수행"""
    algorithms = {
        "edge_otsu": (edge_otsu_thresholding, {"iterations": params["edge_buffer"], "initial_threshold": params["edge_thresh"]}),
        "bmax_otsu": (bmax_otsu_thresholding, {"tile_size": params["bmax_tile_size"], "variance_percentile": params["bmax_percentile"], "initial_threshold": params["bmax_thresh"]}),
        "fuzzy_otsu": (fuzzy_otsu_thresholding, {"spread_factor": params["fuzzy_spread"], "cutoff": params["fuzzy_cutoff"]})
    }

    for name, (func, func_params) in algorithms.items():
        water_map, hist_info = func(post_flood_img, **func_params)
        water_map[nan_mask] = False
        
        if slope_mask is not None:
            water_map[slope_mask] = False
            
        save_raster(water_map, meta, os.path.join(params["output_dir"], f'post_flood_water_{name}.tif'), nan_mask)

        pre_hist_info = None
        if pre_flood_img is not None:
            pre_water_map, pre_hist_info = func(pre_flood_img, **func_params)
            pre_water_map[nan_mask] = False
            if slope_mask is not None:
                pre_water_map[slope_mask] = False
            flood_extent = water_map & ~pre_water_map
            save_raster(flood_extent, meta, os.path.join(params["output_dir"], f'flood_change_{name}.tif'), nan_mask)
        
        plot_and_save_histograms(name, params["output_dir"], hist_info, pre_hist_info)

# ==============================================================================
# --- 메인 실행 로직 ---
# ==============================================================================

# --- [추가된 기능] 분석 시작 전, 설정 파일 저장 ---
file_paths_to_log = {
    "PRE_FLOOD_IMAGE_PATH": PRE_FLOOD_IMAGE_PATH,
    "POST_FLOOD_IMAGE_PATH": POST_FLOOD_IMAGE_PATH,
    "AOI_SHP_PATH": AOI_SHP_PATH,
    "GROUND_TRUTH_PATH": GROUND_TRUTH_PATH,
    "SLOPE_TIF_PATH": SLOPE_TIF_PATH,
}
params_to_log = {
    "THRESHOLDS_TO_TEST": f"{THRESHOLDS_TO_TEST.start} to {THRESHOLDS_TO_TEST.stop} (step {THRESHOLDS_TO_TEST.step})",
    "SLOPE_THRESHOLD": SLOPE_THRESHOLD,
    "EDGE_OTSU_BUFFER_ITERATIONS": EDGE_OTSU_BUFFER_ITERATIONS,
    "BMAX_OTSU_TILE_SIZE": BMAX_OTSU_TILE_SIZE,
    "BMAX_OTSU_VARIANCE_PERCENTILE": BMAX_OTSU_VARIANCE_PERCENTILE,
    "FUZZY_OTSU_SPREAD_FACTOR": FUZZY_OTSU_SPREAD_FACTOR,
    "FUZZY_OTSU_CUTOFF": FUZZY_OTSU_CUTOFF,
}
save_parameters_to_txt(BASE_OUTPUT_DIR, file_paths_to_log, params_to_log)


# 1. 데이터 준비 (분석 시작 전 1회만 실행)
print("="*50)
print("데이터 준비를 시작합니다... (이 과정은 한 번만 실행됩니다)")

aoi_gdf = gpd.read_file(AOI_SHP_PATH)
aoi_geometry = [geom for geom in aoi_gdf.geometry]

# 1a. 기준 데이터 로드 (Post-flood)
post_flood_img, meta = load_and_clip_raster(POST_FLOOD_IMAGE_PATH, aoi_geometry)
if post_flood_img is None:
    raise FileNotFoundError("홍수 후 이미지를 불러오는 데 실패했습니다. 분석을 중단합니다.")
nan_mask = np.isnan(post_flood_img)

# 1b. 홍수 전 이미지 로드 및 정렬
pre_flood_img_raw, pre_meta_raw = load_and_clip_raster(PRE_FLOOD_IMAGE_PATH, aoi_geometry)
resampled_pre_flood = None
if pre_flood_img_raw is not None:
    print("홍수 전 이미지를 홍수 후 이미지 격자에 맞게 정렬(재투영)합니다...")
    resampled_pre_flood = np.empty_like(post_flood_img)
    reproject(
        source=pre_flood_img_raw, destination=resampled_pre_flood,
        src_transform=pre_meta_raw['transform'], src_crs=pre_meta_raw['crs'],
        dst_transform=meta['transform'], dst_crs=meta['crs'],
        resampling=Resampling.bilinear
    )
    print("-> 정렬 완료.")

# 1c. 경사도 파일 로드 및 정렬
slope_img_raw, slope_meta_raw = load_and_clip_raster(SLOPE_TIF_PATH, aoi_geometry)
slope_mask = None
if slope_img_raw is not None:
    print("경사도 파일을 홍수 후 이미지 격자에 맞게 정렬(재투영)합니다...")
    resampled_slope = np.empty_like(post_flood_img)
    reproject(
        source=slope_img_raw, destination=resampled_slope,
        src_transform=slope_meta_raw['transform'], src_crs=slope_meta_raw['crs'],
        dst_transform=meta['transform'], dst_crs=meta['crs'],
        resampling=Resampling.bilinear
    )
    print("-> 정렬 완료.")
    
    h, w = post_flood_img.shape
    resampled_slope = resampled_slope[:h, :w]
    slope_mask = (~np.isnan(resampled_slope)) & (resampled_slope >= SLOPE_THRESHOLD)
else:
    print(f"!!! 경고: 경사도 파일 처리 실패. 경사도 필터링 없이 분석을 진행합니다.")

print("데이터 준비 완료.")
print("="*50)

# 2. 민감도 분석 실행 (임계값 변경하며 반복)
results_log = []
for threshold in THRESHOLDS_TO_TEST:
    print(f"\n{'='*20} 테스트 시작: 초기 임계값 = {threshold} {'='*20}")
    current_output_dir = os.path.join(BASE_OUTPUT_DIR, f"thresh_{threshold}")
    os.makedirs(current_output_dir, exist_ok=True)
    
    params = {
        "output_dir": current_output_dir,
        "slope_thresh": SLOPE_THRESHOLD,
        "edge_buffer": EDGE_OTSU_BUFFER_ITERATIONS,
        "bmax_tile_size": BMAX_OTSU_TILE_SIZE,
        "bmax_percentile": BMAX_OTSU_VARIANCE_PERCENTILE,
        "fuzzy_spread": FUZZY_OTSU_SPREAD_FACTOR,
        "fuzzy_cutoff": FUZZY_OTSU_CUTOFF,
        "edge_thresh": threshold,
        "bmax_thresh": threshold
    }
    
    run_single_threshold_analysis(
        params,
        post_flood_img=post_flood_img,
        pre_flood_img=resampled_pre_flood,
        slope_mask=slope_mask,
        nan_mask=nan_mask,
        meta=meta
    )
    
    print("\n--- 결과 비교 ---")
    for algorithm in ["edge_otsu", "bmax_otsu", "fuzzy_otsu"]:
        result_file = os.path.join(current_output_dir, f"post_flood_water_{algorithm}.tif")
        comparison = compare_rasters(result_file, GROUND_TRUTH_PATH)
        if "error" in comparison: continue
        print(f"[{algorithm.upper():<10}] 차이 비율: {comparison['diff_percent']:.4f}%")
        results_log.append({"threshold": threshold, "algorithm": algorithm, "diff_percent": comparison['diff_percent']})

# 3. 최종 결과 요약
if results_log:
    print(f"\n{'='*25} 최종 분석 결과 요약 {'='*25}")
    df = pd.DataFrame(results_log)
    if not df.empty:
        print("\n--- 전체 결과 ---")
        print(df.to_string())

        print("\n--- 최적 초기 임계값 ---")
        for algo_name in ["edge_otsu", "bmax_otsu", "fuzzy_otsu"]:
            algo_df = df[df['algorithm'] == algo_name]
            if not algo_df.empty:
                best_row = algo_df.loc[algo_df['diff_percent'].idxmin()]
                print(f"[{algo_name.upper():<10}] 최적 초기 임계값: {best_row['threshold']:<3} (차이: {best_row['diff_percent']:.4f}%)")
        
        # --- 결과 그래프 시각화 ---
        print("\n--- 민감도 분석 그래프 ---")
        plt.figure(figsize=(15, 8))
        for algo_name in df['algorithm'].unique():
            algo_df = df[df['algorithm'] == algo_name].sort_values(by='threshold')
            plt.plot(algo_df['threshold'], algo_df['diff_percent'], marker='o', linestyle='-', label=algo_name)
        
        plt.title('Initial Threshold vs. Difference Percentage', fontsize=16)
        plt.xlabel('Initial Threshold (dB)', fontsize=12)
        plt.ylabel('Difference Percentage (%)', fontsize=12)
        plt.gca().invert_xaxis() # X축을 거꾸로 (0, -1, -2...)
        plt.legend(fontsize=10)
        plt.grid(True, which='both', linestyle='--', linewidth=0.5)
        plt.show()

else:
    print("\n분석이 실행되었지만 비교할 결과가 없습니다.")

print("\n모든 분석이 완료되었습니다.")